# Araseの電磁場データについて、64 Hzデータを用いる。

# FACの定義および磁場の衛星スピントーンの補正の付加

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データの取得

In [ ]:
import pyspedas as psp
import pytplot as pt
import ergpyspedas.erg as ergpy

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330'

ergpy.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', get_support_data=True, no_update=True)
ergpy.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True, no_update=True)

print("--- Loaded tplot variables ---")
print(psp.tplot_names())

In [ ]:
import xarray as xr
import numpy as np

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
print(len(ds_B64_dsi_segs))

In [ ]:
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
print(ds_B64_dsi_seg0)

# 磁場のスピントーン除去 [Imajo et al., 2021]

In [ ]:
da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

In [ ]:
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)
os.chdir('./KAW_observation')
print(os.getcwd())

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi_seg0.time.values,
    Bx=ds_B64_dsi_seg0['B64_dsi_x'].values,
    By=ds_B64_dsi_seg0['B64_dsi_y'].values,
    Bz=ds_B64_dsi_seg0['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

ds_B64_dsi_seg0_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

print(ds_B64_dsi_seg0_spt)
print(ds_B64_dsi_seg0_clean)

# 電場データと磁場データの時間を合わせる

In [ ]:
E64_data_dsi_x  = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_quality_flag', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

E64_data_dsi_x_qf = E64_data_dsi_x.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_dsi_y_qf = E64_data_dsi_y.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_dsi_xy   = xr.Dataset({
    'E64_dsi_x': E64_data_dsi_x_qf,
    'E64_dsi_y': E64_data_dsi_y_qf
})
ds_E64_dsi_xy   = ds_E64_dsi_xy.dropna(dim='time', how='all')

In [ ]:
ds_E64_dsi_xy_segs = split_by_gap(ds_E64_dsi_xy, gap_thr=np.timedelta64(63, 'ms'))
print(ds_E64_dsi_xy_segs)

In [ ]:
def make_ds_EB_func(ds_E_xy, E_vars, ds_B_xyz, B_vars, output_vars):
    time_base   = ds_E_xy.time
    ds_B_xyz_interp  = ds_B_xyz.interp(time=time_base, method='linear')

    da_Ex   = ds_E_xy[E_vars[0]]
    da_Ey   = ds_E_xy[E_vars[1]]
    da_Bx   = ds_B_xyz_interp[B_vars[0]]
    da_By   = ds_B_xyz_interp[B_vars[1]]
    da_Bz   = ds_B_xyz_interp[B_vars[2]]

    da_Ez   = xr.where(np.abs(da_Bz) > 1E-2, -(da_Ex * da_Bx + da_Ey * da_By) / da_Bz, np.nan)

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [ ]:
E64_xy_vars     = ['E64_dsi_x', 'E64_dsi_y']
B64_xyz_vars    = ['B64_dsi_x_clean', 'B64_dsi_y_clean', 'B64_dsi_z_clean']
EB64_vars       = ['E64_dsi_x', 'E64_dsi_y', 'E64_dsi_z', 'B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']

In [ ]:
ds_EB64_dsi_segs    = []

for count in range(len(ds_E64_dsi_xy_segs)):
    ds_     = make_ds_EB_func(ds_E64_dsi_xy_segs[count], E64_xy_vars, ds_B64_dsi_seg0_clean, B64_xyz_vars, EB64_vars)
    ds_EB64_dsi_segs.append(ds_)

print(ds_EB64_dsi_segs)

In [ ]:
ds_EB64_dsi_segs = [ds_EB64_dsi_segs[0], ds_EB64_dsi_segs[2], ds_EB64_dsi_segs[3], ds_EB64_dsi_segs[4], ds_EB64_dsi_segs[6]]

In [ ]:
print(ds_EB64_dsi_segs)

In [ ]:
path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330'
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T21:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E64_dsi_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E64_dsi_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E64_dsi_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B64_dsi_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B64_dsi_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B64_dsi_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (DSI)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (DSI)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (DSI)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (DSI)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (DSI)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (DSI)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_dsi'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB64_dsi_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# FAC座標系を定義、DSI座標系 -> FAC座標系変換行列の作成

# FAC座標系の定義
- z軸は、背景磁場$B_{0}$の単位ベクトルで与える。
- x軸は、反地球方向かつz軸と垂直な単位ベクトルで与える。 ($E_{x}$: Toroidal component, $B_{x}$: Poloidal component)
- y軸は、z軸とx軸の外積で与える。 ($E_{y}$: Poloidal component, $B_{y}$: Toroidal component)

In [ ]:
ergpy.orb(trange=time_range, level='l2', datatype='def')
da_Arase_pos_gsm    = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)
da_Arase_pos_gsm    = da_Arase_pos_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

da_Arase_pos_unit_gsm   = da_Arase_pos_gsm / np.sqrt((da_Arase_pos_gsm * da_Arase_pos_gsm).sum(dim='v_dim'))

psp.store_data('Arase_pos_unit_gsm', data={'x': da_Arase_pos_unit_gsm.time, 'y': da_Arase_pos_unit_gsm.data})
psp.cotrans(name_in='Arase_pos_unit_gsm', name_out='Arase_pos_unit_j2000', coord_in='gsm', coord_out='j2000')
psp.projects.erg.erg_cotrans(in_name='Arase_pos_unit_j2000', out_name='Arase_pos_unit_dsi', in_coord='j2000', out_coord='dsi')

da_Arase_pos_unit_dsi   = psp.get_data('Arase_pos_unit_dsi', xarray=True)

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
time_width_B64          = (ds_B64_dsi_seg0_clean.time[10] - ds_B64_dsi_seg0_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_seg0_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)
print(np.nanmin(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))
print(np.nanmax(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))

In [ ]:
da_Arase_pos_unit_dsi_interp    = da_Arase_pos_unit_dsi.interp(time=da_B_background_unit.time)

da_u_   = da_Arase_pos_unit_dsi_interp - (da_Arase_pos_unit_dsi_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inDSI    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inDSI    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inDSI    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inDSI, da_e_x_FAC_inDSI, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

print(da_e_x_FAC_inDSI)
print('')
print(da_e_y_FAC_inDSI)
print('')
print(da_e_z_FAC_inDSI)

In [ ]:
R_FAC_to_DSI = xr.concat(
    [da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI],
    dim='axis'
)
R_FAC_to_DSI = R_FAC_to_DSI.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))

R_DSI_to_FAC = R_FAC_to_DSI.transpose('time', 'v_dim', 'axis')

print(R_FAC_to_DSI)
print('')
print(R_DSI_to_FAC)

In [ ]:
da_e_x_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=0)
da_e_y_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=1)
da_e_z_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=2)

In [ ]:
#import os
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#mpl.rcParams['font.size'] = 15
#
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330/coordinate"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#def cart_to_polar(u, v):
#    r = np.hypot(u, v)
#    theta = np.arctan2(v, u)  # rad
#    return theta, r
#
#def setup_polar_ax(ax, title):
#    ax.set_title(title)
#    ax.set_theta_zero_location("E")   # 0 deg を右向き（x軸正方向）
#    ax.set_theta_direction(1)         # 反時計回りを正
#    ax.minorticks_on()
#    ax.set_rmax(1.0)
#    ax.set_rticks([0.25, 0.5, 0.75, 1.0])
#    ax.set_rlabel_position(135)
#    ax.grid(True)
#
#def plot_polar_vector(ax, u, v, color, label):
#    theta, r = cart_to_polar(u, v)
#    ax.plot([theta, theta], [0, r], color=color, lw=2, label=label)
#    ax.plot(theta, r, 'o', color=color, ms=5)
#
#def plot_fac_dsi_frame_polar(it, frame_idx, save_dir):
#    ex = da_e_x_DSI_inFAC.isel(time=it)
#    ey = da_e_y_DSI_inFAC.isel(time=it)
#    ez = da_e_z_DSI_inFAC.isel(time=it)
#
#    # DSI-x in FAC
#    ex_x = ex.sel(axis='x_FAC').item()
#    ex_y = ex.sel(axis='y_FAC').item()
#    ex_z = ex.sel(axis='z_FAC').item()
#
#    # DSI-y in FAC
#    ey_x = ey.sel(axis='x_FAC').item()
#    ey_y = ey.sel(axis='y_FAC').item()
#    ey_z = ey.sel(axis='z_FAC').item()
#
#    # DSI-z in FAC
#    ez_x = ez.sel(axis='x_FAC').item()
#    ez_y = ez.sel(axis='y_FAC').item()
#    ez_z = ez.sel(axis='z_FAC').item()
#
#    fig, axs = plt.subplots(
#        1, 3, figsize=(15, 5),
#        subplot_kw={'projection': 'polar'}
#    )
#
#    # (x, y) plane
#    ax = axs[0]
#    setup_polar_ax(ax, '(x, y) plane')
#    plot_polar_vector(ax, 1, 0, 'k', 'FAC-x')
#    plot_polar_vector(ax, 0, 1, 'gray', 'FAC-y')
#    ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=0.01,
#              color='purple', label='FAC-z', linewidth=0)
#    plot_polar_vector(ax, ex_x, ex_y, 'r', 'DSI-x')
#    plot_polar_vector(ax, ey_x, ey_y, 'b', 'DSI-y')
#    plot_polar_vector(ax, ez_x, ez_y, 'g', 'DSI-z')
#    ax.legend(loc='lower left', bbox_to_anchor=(-0.15, -0.15))
#
#    # (x, z) plane
#    ax = axs[1]
#    setup_polar_ax(ax, '(x, z) plane')
#    plot_polar_vector(ax, 1, 0, 'k', 'FAC-x')
#    plot_polar_vector(ax, 0, 1, 'purple', 'FAC-z')
#    plot_polar_vector(ax, ex_x, ex_z, 'r', 'DSI-x')
#    plot_polar_vector(ax, ey_x, ey_z, 'b', 'DSI-y')
#    plot_polar_vector(ax, ez_x, ez_z, 'g', 'DSI-z')
#
#    # (y, z) plane
#    ax = axs[2]
#    setup_polar_ax(ax, '(y, z) plane')
#    plot_polar_vector(ax, 1, 0, 'gray', 'FAC-y')
#    plot_polar_vector(ax, 0, 1, 'purple', 'FAC-z')
#    plot_polar_vector(ax, ex_y, ex_z, 'r', 'DSI-x')
#    plot_polar_vector(ax, ey_y, ey_z, 'b', 'DSI-y')
#    plot_polar_vector(ax, ez_y, ez_z, 'g', 'DSI-z')
#
#    fig.suptitle(str(da_e_x_DSI_inFAC.time.values[it]))
#    plt.tight_layout()
#
#    fname = os.path.join(save_dir, f"coord_polar_{frame_idx:06d}.png")
#    fig.savefig(fname, dpi=150)
#    plt.close(fig)

In [ ]:
#import os
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#mpl.rcParams['font.size'] = 15
#
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330/coordinate"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#def setup_ax(ax, xlabel, ylabel, title):
#    ax.axhline(0, color='k', linewidth=0.5)
#    ax.axvline(0, color='k', linewidth=0.5)
#    ax.set_aspect('equal', adjustable='box')
#    ax.set_xlim(-1, 1)
#    ax.set_ylim(-1, 1)
#    ax.set_xlabel(xlabel)
#    ax.set_ylabel(ylabel)
#    ax.minorticks_on()
#    ax.grid(True, which='both', linestyle=':')
#    ax.set_title(title)
#
#def plot_fac_dsi_frame(it, frame_idx, save_dir):
#    ex = da_e_x_DSI_inFAC.isel(time=it)
#    ey = da_e_y_DSI_inFAC.isel(time=it)
#    ez = da_e_z_DSI_inFAC.isel(time=it)
#
#    # FAC 成分
#    ex_x = ex.sel(axis='x_FAC').item()
#    ex_y = ex.sel(axis='y_FAC').item()
#    ex_z = ex.sel(axis='z_FAC').item()
#
#    ey_x = ey.sel(axis='x_FAC').item()
#    ey_y = ey.sel(axis='y_FAC').item()
#    ey_z = ey.sel(axis='z_FAC').item()
#
#    ez_x = ez.sel(axis='x_FAC').item()
#    ez_y = ez.sel(axis='y_FAC').item()
#    ez_z = ez.sel(axis='z_FAC').item()
#
#    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
#
#    # ------------- (x, y) plane -------------
#    ax = axs[0]
#    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
#              color='k',   label='FAC-x')
#    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
#              color='gray', label='FAC-y')
#    ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=1,
#              color='purple', label='FAC-z', linewidth=0)
#
#    ax.quiver(0, 0, ex_x, ex_y, angles='xy', scale_units='xy', scale=1,
#              color='r', label='DSI-x')
#    ax.quiver(0, 0, ey_x, ey_y, angles='xy', scale_units='xy', scale=1,
#              color='b', label='DSI-y')
#    ax.quiver(0, 0, ez_x, ez_y, angles='xy', scale_units='xy', scale=1,
#              color='g', label='DSI-z')
#
#    setup_ax(ax, 'FAC-x (Radial)', 'FAC-y (Longitudinal)', '(x, y) plane')
#    ax.legend(loc='lower left')
#
#    # ------------- (x, z) plane -------------
#    ax = axs[1]
#    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
#              color='k',   label='FAC-x')
#    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
#              color='purple', label='FAC-z')
#
#    ax.quiver(0, 0, ex_x, ex_z, angles='xy', scale_units='xy', scale=1,
#              color='r', label='DSI-x')
#    ax.quiver(0, 0, ey_x, ey_z, angles='xy', scale_units='xy', scale=1,
#              color='b', label='DSI-y')
#    ax.quiver(0, 0, ez_x, ez_z, angles='xy', scale_units='xy', scale=1,
#              color='g', label='DSI-z')
#
#    setup_ax(ax, 'FAC-x (Radial)', 'FAC-z (Parallel)', '(x, z) plane')
#
#    # ------------- (y, z) plane -------------
#    ax = axs[2]
#    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
#              color='gray',   label='FAC-y')
#    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
#              color='purple', label='FAC-z')
#
#    ax.quiver(0, 0, ex_y, ex_z, angles='xy', scale_units='xy', scale=1,
#              color='r', label='DSI-x')
#    ax.quiver(0, 0, ey_y, ey_z, angles='xy', scale_units='xy', scale=1,
#              color='b', label='DSI-y')
#    ax.quiver(0, 0, ez_y, ez_z, angles='xy', scale_units='xy', scale=1,
#              color='g', label='DSI-z')
#
#    setup_ax(ax, 'FAC-y (Longitudinal)', 'FAC-z (Parallel)', '(y, z) plane')
#
#    fig.suptitle(str(da_e_x_DSI_inFAC.time.values[it]))
#    plt.tight_layout()
#
#    # ファイル名：time index をゼロ埋め
#    fname = os.path.join(save_dir, f"coord_{frame_idx:06d}.png")
#    fig.savefig(fname, dpi=150)
#    plt.close(fig)


In [ ]:
#from concurrent.futures import ProcessPoolExecutor, as_completed
#import multiprocessing as mp
#from tqdm import tqdm
#import numpy as np
#
## --- 1) タスク作成 ---
#n_time = da_e_x_DSI_inFAC.sizes['time']
#step = 64 * 60   # 64Hz × 60 sec
#
#tasks = []
#frame_idx = 0
#for it in range(0, n_time, step):
#    tasks.append((it, frame_idx))
#    frame_idx += 1
#
#print("num frames:", len(tasks))
#
#
## --- 2) worker ---
#def worker(args):
#    it, frame_idx, save_path = args
#    plot_fac_dsi_frame_polar(it, frame_idx, save_path)
#    return frame_idx
#
#
## --- 3) 並列 + tqdm ---
#save_path = path_base_save_plot
#n_workers = max(1, mp.cpu_count() - 1)
#
#with ProcessPoolExecutor(max_workers=n_workers) as exe:
#    futures = [
#        exe.submit(worker, (it, idx, save_path))
#        for it, idx in tasks
#    ]
#
#    for f in tqdm(as_completed(futures), total=len(futures)):
#        _ = f.result()   # 例外を拾うため

# DSI座標系 -> FAC座標系変換の実行

In [ ]:
ds_EB64_fac_segs = []

for ds_seg in ds_EB64_dsi_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_DSI_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E64_dsi_x'], ds_seg['E64_dsi_y'], ds_seg['E64_dsi_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B64_dsi_x'], ds_seg['B64_dsi_y'], ds_seg['B64_dsi_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E64_fac_x',
            'y_FAC': 'E64_fac_y',
            'z_FAC': 'E64_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B64_fac_x',
            'y_FAC': 'B64_fac_y',
            'z_FAC': 'B64_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds])
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB64_fac_segs.append(ds_fac)


In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T21:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E64_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E64_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E64_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B64_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B64_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B64_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB64_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# 軌道データから、衛星速度(DSI)を導出

In [ ]:
pos   = psp.get_data('erg_orb_l2_pos_gse', xarray=True).data   # (Nt,3)[R_E]
t_pos = psp.get_data('erg_orb_l2_pos_gse', xarray=True).time.values

R_E_m = 6.378137e6
t_s   = t_pos.astype('datetime64[ns]').astype(float) * 1e-9
v_gse = np.gradient(pos * R_E_m, t_s, axis=0)

v_sc_gse = xr.DataArray(
    data=v_gse, dims=('time','v_dim'),
    coords={'time': t_pos},
    attrs={'units':'m/s','desc':'$V_{sc}$ (GSE)'}
)
v_sc_gse.name = 'v_sc_gse'

print(v_sc_gse)

In [ ]:
path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330"
)

In [ ]:
#import matplotlib.pyplot as plt
#import datetime
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_gse_analysis  = v_sc_gse.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSE)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSE)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSE)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_gse_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_gse.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import numpy as np, pytplot as pt, pyspedas as psp
from pyspedas.projects.erg.satellite.erg.common.cotrans.dsi2j2000 import dsi2j2000

def store64(name, t, y):
    psp.store_data(name, data={'x': t.astype('datetime64[ns]'),
                              'y': np.asarray(y, dtype=np.float64)})

# 64bitで登録
store64('v_sc_gse_64', v_sc_gse.time.values, v_sc_gse.data)

# 変換
psp.cotrans(name_in='v_sc_gse_64', name_out='v_sc_j2000_64', coord_in='gse', coord_out='j2000')
dsi2j2000(name_in='v_sc_j2000_64', name_out='v_sc_dsi_64', J20002DSI=True)

# 検証
def vnorm(name): dq=psp.get_data(name, xarray=True); return np.linalg.norm(dq.data,axis=1)
ng, nj, nd = vnorm('v_sc_gse_64'), vnorm('v_sc_j2000_64'), vnorm('v_sc_dsi_64')

def check(a,b,tag,thr_rel=1e-12):
    rel = np.abs(a-b)/np.maximum(a,1e-30)
    print(f"{tag}: rel mean={rel.mean():.3e}, max={rel.max():.3e}")
    assert np.nanmax(rel) < thr_rel, f"{tag} norm not preserved"

check(ng,nj,"GSE→J2000")
check(nj,nd,"J2000→DSI")

v_sc_j2000  = psp.get_data('v_sc_j2000_64', xarray=True)
v_sc_dsi    = psp.get_data('v_sc_dsi_64', xarray=True)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_j2000_analysis  = v_sc_j2000.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (J2000)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (J2000)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (J2000)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_j2000_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_j2000.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_dsi_analysis  = v_sc_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (DSI)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (DSI)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (DSI)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_dsi_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_dsi.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
R_interp    = R_DSI_to_FAC.interp(time=v_sc_dsi.time)

v_sc_fac    = xr.dot(v_sc_dsi, R_interp, dims='v_dim')
v_sc_fac    = v_sc_fac.dropna(dim='time', how='any')
print(v_sc_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_fac_analysis  = v_sc_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sc_fac_analysis.time, np.sqrt(v_sc_fac_analysis.data[:, 0]**2E0 + v_sc_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sc_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
psp.store_data('erg_mgf_l2_mag_64hz_background_dsi', data={'x': da_B_background.time, 'y': da_B_background.data})

In [ ]:
import pyspedas as psp
import pytplot as pt

ergpy.lepe(trange=time_range, datatype='3dflux', level='l2')
ergpy.lepi(trange=time_range, datatype='3dflux', level='l2')
ergpy.pwe_hfa(trange=time_range, level='l3')

psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FHEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FODU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

In [ ]:
ND_electron_LEP = psp.get_data('erg_lepe_l2_3dflux_FEDU_density', xarray=True)
Temp_electron   = psp.get_data('erg_lepe_l2_3dflux_FEDU_avgtemp', xarray=True)

ND_electron_HFA = psp.get_data('erg_pwe_hfa_l3_1min_ne_mgf', xarray=True)
ND_electron_HFA_flag    = psp.get_data('erg_pwe_hfa_l3_1min_quality_flag', xarray=True)
ND_electron_HFA = xr.where(ND_electron_HFA_flag < 1, ND_electron_HFA, np.nan)

ND_proton       = psp.get_data('erg_lepi_l2_3dflux_FPDU_density', xarray=True).fillna(0)   # [/cc]
Flux_proton     = psp.get_data('erg_lepi_l2_3dflux_FPDU_flux', xarray=True).fillna(0)      # [/s/cm2]
Temp_proton     = psp.get_data('erg_lepi_l2_3dflux_FPDU_avgtemp', xarray=True).fillna(0)   # [eV]

ND_Helium       = psp.get_data('erg_lepi_l2_3dflux_FHEDU_density', xarray=True).fillna(0)
Flux_Helium     = psp.get_data('erg_lepi_l2_3dflux_FHEDU_flux', xarray=True).fillna(0)
Temp_Helium     = psp.get_data('erg_lepi_l2_3dflux_FHEDU_avgtemp', xarray=True).fillna(0)

ND_Oxygen       = psp.get_data('erg_lepi_l2_3dflux_FODU_density', xarray=True).fillna(0)
Flux_Oxygen     = psp.get_data('erg_lepi_l2_3dflux_FODU_flux', xarray=True).fillna(0)
Temp_Oxygen     = psp.get_data('erg_lepi_l2_3dflux_FODU_avgtemp', xarray=True).fillna(0)

ND_ion          = ND_proton + ND_Helium + ND_Oxygen                                     # [/cc]
Flux_ion        = Flux_proton + Flux_Helium + Flux_Oxygen                               # [/s/cm2]
Ptot_ion        = ND_proton*Temp_proton + ND_Helium*Temp_Helium + ND_Oxygen*Temp_Oxygen # [eV/cc]

v_ion_dsi       = Flux_ion / ND_ion * 1E-2  # [m/s]
Temp_ion        = Ptot_ion / ND_ion         # [eV]

proton_mass_kg  = 1.6726219e-27  # kg
Helium_mass_kg  = proton_mass_kg * 4
Oxygen_mass_kg  = proton_mass_kg * 16

ion_mass        = (ND_proton*proton_mass_kg + ND_Helium*Helium_mass_kg + ND_Oxygen*Oxygen_mass_kg) / ND_ion #[kg]

In [ ]:
R_interp    = R_DSI_to_FAC.interp(time=v_ion_dsi.time)

v_ion_fac   = xr.dot(v_ion_dsi, R_interp, dims='v_dim')
v_ion_fac   = v_ion_fac.dropna(dim='time', how='any')

print(v_ion_fac.time)
print(v_sc_fac.time)

In [ ]:
v_sys_fac   = v_ion_fac.interp(time=v_sc_fac.time, method='linear') - v_sc_fac
v_sys_fac   = v_sys_fac.dropna(dim='time', how='any')
print(v_sys_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_dsi_analysis  = v_ion_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{i}x}$ (DSI)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{i}y}$ (DSI)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{i}z}$ (DSI)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_ion_dsi_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_dsi.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_fac_analysis  = v_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{i}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{i}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{i}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{i}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sys_fac_analysis  = v_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis.time, np.sqrt(v_sys_fac_analysis.data[:, 0]**2E0 + v_sys_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_sys_fac        = (v_sys_fac.time.data[1] - v_sys_fac.time.data[0]) / np.timedelta64(1, 's')
#v_sys_fac_mean      = v_sys_fac.rolling(time=int(100/dt_v_sys_fac), center=True).mean()
#v_sys_fac_analysis_mean     = v_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis_mean.time, np.sqrt(v_sys_fac_analysis_mean.data[:, 0]**2E0 + v_sys_fac_analysis_mean.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ND_proton_analysis  = ND_proton.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Helium_analysis  = ND_Helium.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Oxygen_analysis  = ND_Oxygen.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_ion_analysis     = ND_ion.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ion_mass_analysis  = ion_mass.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ND_proton_analysis.time,  ND_proton_analysis.data,    lw=1, c='k')
#ax_1.plot(ND_Helium_analysis.time,  ND_Helium_analysis.data,    lw=1, c='k')
#ax_2.plot(ND_Oxygen_analysis.time,  ND_Oxygen_analysis.data,    lw=1, c='k')
#ax_3.plot(ND_ion_analysis.time,     ND_ion_analysis.data,       lw=1, c='k')
#ax_4.plot(ion_mass_analysis.time,   ion_mass_analysis.data/proton_mass_kg, lw=1, c='k')
#
#ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_0.set_ylim(ymax=2)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1E-4)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_2.set_yscale('log')
#ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_3.set_yscale('log')
#ax_3.set_ylim(ymax=2)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_4.set_ylim(ymin=1, ymax=2.5)
#
#ax_4.set_xlim(ion_mass_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_ND               = (ND_proton.time.data[1] - ND_proton.time.data[0]) / np.timedelta64(1, 's')
#
#ND_proton_mean  = ND_proton.rolling(time=int(100/dt_ND), center=True).mean()
#ND_Helium_mean  = ND_Helium.rolling(time=int(100/dt_ND), center=True).mean()
#ND_Oxygen_mean  = ND_Oxygen.rolling(time=int(100/dt_ND), center=True).mean()
#ND_ion_mean     = ND_ion.rolling(time=int(100/dt_ND), center=True).mean()
#ion_mass_mean   = ion_mass.rolling(time=int(100/dt_ND), center=True).mean()
#
#ND_proton_analysis_mean  = ND_proton_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Helium_analysis_mean  = ND_Helium_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Oxygen_analysis_mean  = ND_Oxygen_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_ion_analysis_mean     = ND_ion_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ion_mass_analysis_mean   = ion_mass_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ND_proton_analysis_mean.time,  ND_proton_analysis_mean.data,    lw=1, c='k')
#ax_1.plot(ND_Helium_analysis_mean.time,  ND_Helium_analysis_mean.data,    lw=1, c='k')
#ax_2.plot(ND_Oxygen_analysis_mean.time,  ND_Oxygen_analysis_mean.data,    lw=1, c='k')
#ax_3.plot(ND_ion_analysis_mean.time,     ND_ion_analysis_mean.data,       lw=1, c='k')
#ax_4.plot(ion_mass_analysis_mean.time,   ion_mass_analysis_mean.data/proton_mass_kg, lw=1, c='k')
#
#ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_0.set_ylim(ymax=2)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1E-4)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_2.set_yscale('log')
#ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_3.set_yscale('log')
#ax_3.set_ylim(ymax=2)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_4.set_ylim(ymin=1, ymax=2.5)
#
#ax_4.set_xlim(ion_mass_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{i}}}}
```
- Ion thermal speed
```math
v_{\mathrm{thi}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{i}}}}
```
- Ion acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{i}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thi}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thi}}}{c_{\mathrm{s}}} \right)^{2}
```

In [ ]:
da_B64_dsi_seg0_clean       = ds_B64_dsi_seg0_clean.to_dataarray(dim='v_dim').assign_coords(v_dim=np.arange(3))

print(da_B64_dsi_seg0_clean)

da_B64_dsi_seg0_clean_total = np.sqrt((da_B64_dsi_seg0_clean * da_B64_dsi_seg0_clean).sum(dim='v_dim'))

print(da_B64_dsi_seg0_clean_total)

In [ ]:
B_total = da_B64_dsi_seg0_clean_total

time_base = B_total.time
print(time_base)

ND_electron_LEP_interp  = ND_electron_LEP.interp(time=time_base, method='linear')
ND_electron_HFA_interp  = ND_electron_HFA.interp(time=time_base, method='linear')
ND_electron_MID_interp  = (ND_electron_HFA_interp + ND_electron_LEP_interp) / 2E0


Temp_electron_interp    = Temp_electron.interp(time=time_base, method='linear')
Temp_ion_interp         = Temp_ion.interp(time=time_base, method='linear')
v_ion_fac_interp        = v_ion_fac.interp(time=time_base, method='linear')
v_sys_fac_interp        = v_sys_fac.interp(time=time_base, method='linear')
ion_mass_interp         = ion_mass.interp(time=time_base, method='linear')

v_ion_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_ion_fac_interp.data[:, 0]**2E0 + v_ion_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_ion_fac_interp.time},
    attrs=v_ion_fac_interp.attrs
)
v_sys_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_sys_fac_interp.data[:, 0]**2E0 + v_sys_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_sys_fac_interp.time},
    attrs=v_sys_fac_interp.attrs
)

elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

Alfven_speed_LEP    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_LEP_interp*1E6 * ion_mass_interp)
Alfven_speed_HFA    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_HFA_interp*1E6 * ion_mass_interp)
Alfven_speed_MID    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_MID_interp*1E6 * ion_mass_interp)

ion_thermal_speed   = np.sqrt(2E0 * Temp_ion_interp*elementary_charge / ion_mass_interp)
ion_acoustic_speed  = np.sqrt(Temp_electron_interp*elementary_charge / ion_mass_interp)

electron_mass_kg        = 9.1093837E-31
electron_thermal_speed  = np.sqrt(2E0 * Temp_electron_interp*elementary_charge / electron_mass_kg)

proton_cycl_freq    = elementary_charge * B_total*1E-9 / proton_mass_kg / 2E0 / np.pi

ion_plasma_beta_LEP = (ion_thermal_speed / Alfven_speed_LEP)**2E0
ion_plasma_beta_HFA = (ion_thermal_speed / Alfven_speed_HFA)**2E0
ion_plasma_beta_MID = (ion_thermal_speed / Alfven_speed_MID)**2E0

ion_to_electron_temp_ratio  = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0

# moving mean
dt_time_base            = (time_base.data[1] - time_base.data[0]) / np.timedelta64(1, 's')

Alfven_speed_LEP_mean   = Alfven_speed_LEP.rolling(time=int(100/dt_time_base), center=True).mean()
Alfven_speed_HFA_mean   = Alfven_speed_HFA.rolling(time=int(100/dt_time_base), center=True).mean()
Alfven_speed_MID_mean   = Alfven_speed_MID.rolling(time=int(100/dt_time_base), center=True).mean()

ion_thermal_speed_mean  = ion_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
electron_thermal_speed_mean = electron_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_acoustic_speed_mean = ion_acoustic_speed.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_perp_mean     = v_sys_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_x_mean        = v_sys_fac_interp[:, 0].rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_y_mean        = v_sys_fac_interp[:, 1].rolling(time=int(100/dt_time_base), center=True).mean()

v_ion_fac_perp_mean     = v_ion_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()
v_ion_fac_x_mean        = v_ion_fac_interp[:, 0].rolling(time=int(100/dt_time_base), center=True).mean()
v_ion_fac_y_mean        = v_ion_fac_interp[:, 1].rolling(time=int(100/dt_time_base), center=True).mean()

ion_plasma_beta_LEP_mean            = ion_plasma_beta_LEP.rolling(time=int(100/dt_time_base), center=True).mean()
ion_plasma_beta_HFA_mean            = ion_plasma_beta_HFA.rolling(time=int(100/dt_time_base), center=True).mean()
ion_plasma_beta_MID_mean            = ion_plasma_beta_MID.rolling(time=int(100/dt_time_base), center=True).mean()

ion_to_electron_temp_ratio_mean = ion_to_electron_temp_ratio.rolling(time=int(100/dt_time_base), center=True).mean()
proton_cycl_freq_mean           = proton_cycl_freq.rolling(time=int(100/dt_time_base), center=True).mean()

# DataSet格納
ds_velocity_ms_perp = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_ion_speed':           v_ion_fac_perp_mean,
        'perp_sys_speed':           v_sys_fac_perp_mean
    }
)
ds_velocity_ms_perp = ds_velocity_ms_perp.dropna(dim='time', how='all')

print(ds_velocity_ms_perp)

ds_velocity_ms_toroidal = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_ion_speed':           v_ion_fac_x_mean,
        'perp_sys_speed':           v_sys_fac_x_mean
    }
)
ds_velocity_ms_toroidal = ds_velocity_ms_toroidal.dropna(dim='time', how='all')

print(ds_velocity_ms_toroidal)

ds_velocity_ms_poloidal = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_ion_speed':           v_ion_fac_y_mean,
        'perp_sys_speed':           v_sys_fac_y_mean
    }
)
ds_velocity_ms_poloidal = ds_velocity_ms_poloidal.dropna(dim='time', how='all')

print(ds_velocity_ms_poloidal)

ds_parameter = xr.Dataset(
    {
        'ion_plasma_beta_LEP':      ion_plasma_beta_LEP_mean,
        'ion_plasma_beta_MID':      ion_plasma_beta_MID_mean,
        'ion_plasma_beta_HFA':      ion_plasma_beta_HFA_mean,
        'i-e_temp_ratio':           ion_to_electron_temp_ratio_mean,
        'proton_cycl_freq_Hz':      proton_cycl_freq_mean,
        'number_density_LEP_cc':    ND_electron_LEP_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'number_density_MID_cc':    ND_electron_MID_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'number_density_HFA_cc':    ND_electron_HFA_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'ion_mass_kg':              ion_mass_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_ion_eV':              Temp_ion_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_electron_eV':         Temp_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'B_total_nT':               B_total.rolling(time=int(100/dt_time_base), center=True).mean()
    }
)
ds_parameter = ds_parameter.dropna(dim='time', how='all')

print(ds_parameter)

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from datetime import datetime
import matplotlib.dates as mdates

mpl.rcParams['font.size'] = 25

fig = plt.figure(figsize=(11, 21))
gs = fig.add_gridspec(6, 1, hspace=0)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_LEP_cc'],       lw=2, c='b', linestyle='-.', label=r'LEP-e')
ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_MID_cc'],       lw=1, c='k', label=r'MID')
ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_HFA_cc'],       lw=2, c='r', linestyle='-.', label=r'HFA')
ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],                  lw=1, c='k')
ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_MID'],         lw=1, c='k', label=r'$\beta_{\mathrm{i}}$')
ax_3.plot(ds_parameter_analysis.time, electron_mass_kg / ds_parameter_analysis['ion_mass_kg'], lw=2, c='green', linestyle='-.', label=r'$m_{\mathrm{e}}/m_{\mathrm{i}}$')
ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='b', label=r'$v_{\mathrm{the}}$')
ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,   lw=1, c='k', label=r'$v_{\mathrm{A}}$')
ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='red')

ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + r'[$\mathrm{cm}^{-3}$]')
ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
ax_2.set_ylabel(r'$B_{0}$'                              + '\n' + '[nT]')
ax_3.set_ylabel(r'$\beta_{\mathrm{i}}$')
ax_4.set_ylabel(r'$v_{\mathrm{the}}$, $v_{\mathrm{A}}$' + '\n' + '[km/s]')
ax_5.set_ylabel(r'$v_{\mathrm{thi}}$'                   + '\n' + '[km/s]')

#ax_1.set_ylim(ymin=0)
ax_1.set_yscale('log')
ax_3.set_yscale('log')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)
ax_4.minorticks_on()
ax_4.grid(which='both', alpha=0.5)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)

ax_0.legend(fontsize=20, ncol=3)
ax_1.legend(fontsize=25, ncol=2)
ax_3.legend(fontsize=25, ncol=2)
ax_4.legend(fontsize=25, ncol=2)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_5.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_5.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
pos_da = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)  # (Nt, 3)
t_pos_py = to_py_datetime(pos_da.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

pos_X   = np.asarray(pos_da[:, 0], dtype=float)
pos_Y   = np.asarray(pos_da[:, 1], dtype=float)
pos_Z   = np.asarray(pos_da[:, 2], dtype=float)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    pos_X_i = np.interp(x_num, t_pos_num, pos_X, left=np.nan, right=np.nan)
    pos_Y_i = np.interp(x_num, t_pos_num, pos_Y, left=np.nan, right=np.nan)
    pos_Z_i = np.interp(x_num, t_pos_num, pos_Z, left=np.nan, right=np.nan)
    return pos_X_i, pos_Y_i, pos_Z_i

# 目盛フォーマッタ
def pos_formatter(x, pos=None):
    pos_X_i, pos_Y_i, pos_Z_i = interp_at(x)
    if np.any(~np.isfinite([pos_X_i, pos_Y_i, pos_Z_i])):
        return ""  # 範囲外は空
    return (f"{pos_X_i:0.2f}\n"
            f"{pos_Y_i:0.2f}\n"
            f"{pos_Z_i:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_5.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(pos_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_5.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_5.get_xticks())

fig.text(0.07, 0.067, "hhmm", ha='center', va='center')
fig.text(0.07, 0.047, r"X-GSM", ha='center', va='center')
fig.text(0.07, 0.027, r"Y-GSM", ha='center', va='center')
fig.text(0.07, 0.007, r"Z-GSM", ha='center', va='center')

add_panel_label(ax_0, '(g)')
add_panel_label(ax_1, '(h)')
add_panel_label(ax_2, '(i)')
add_panel_label(ax_3, '(j)')
add_panel_label(ax_4, '(k)')
add_panel_label(ax_5, '(l)')

fig.suptitle('Arase', y=0.99)

fig.subplots_adjust(hspace=0)
fig.tight_layout(pad=0)

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'Arase_parameter.png')
    print(fig_path)
    fig.savefig(fig_path)
    fig.savefig(os.path.join(path_base_save_plot, 'Arase_parameter.pdf'))
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       s=1E-4, c='b')
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       s=1E-4, c='k')
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       s=1E-4, c='r')
#ax_1.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      s=1E-4, c='k')
#ax_2.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, s=1E-4, c='k')
#ax_3.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     s=1E-4, c='k')
#ax_4.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         s=1E-4, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_perp.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       s=1E-4, c='b')
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       s=1E-4, c='k')
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       s=1E-4, c='r')
#ax_1.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      s=1E-4, c='k')
#ax_2.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, s=1E-4, c='k')
#ax_3.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     s=1E-4, c='k')
#ax_4.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         s=1E-4, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}x}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_toroidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       s=1E-4, c='b')
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       s=1E-4, c='k')
#ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       s=1E-4, c='r')
#ax_1.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      s=1E-4, c='k')
#ax_2.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, s=1E-4, c='k')
#ax_3.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     s=1E-4, c='k')
#ax_4.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         s=1E-4, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}y}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_poloidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#import matplotlib.ticker as mticker
#from datetime import datetime
#import matplotlib.dates as mdates
#
#mpl.rcParams['font.size'] = 25
#
#fig = plt.figure(figsize=(11, 21))
#gs = fig.add_gridspec(7, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
##ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
##ax_6.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.scatter(ds_parameter_analysis.time, ds_parameter_analysis['number_density_MID_cc'],    s=1E-4, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],                  lw=1, c='k')
#ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_mass_kg']/proton_mass_kg,  lw=1, c='k')
#ax_4.scatter(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_MID'],      s=1E-4, c='k', label=r'$\beta_{\mathrm{i}}$')
#ax_4.plot(ds_parameter_analysis.time, electron_mass_kg / ds_parameter_analysis['ion_mass_kg'], lw=2, c='green', linestyle='-.', label=r'$m_{\mathrm{e}}/m_{\mathrm{i}}$')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='b', label=r'$v_{\mathrm{the}}$')
#ax_5.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,   s=1E-4, c='k', label=r'$v_{\mathrm{A}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='red')
##ax_7.plot(ds_velocity_ms_toroidal_analysis.time, ds_velocity_ms_toroidal_analysis['perp_sys_speed']*1E-3, lw=1, c='orange', label=r'$v_{\mathrm{sys}x}$')
##ax_7.plot(ds_velocity_ms_poloidal_analysis.time, ds_velocity_ms_poloidal_analysis['perp_sys_speed']*1E-3, lw=1, c='green', label=r'$v_{\mathrm{sys}y}$')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
#ax_2.set_ylabel(r'$B_{0}$'                              + '\n' + '[nT]')
#ax_3.set_ylabel(r'$m_{\mathrm{i}}$'                     + '\n' + r'[$m_{\mathrm{H}}$]')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$v_{\mathrm{the}}$, $v_{\mathrm{A}}$' + '\n' + '[km/s]')
#ax_6.set_ylabel(r'$v_{\mathrm{thi}}$'                   + '\n' + '[km/s]')
##ax_7.set_ylabel(r'$v_{\mathrm{sys}\perp}$'              + '\n' + '[km/s]')
#
#ax_1.set_ylim(ymin=0)
#ax_3.set_ylim(ymin=1, ymax=2)
#ax_4.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
##ax_7.minorticks_on()
##ax_7.grid(which='both', alpha=0.5)
#
#ax_1.legend(fontsize=20, ncol=2)
#ax_4.legend(fontsize=20, ncol=2)
##ax_5.legend(fontsize=20, ncol=2)
##ax_7.legend(fontsize=20, ncol=2)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_6.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#ax_6.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#ax_6.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))
#
#def add_panel_label(ax, label, x=-0.15, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#def to_py_datetime(t_np64):
#    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)
#
## 軌道データ
#pos_da = psp.get_data('erg_orb_l2_pos_rmlatmlt', xarray=True)  # (Nt, 3)
#t_pos_py = to_py_datetime(pos_da.time.values)
#t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数
#
#R   = np.asarray(pos_da.values[:, 0], dtype=float)  # Re
#mlat= np.asarray(pos_da.values[:, 1], dtype=float)  # deg
#mlt = np.asarray(pos_da.values[:, 2], dtype=float)  # hour [0,24)
#
#L_shell = R / np.cos(np.deg2rad(mlat))**2E0
#
## --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
#mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)
#
## 補間関数（tick の x は「日数」なのでそのまま使う）
#def interp_at(x_num):
#    #Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
#    Ri    = np.interp(x_num, t_pos_num, L_shell, left=np.nan, right=np.nan)     # L-shellを示す
#    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
#    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
#    mlti  = np.mod(mltiu, 24.0)
#    return Ri, mlati, mlti
#
## 目盛フォーマッタ
#def rmlt_formatter(x, pos=None):
#    Ri, mlati, mlti = interp_at(x)
#    if np.any(~np.isfinite([Ri, mlati, mlti])):
#        return ""  # 範囲外は空
#    return (f"{Ri:0.2f}\n"
#            f"{mlati:0.2f}\n"
#            f"{mlti:0.2f}")
#
## セカンダリ x 軸（底 side）を作ってラベルを差し替え
#secax = ax_6.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
#secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))
#
## メインの時間ラベルと重ならないよう余白を広げる
#ax_6.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
#secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル
#
## 好みで：目盛間隔をメイン x と合わせる
#secax.set_ticks(ax_6.get_xticks())
#
#fig.text(0.07, 0.067, "hhmm", ha='center', va='center')
##fig.text(0.07, 0.047, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
#fig.text(0.07, 0.047, r"L-shell", ha='center', va='center')
#fig.text(0.07, 0.027, r"MLAT", ha='center', va='center')
#fig.text(0.07, 0.007, r"MLT", ha='center', va='center')
#
#add_panel_label(ax_0, '(l)')
#add_panel_label(ax_1, '(m)')
#add_panel_label(ax_2, '(n)')
#add_panel_label(ax_3, '(o)')
#add_panel_label(ax_4, '(p)')
#add_panel_label(ax_5, '(q)')
#add_panel_label(ax_6, '(r)')
##add_panel_label(ax_7, '(t)')
#
#fig.suptitle('Arase', y=0.99)
#
#fig.subplots_adjust(hspace=0)
#fig.tight_layout(pad=0)
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Figure_1_d.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    fig.savefig(os.path.join(path_base_save_plot, 'Figure_1_d.pdf'))
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWの確認に適した時間窓$T_{\mathrm{window}}$の検討

```math
\frac{1}{v_{\mathrm{A}}} \frac{|\bf{E}_{\perp}|}{|\bf{B}_{\perp}|} = \frac{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2}}{\sqrt{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}} = \sqrt{10} \\
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
\therefore T_{\mathrm{window}} := \frac{1}{f_{\mathrm{sc}}} = \frac{1}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \left[ 9 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \left\{ 10 + \sqrt{117 \left( \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)^{2} + 180 \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} + 100} \right\} \right]^{-\frac{1}{2}}
```

In [ ]:
def dedup_and_sort(ds):
    ds = ds.sortby("time")
    t = ds["time"].values
    _, keep = np.unique(t, return_index=True)  # 先勝ちで一意化
    return ds.isel(time=np.sort(keep))

ds_parameter_clean = dedup_and_sort(ds_parameter)
ds_velocity_ms_perp_clean = dedup_and_sort(ds_velocity_ms_perp)
ds_velocity_ms_poloidal_clean = dedup_and_sort(ds_velocity_ms_poloidal)
ds_velocity_ms_toroidal_clean = dedup_and_sort(ds_velocity_ms_toroidal)

ds_parameter_interp = ds_parameter_clean.interp(time=ds_velocity_ms_perp_clean.time)
print(ds_parameter_interp)
print(ds_velocity_ms_perp_clean)

In [ ]:
T_window = ds_parameter_interp['ion_mass_kg'] / proton_mass_kg / ds_parameter_interp['proton_cycl_freq_Hz'].data * ds_velocity_ms_perp_clean['ion_thermal_speed'].data / ds_velocity_ms_perp_clean['perp_sys_speed'].data / np.sqrt(9. + 1. / ds_parameter_interp['i-e_temp_ratio'].data * (10. + np.sqrt(117. / (ds_parameter_interp['i-e_temp_ratio'].data)**(2.) + 180. / ds_parameter_interp['i-e_temp_ratio'].data + 100.)))

da_T_window = xr.DataArray(data=T_window, dims=('time'), coords={'time': ds_velocity_ms_perp_clean.time}, name='T_window')
da_T_window

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_T_window_analysis))

In [ ]:
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#mpl.rcParams['font.size'] = 15
#fig = plt.figure(figsize=(10, 4))
#ax = fig.add_subplot(111)
#ax.plot(da_T_window_analysis.time, da_T_window_analysis.data, c='k', lw=1)
#ax.minorticks_on()
#ax.set_ylabel(r'$T_{\mathrm{window}}$' + '\n[sec]')
#ax.set_yscale('log')
#ax.set_ylim(ymin=0.1)
#ax.grid(which='both', alpha=0.5)
#ax.set_xlim(da_T_window_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'T_window.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWが期待される周波数の最小値$f_{\mathrm{predict}}$の検討

```math
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
f_{\mathrm{predict}} := 1 \times f_{\mathrm{ci}} \frac{V_{\mathrm{sys}\perp}}{v_{\mathrm{thi}}}
```

In [ ]:
da_f_predict    = 1E0 * ds_parameter_clean['proton_cycl_freq_Hz'] * ds_velocity_ms_perp_clean['perp_sys_speed'] / ds_velocity_ms_perp_clean['ion_thermal_speed']

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_f_predict_window     = da_f_predict.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_f_predict_window))

In [ ]:
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#da_f_predict_window     = da_f_predict.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#mpl.rcParams['font.size'] = 15
#fig = plt.figure(figsize=(10, 4))
#ax = fig.add_subplot(111)
#ax.plot(da_f_predict_window.time, da_f_predict_window.data, c='k', lw=1)
#ax.minorticks_on()
#ax.set_ylabel(r'$f_{\mathrm{predict}}$' + '\n[Hz]')
#ax.grid(which='both', alpha=0.5)
#ax.set_xlim(da_T_window_analysis.time.values[[0, -1]])
#
#ax.set_yscale('log')
#
#fig.tight_layout()
#
#print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'f_predict.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# Taylor's hypothesisが成り立つ平行波長 $\Lambda_{\parallel}$ [Hows et al., 2014]
```math
\Lambda_{\parallel} = \frac{2\pi}{\left| k_{\parallel} \right|} \gg \frac{v_{\mathrm{thi}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{A}}}{\left| V_{\mathrm{sys}\perp} \right|} \sqrt{\frac{1}{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}
```

In [ ]:
da_Lambda_para_toroidal = ds_velocity_ms_toroidal_clean['Alfven_speed_MID'] * ds_velocity_ms_toroidal_clean['ion_thermal_speed'] / np.abs(ds_velocity_ms_toroidal_clean['perp_sys_speed']) / (ds_parameter_clean['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter_clean['ion_mass_kg']) * np.sqrt(0.5 * (1. + 1. / ds_parameter_clean['i-e_temp_ratio']))

da_Lambda_para_poloidal = ds_velocity_ms_poloidal_clean['Alfven_speed_MID'] * ds_velocity_ms_poloidal_clean['ion_thermal_speed'] / np.abs(ds_velocity_ms_poloidal_clean['perp_sys_speed']) / (ds_parameter_clean['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter_clean['ion_mass_kg']) * np.sqrt(0.5 * (1. + 1. / ds_parameter_clean['i-e_temp_ratio']))

In [ ]:
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#from datetime import datetime
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#da_Lambda_para_toroidal_analysis    = da_Lambda_para_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#da_perp_sys_speed_toroidal_analysis = ds_velocity_ms_toroidal_clean['perp_sys_speed'].sel(time=slice(time_range_T_analysis[0], #time_range_T_analysis[1]))
#
#print(da_perp_sys_speed_toroidal_analysis)
#
#mpl.rcParams['font.size'] = 15
#fig = plt.figure(figsize=(10, 6))
#
#ax_0 = fig.add_subplot(211)
#ax_0.plot(da_perp_sys_speed_toroidal_analysis.time, da_perp_sys_speed_toroidal_analysis.data * 1E-3, c='k', lw=1)
#ax_0.axhline(0, c='grey', linestyle='dashed', alpha=0.7)
#ax_0.minorticks_on()
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ [km/s]')
#ax_0.grid(which='both', alpha=0.5)
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_1 = fig.add_subplot(212, sharex=ax_0)
#ax_1.plot(da_Lambda_para_toroidal_analysis.time, da_Lambda_para_toroidal_analysis.data / (6378.1*1E3), c='k', lw=1)
#ax_1.minorticks_on()
#ax_1.set_ylabel(r'$\Lambda_{\parallel \mathrm{tor}}$ [$R_{\mathrm{E}}$]')
#ax_1.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#ax_1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#ax_1.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))
#
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1, ymax=100)
#
#add_panel_label(ax_0, '(c)', x=-0.05)
#add_panel_label(ax_1, '(d)', x=-0.05)
#
#ax_0.set_title('Arase')
#
#fig.tight_layout()
#
#print(np.nanmin(da_Lambda_para_toroidal_analysis), np.nanmean(da_Lambda_para_toroidal_analysis))
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Arase_min_Lambda_para_toroidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    fig.savefig(os.path.join(path_base_save_plot, 'Arase_min_Lambda_para_toroidal.pdf'))
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

```math
\left| k_{x} \right| \rho_{\mathrm{i}} = 2 \pi \\
\left| k_{x} V_{\mathrm{sys}x} \right| = \left( 2 \pi \right)^{2} f_{\mathrm{ci}} \frac{\left| V_{\mathrm{sys}x} \right|}{v_{\mathrm{thi}}}
```

In [ ]:
da_kperp_Vsys_toroidal = (2. * np.pi)**2. * (ds_parameter_clean['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter_clean['ion_mass_kg']) * np.abs(ds_velocity_ms_toroidal_clean['perp_sys_speed']) / ds_velocity_ms_toroidal_clean['ion_thermal_speed']

da_kperp_Vsys_poloidal = (2. * np.pi)**2. * (ds_parameter_clean['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter_clean['ion_mass_kg']) * np.abs(ds_velocity_ms_poloidal_clean['perp_sys_speed']) / ds_velocity_ms_poloidal_clean['ion_thermal_speed']

da_ion_cycl_freq    = ds_parameter_clean['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter_clean['ion_mass_kg']

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from datetime import datetime

time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_kperp_Vsys_toroidal_analysis     = da_kperp_Vsys_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_perp_sys_speed_toroidal_analysis = ds_velocity_ms_toroidal_clean['perp_sys_speed'].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_ion_cycl_freq_analysis   = da_ion_cycl_freq.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(da_perp_sys_speed_toroidal_analysis)

mpl.rcParams['font.size'] = 15
fig = plt.figure(figsize=(10, 6))

ax_0 = fig.add_subplot(211)
ax_0.plot(da_perp_sys_speed_toroidal_analysis.time, da_perp_sys_speed_toroidal_analysis.data * 1E-3, c='k', lw=1)
ax_0.axhline(0, c='grey', linestyle='dashed', alpha=0.7)
ax_0.minorticks_on()
ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ [km/s]')
ax_0.grid(which='both', alpha=0.5)
ax_0.tick_params(axis='x', which='both', labelbottom=False)

ax_1 = fig.add_subplot(212, sharex=ax_0)
ax_1.plot(da_kperp_Vsys_toroidal_analysis.time, da_kperp_Vsys_toroidal_analysis.data / (2 * np.pi), c='k', lw=1)
ax_1.plot(da_ion_cycl_freq_analysis.time, da_ion_cycl_freq_analysis.data, c='red', lw=1)
ax_1.plot(da_ion_cycl_freq_analysis.time, da_ion_cycl_freq_analysis.data / 10., c='red', lw=2, linestyle='dashed')
ax_1.minorticks_on()
ax_1.set_ylabel(r'$\left| k_{x} V_{\mathrm{sys}x} \right| / 2 \pi$ [Hz]' + '\n' + r'($\left| k_{x} \right| \rho_{\mathrm{i}} = 2 \pi$)')
ax_1.grid(which='both', alpha=0.5)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_1.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

ax_1.set_yscale('log')
ax_1.set_ylim(ymin=0.01, ymax=64)

add_panel_label(ax_0, '(c)', x=-0.05)
add_panel_label(ax_1, '(d)', x=-0.05)

ax_0.set_title('Arase')

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'Arase_kperp_Vsys_toroidal.png')
    print(fig_path)
    fig.savefig(fig_path)
    fig.savefig(os.path.join(path_base_save_plot, 'Arase_kperp_Vsys_toroidal.pdf'))
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
ion_Larmor_radius = ion_thermal_speed_mean / (proton_cycl_freq_mean*2.*np.pi)
earth_radius = 6371E3
print(ion_Larmor_radius / earth_radius)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(ion_Larmor_radius.time, (ion_Larmor_radius / (earth_radius)).data, lw=1, c='k')
ax.set_xlabel('Time')
ax.set_ylabel(r'$\rho_{\mathrm{i}} / R_{\mathrm{E}}$')
ax.set_yscale('log')
ax.set_title('Ion Larmor radius normalized by Earth radius')
ax.grid(which='both', alpha=0.5)
fig.tight_layout()

# Wavelet analysis

In [ ]:
import os
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.tdwavelet_themis as tw
import importlib
importlib.reload(tw)

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

# --- 設定 ---
SAVE_DIR = Path("/mnt/j/observation_data/Arase_analysis_save_data")
MC_CACHE_DIR = SAVE_DIR / "mc_cache"  # 点数ごとの有意水準を保存するフォルダ
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

fs = 64.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB64_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

for i, ds_seg in enumerate(ds_EB64_fac_segs):
    n_points        = len(ds_seg.time)
    print(f'seg_{i}: n_points = {n_points}')

# ---- 5) Poynting flux calibration curve を計算 ----
#ds_KS, df_cal_scaled = tw.build_poynting_calibration_curve(
#    tw=tw,
#    fs=fs,
#    duration=T_max,
#    E0=10.0,
#    B0=2.0,
#    phase_deg=0.0,
#    s0=s0,
#    dj=dj,
#    J=J_longest,
#    use_coi_mask=True,
#    trim_cycles=5,
#)
#
#print(df_cal_scaled[[
#    "f0_input_hz", "f_bin_hz", "scale_bin",
#    "K_signed_for_mean_flux", "K_abs_for_absflux"
#]])

## ここでmc_sig95.pyを実行

In [ ]:
#import xarray as xr
#import matplotlib.pyplot as plt
#from pathlib import Path
#from tqdm.auto import tqdm
#import matplotlib as mpl
#
## --- 設定 ---
#target_dir = Path("/mnt/j/observation_data/Arase_analysis_save_data/mc_cache/")
#
## ディレクトリ内のすべての .nc ファイルを取得
#nc_files = list(target_dir.glob("*.nc"))
#
#print(f"Found {len(nc_files)} files in {target_dir}")
#
#mpl.rcParams['font.size']=15
#
## 各ファイルに対して処理を実行
#for nc_path in tqdm(nc_files, desc="Generating plots"):
#    # 出力ファイルパスの生成 (.nc -> .png)
#    png_path = nc_path.with_suffix(".png")
#
#    try:
#        # データの読み込み
#        with xr.open_dataset(nc_path) as ds:
#            freq = ds.freq.values
#            sig95 = ds.sig95.values
#            
#            # 図の作成
#            plt.figure(figsize=(10, 5))
#            plt.semilogx(freq, sig95, color='red', lw=2, label='95% Significance Level')
#
#            plt.grid(True, which="both", ls="-", alpha=0.5)
#            plt.xlabel('Frequency [Hz]')
#            plt.ylabel('WCO Significance Threshold')
#            plt.title(f'95% Significance Level vs. Frequency\n{nc_path.name}')
#            plt.legend()
#
#            # 保存
#            plt.savefig(png_path, dpi=150, bbox_inches='tight')
#            plt.close() # メモリ解放のために閉じる
#
#    except Exception as e:
#        print(f"Error processing {nc_path.name}: {e}")
#
#print("Done. All plots saved in the same folder.")
#
#mpl.rcParams['font.size']=25

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

# --- 設定 ---
SAVE_DIR = Path("/mnt/j/observation_data/Arase_analysis_save_data")
MC_CACHE_DIR = SAVE_DIR / "mc_cache"  # 点数ごとの有意水準を保存するフォルダ
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

fs = 64.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB64_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

vars_64 = ['E64_fac_x','E64_fac_y','E64_fac_z', 'B64_fac_x','B64_fac_y','B64_fac_z']

#for i, ds_seg in enumerate(ds_EB64_fac_segs):
#
#    n_points        = len(ds_seg.time)
#    start_time_str  = pd.to_datetime(ds_seg.time.values[0]).strftime('%Y%m%d_%H%M%S')
#    tmp_path = Path(os.getcwd()) / f"tmp_cwt_fs{fs:.4f}_seg{i}_J{J_longest}.nc"
#    out_path = SAVE_DIR / f"Arase_cwt_xwt_fs{fs:.4f}_seg{i}.nc"
#
#    if tmp_path.exists():
#        tmp_path.unlink()
#
#    # 1) CWT+WCO計算
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg,
#        dt=dt,
#        s0=s0,
#        dj=dj,
#        J=J_longest,
#        variables=vars_64,
#        auto_calibrate_psd=False,
#        calibration_duration=T_max,
#        calibration_A0=1.0,
#        e_unit="mV/m",
#        b_unit="nT",
#    )
#    print(list(ds_cwt.data_vars.keys()))
#    if not ds_cwt.dims:
#        print(f"skip seg_{i} because of empty.")
#        continue
#
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'E64_fac_x_coef', 'B64_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'E64_fac_y_coef', 'B64_fac_x_coef', dt, dj)
#
#    # ---- parallel Poynting flux (frequency-dependent calibrated, scale-corrected) ----
#    WE_x = ds_cwt['E64_fac_x_coef'].values.astype(np.complex64)
#    WE_y = ds_cwt['E64_fac_y_coef'].values.astype(np.complex64)
#    WB_x = ds_cwt['B64_fac_x_coef'].values.astype(np.complex64)
#    WB_y = ds_cwt['B64_fac_y_coef'].values.astype(np.complex64)
#
#    freqs = ds_cwt['freq'].values.astype(np.float64)
#    scales = tw._get_scale_from_freq(freqs, dt).astype(np.float32)
#
#    K_signed_f = ds_KS["K_signed"].interp(
#        freq=xr.DataArray(freqs, dims="freq")
#    ).values.astype(np.float32)
#
#    cross_exby = np.real(WE_x * np.conj(WB_y))
#    cross_eybx = - np.real(WE_y * np.conj(WB_x))
#
#    S_para_cwt_exby = K_signed_f[None, :] * cross_exby / scales[None, :] * 1E-12   # [W/m2]
#    S_para_cwt_eybx = K_signed_f[None, :] * cross_eybx / scales[None, :] * 1E-12   # [W/m2]
#
#    ds_cwt["EB64_wco_exby"]     = (("time", "freq"), wco_exby.astype(np.float32))
#    ds_cwt["EB64_phase_exby"]   = (("time", "freq"), phase_exby.astype(np.float32))
#    ds_cwt["EB64_wco_eybx"]     = (("time", "freq"), wco_eybx.astype(np.float32))
#    ds_cwt["EB64_phase_eybx"]   = (("time", "freq"), phase_eybx.astype(np.float32))
#
#    ds_cwt["EB64_Spara_exby"] = (('time', 'freq'), S_para_cwt_exby.astype(np.float32))
#    ds_cwt["EB64_Spara_eybx"] = (('time', 'freq'), S_para_cwt_eybx.astype(np.float32))
#
#    vars_to_save = [
#        name for name in ds_cwt.data_vars.keys()
#        if isinstance(name, str) and not name.endswith('_coef')
#    ]
#
#    # 2) いったん tmp に保存
#    ds_cwt[vars_to_save].to_netcdf(tmp_path)
#    ds_cwt.close()
#    del ds_cwt
#
#    # 3) sig95 の計算/読み出し
#    mc_filename = MC_CACHE_DIR / f"sig95_fs{fs:.4f}_seg{i}_J{J_longest}.nc"
#    if mc_filename.exists():
#        ds_sig = xr.open_dataset(mc_filename)
#    else:
#        continue
#
#    sig95_values = ds_sig.sig95.values.astype(np.float32)
#    ds_sig.close()
#    del ds_sig
#
#    # 4) tmp を読み、wco_sig95 を追加して out に書く（tmpは読んでるので上書きしない）
#    with xr.open_dataset(tmp_path) as ds_tmp:
#        ds_out = ds_tmp.assign(wco_sig95=("freq", sig95_values))
#        ds_out.to_netcdf(out_path)
#
#    # 5) tmp を消すなら
#    tmp_path.unlink(missing_ok=True)
#
#    print(f"[{i+1}/{len(ds_EB64_fac_segs)}] Done: {out_path.name}")
#
#    # メモリ解放
#    del ds_out

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
from matplotlib.colors import LogNorm

mpl.rcParams['font.size'] = 11

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, t1=None, minutes=5, f_range=(1e-2, 64.0), zrange=(1e-6, 1e3), cmap="turbo", ylabel="", unit_right="", cax=None):
    # 時間切り出し
    if t0 is not None:
        if t1 is None:
            t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.set_yscale("log")
    ax.set_ylim(f_range[0], f_range[1])
    ax.set_ylabel(f"{ylabel}\n[Hz]")
    ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.minorticks_on()
    ax.grid(which="both", alpha=0.3)

    if cax == None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(unit_right)

    return pcm, cb


In [ ]:
targets_EB64_fac_cwt = [
    ("E64_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B64_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB64_fac_cwt = {}

path_seg_EB64_list = [
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg1.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg2.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg3.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg4.nc"
]

ds_EB64_fac_cwt_segs = []

for path in path_seg_EB64_list:
    ds_seg = xr.open_dataset(path)
    ds_EB64_fac_cwt_segs.append(ds_seg)

for v, _, _ in targets_EB64_fac_cwt:
    da, coi = concat_cwt_segments(ds_EB64_fac_cwt_segs, v)
    if da is not None: joined_EB64_fac_cwt[v] = (da, coi)

In [ ]:
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330/wavelet_PSD"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB64_fac_cwt), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EB64_fac_cwt):
#        if v not in joined_EB64_fac_cwt: continue
#        da, coi = joined_EB64_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), f_range=(1E-2, 32.0),
#                       cmap="turbo", ylabel=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
import matplotlib as mpl
mpl.rcParams['font.size'] = 11

def concat_xwt_segments(dsets, var):
    das = []
    for ds in dsets:
        if ds is not None and var in ds.data_vars:
            das.append(ds[var])
    if not das: return None
    return xr.concat(das, dim="time").sortby("time")

from matplotlib.colors import Normalize, LogNorm

def plot_xwt_phase_on_ax(ax, da_wco, da_phase, t0=None, minutes=5,
                         wco_thresh=0, yrange=(1e-2, 4.0),
                         mode="wco", label_left="", unit=None, cax=None):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        wco = da_wco.sel(time=slice(t0, t1))
        phase = da_phase.sel(time=slice(t0, t1))
    else:
        wco, phase = da_wco, da_phase

    if wco.time.size == 0: return None

    T = mdates.date2num(wco.time.values)
    F = wco.freq.values
    Tm, Fm = np.meshgrid(T, F, indexing='ij')

    if mode == "wco":
        Z = wco.values.astype(float)
        cmap = "viridis"
        norm = LogNorm(vmin=1E-3, vmax=1E0)
        if unit==None:
            unit = "Coherency"
    else: # mode == "phase"
        # WCOが低い領域をマスクする
        Z = np.abs(phase.where(wco > wco_thresh).values.astype(float))
        cmap = "Spectral"
        norm = Normalize(vmin=0, vmax=180)
        if unit==None:
            unit = "Phase (abs) [deg]"

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto", norm=norm, cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    ax.grid(which='both', alpha=0.3)

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)

    if cax == None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(unit)
    if mode == "phase":
        cb.set_ticks([0, 45, 90, 135, 180])
    
    return pcm, cb

In [ ]:
targets_EB64_xwt = [
    ("EB64_wco_exby", "EB64_phase_exby", r"$E_{x}$"+r' & '+r"$B_{y}$"),
    ("EB64_wco_eybx", "EB64_phase_eybx", r"$E_{y}$"+r' & '+r"$B_{x}$"),
]

joined_EB64_xwt = {}

path_seg_EB64_mc_list = [
    "/mnt/j/observation_data/Arase_analysis_save_data/mc_cache/sig95_fs64.0000_seg1_J373.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/mc_cache/sig95_fs64.0000_seg2_J373.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/mc_cache/sig95_fs64.0000_seg3_J373.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/mc_cache/sig95_fs64.0000_seg4_J373.nc",
]

for i, path in enumerate(path_seg_EB64_mc_list):
    ds_seg_mc = xr.open_dataset(path)

    sig95 = ds_seg_mc["sig95"]

    ds = ds_EB64_fac_cwt_segs[i]

    sig95_on_ds = sig95.copy()
    sig95_on_ds = sig95_on_ds.assign_coords(freq=ds["freq"])  # freq座標を強制的に一致させる

    sig95_2d = sig95_on_ds.broadcast_like(ds["EB64_wco_exby"])

    m_exby = ds["EB64_wco_exby"] >= sig95_2d
    m_eybx = ds["EB64_wco_eybx"] >= sig95_2d

    ds = ds.assign(
        EB64_phase_exby = ds["EB64_phase_exby"].where(m_exby),
        EB64_phase_eybx = ds["EB64_phase_eybx"].where(m_eybx),
    )

    ds_EB64_fac_cwt_segs[i] = ds

for w_var, p_var, lab in targets_EB64_xwt:
    w_da = concat_xwt_segments(ds_EB64_fac_cwt_segs, w_var)
    p_da = concat_xwt_segments(ds_EB64_fac_cwt_segs, p_var)
    if w_da is not None:
        joined_EB64_xwt[lab] = (w_da, p_da)

print(joined_EB64_xwt)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
#    
#    for i, (w_var, p_var, lab) in enumerate(targets_EB64_xwt):
#        if lab not in joined_EB64_xwt: continue
#        w_da, p_da = joined_EB64_xwt[lab]
#        
#        # WCOプロット
#        plot_xwt_phase_on_ax(axes[2*i], w_da, p_da, t0=t0, mode="wco",
#                             label_left=f"Coherence ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#        
#        # Phaseプロット
#        plot_xwt_phase_on_ax(axes[2*i+1], w_da, p_da, t0=t0, mode="phase", wco_thresh=0.5,
#                             label_left=f"Phase ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_64_xwt_{fn_time}_wco0.5.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
from matplotlib.colors import SymLogNorm

mpl.rcParams['font.size'] = 11

# ---- 1面描画：外でax/caxを用意する ----
def plot_Spara_on_ax(
    ax, da, da_coi=None, t0=None, t1=None, minutes=5,
    f_range=(1e-2, 64.0), zrange=(-1e-3, 1e-3),
    cmap="turbo", ylabel="", unit_right="", cax=None,
    linthresh=1e-6
):
    if t0 is not None:
        if t1 is None:
            t1 = t0 + np.timedelta64(minutes, "m")
        da = da.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
    else:
        coi = da_coi

    if da.time.size == 0:
        return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # 有限値チェック
    finite = np.isfinite(Z)
    if not finite.any():
        ax.text(0.5, 0.5, "No finite data in this window",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_yscale("log")
        ax.set_ylim(f_range[0], f_range[1])
        ax.set_ylabel(f"{ylabel}\n[Hz]")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
        return None, None

    # zrange を使う
    vmin, vmax = zrange
    if not (np.isfinite(vmin) and np.isfinite(vmax)) or vmin >= vmax:
        vals = Z[finite]
        vmax = np.nanmax(np.abs(vals))
        vmin = -vmax

    norm = SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax, base=10)

    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto", norm=norm, cmap=cmap)

    ax.set_yscale("log")
    ax.set_ylim(f_range[0], f_range[1])
    ax.set_ylabel(f"{ylabel}\n[Hz]")
    ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.minorticks_on()
    ax.grid(which="both", alpha=0.3)

    if cax is None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])

    cb = ax.figure.colorbar(pcm, cax=cax)
    cb.set_label(unit_right)

    return pcm, cb

In [ ]:
targets_EB64_fac_Spara = [
    ("EB64_Spara_exby", r"$S_{\parallel \mathrm{tor}}$ (FAC)", "[W/m$^2$]"),
    ("EB64_Spara_eybx", r"$S_{\parallel \mathrm{pol}}$ (FAC)", "[W/m$^2$]"),
]
joined_EB64_fac_Spara = {}

path_seg_EB64_list = [
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg1.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg2.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg3.nc",
    "/mnt/j/observation_data/Arase_analysis_save_data/Arase_cwt_xwt_fs64.0000_seg4.nc"
]

ds_EB64_fac_Spara_segs = []

for path in path_seg_EB64_list:
    ds_seg = xr.open_dataset(path)
    ds_EB64_fac_Spara_segs.append(ds_seg)

for v, _, _ in targets_EB64_fac_Spara:
    da = concat_xwt_segments(ds_EB64_fac_Spara_segs, v)
    if da is not None: joined_EB64_fac_Spara[v] = da

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#os.makedirs(f'{path_base_save_plot}/wavelet_Spara', exist_ok=True)
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB64_fac_Spara), 1, figsize=(10, 5), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EB64_fac_Spara):
#        if v not in joined_EB64_fac_Spara: continue
#        da = joined_EB64_fac_Spara[v]
#        plot_Spara_on_ax(ax, da*1E3, t0=t0, minutes=5,
#                       zrange=(-1e-2, 1e-2), f_range=(np.nanmin(da.freq.data), np.nanmax(da.freq.data)),
#                       cmap="BrBG", ylabel=ylab, unit_right="[mW/m$^2$]", linthresh=1E-5)
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'wavelet_Spara/EB_fields_fac_Spara_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# 5分毎のPSDのMedianをplot

In [ ]:
da_E64_fac_x_cwt    = joined_EB64_fac_cwt["E64_fac_x_cwt"]
da_E64_fac_y_cwt    = joined_EB64_fac_cwt["E64_fac_y_cwt"]
da_E64_fac_z_cwt    = joined_EB64_fac_cwt["E64_fac_z_cwt"]
da_B64_fac_x_cwt    = joined_EB64_fac_cwt["B64_fac_x_cwt"]
da_B64_fac_y_cwt    = joined_EB64_fac_cwt["B64_fac_y_cwt"]
da_B64_fac_z_cwt    = joined_EB64_fac_cwt["B64_fac_z_cwt"]

In [ ]:
#import numpy as np
#import xarray as xr
#import matplotlib.pyplot as plt
#import os
#import matplotlib as mpl
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
## ---- 入力 ----
#pairs = [
#    ("E_64_FAC_x_cwt",   da_E64_fac_x_cwt),
#    ("E_64_FAC_y_cwt",   da_E64_fac_y_cwt),
#    ("E_64_FAC_z_cwt",   da_E64_fac_z_cwt),
#    ("B_64_FAC_x_cwt",   da_B64_fac_x_cwt),
#    ("B_64_FAC_y_cwt",   da_B64_fac_y_cwt),
#    ("B_64_FAC_z_cwt",   da_B64_fac_z_cwt),
#]
#t_all_start = np.datetime64('2022-09-01T21:00:00')
#t_all_end   = np.datetime64('2022-09-02T00:00:00')
#step        = np.timedelta64(5, 'm')  # 5分
#outdir      = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330/wavelet_PSD/5min_PSD"
#)
#os.makedirs(outdir, exist_ok=True)
#
## 事前に CWT 本体のみ取り出し、timeでソート
#pairs_sorted = []
#for name, da in pairs:
#    if isinstance(da, tuple):
#        da = da[0]
#    if isinstance(da, xr.DataArray):
#        pairs_sorted.append((name, da.sortby('time')))
#
#def compute_median_dict(pairs_sorted, t0, t1):
#    d = {}
#    for name, da in pairs_sorted:
#        sub = da.sel(time=slice(t0, t1))
#        if sub.sizes.get('time', 0) == 0:
#            continue
#        if np.iscomplexobj(sub.data):
#            sub = (sub.real**2 + sub.imag**2)
#        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
#    return d
#
#def plot_median_dict(mdict, t0, t1, outdir=None):
#    fig, ax = plt.subplots(figsize=(8, 8))
#
#    color_map = {
#        ('E', 'x'): 'blue',      # Ex
#        ('E', 'y'): 'orange',    # Ey
#        ('E', 'z'): 'brown',     # Ez
#        ('B', 'x'): 'green',     # Bx
#        ('B', 'y'): 'red',       # By
#        ('B', 'z'): 'purple',    # Bz
#    }
#
#    for name, med in mdict.items():
#        if name.split('_')[3] == 'z':
#            continue
#        prefix  = name.split('_')[0]
#        comp    = name.split('_')[3]
#        coor    = name.split('_')[2]
#        label  = f"${prefix}_{comp}$ ({coor})"
#        col     = color_map.get((prefix, comp), 'gray')
#        ax.loglog(med['freq'], med, label=label, lw=1, color=col)
#
#    ax.minorticks_on()
#    ax.set_xlabel('Frequency [Hz]')
#    ax.set_ylabel('Median PSD')
#    ax.set_title(f"Median {str(t0)[11:]}–{str(t1)[11:]}\n (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
#    ax.grid(True, which='both', ls=':')
#    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
#    ax.legend(ncol=2, fontsize=15)
#    ax.set_xlim(1e-2, 32)
#    ax.set_ylim(1e-8, 1e4)
#    plt.tight_layout()
#
#    if outdir and os.path.isdir(outdir):
#        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
#        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)
#
## ---- 5分窓でループ ----
#t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
#for t0 in t_starts:
#    t1 = t0 + step
#    mdict = compute_median_dict(pairs_sorted, t0, t1)
#    if not mdict:  # その窓でデータ無し
#        continue
#    plot_median_dict(mdict, t0, t1, outdir=outdir)
#
##t_starts    = np.datetime64('2022-09-01T23:09:00')
##t_ends      = np.datetime64('2022-09-01T23:12:00')
##mdict       = compute_median_dict(pairs_sorted, t_starts, t_ends)
##plot_median_dict(mdict, t_starts, t_ends, outdir)

In [ ]:
ds_EB64_fac_segs

In [ ]:
mu_0 = 4.*np.pi*1E-7

Ex_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_x'], ds_EB64_fac_segs[2]['E64_fac_x'], ds_EB64_fac_segs[3]['E64_fac_x'], ds_EB64_fac_segs[4]['E64_fac_x']], dim='time')
Ey_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_y'], ds_EB64_fac_segs[2]['E64_fac_y'], ds_EB64_fac_segs[3]['E64_fac_y'], ds_EB64_fac_segs[4]['E64_fac_y']], dim='time')
Bx_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_x'], ds_EB64_fac_segs[2]['B64_fac_x'], ds_EB64_fac_segs[3]['B64_fac_x'], ds_EB64_fac_segs[4]['B64_fac_x']], dim='time')
By_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_y'], ds_EB64_fac_segs[2]['B64_fac_y'], ds_EB64_fac_segs[3]['B64_fac_y'], ds_EB64_fac_segs[4]['B64_fac_y']], dim='time')

S_para  = (Ex_fac * By_fac - Ey_fac * Bx_fac) / mu_0 * 1E-12

S_para_toroidal = Ex_fac * By_fac / mu_0 * 1E-12
S_para_poloidal = - Ey_fac * Bx_fac / mu_0 * 1E-12

print(S_para)
print(S_para_toroidal)
print(S_para_poloidal)

In [ ]:
import matplotlib as mpl

import matplotlib.pyplot as plt

mpl.rcParams['font.size'] = 20

def add_panel_label(ax, label, x=-0.075, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

# S_para_spin をプロット
fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(S_para.time, np.abs(S_para.values*1E3) * np.sqrt(1E6 * ds_parameter['number_density_MID_cc'].interp(time=S_para.time)) / (ds_parameter['B_total_nT'].interp(time=S_para.time))**2, s=0.1, c='k')   #
ax.set_yscale('log')
ax.set_xlabel('Time')
ax.set_ylabel(r'$|S_{\parallel}| \sqrt{n}/ B_{0}^{2}$ [mW/m$^{7/2}$/nT$^{2}$]')
ax.set_ylim(bottom=1E-7, top=1E0)

ax.minorticks_on()
ax.grid(which='both', alpha=0.5)
ax.axhline(y=1E-4, c='r', linestyle='--', alpha=0.5)

time_range_analysis     = ['20220901/21:00:00', '20220902/00:00:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

add_panel_label(ax, '(b)')

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'Arase_S_para.png')
    print(fig_path)
    fig.savefig(fig_path, dpi=150)
    fig_path = os.path.join(path_base_save_plot, 'Arase_S_para.pdf')
    print(fig_path)
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
Vph_toroidal    = np.abs(Ey_fac / Bx_fac) * 1E6 # [m/s]
Vph_poloidal    = np.abs(Ex_fac / By_fac) * 1E6 # [m/s]

Vph_perp_comp   = np.sqrt((Ex_fac**2E0 + Ey_fac**2E0) / (Bx_fac**2E0 + By_fac**2E0)) * 1E6 # [m/s]

In [ ]:
targets_EB64_analysis = [
    ("E64_fac_x_cwt",  r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B64_fac_y_cwt",  r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("EB64_Spara_exby", r"$S_{\parallel \mathrm{tor}}$", r"[mW/m$^{2}$]"),
#    ("E64_fac_y_cwt",  r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B64_fac_x_cwt",  r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("EB64_Spara_eybx", r"$S_{\parallel \mathrm{pol}}$", r"[mW/m$^{2}$]"),
]
joined_EB64_fac_cwt_analysis = {}
for v, _, _ in targets_EB64_analysis:
    da, coi = concat_cwt_segments(ds_EB64_fac_cwt_segs,  v)
    if da is not None:
        joined_EB64_fac_cwt_analysis[v] = (da, coi)

print(joined_EB64_fac_cwt_analysis)

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
time_windows_analysis   = [np.datetime64('2022-09-01T22:25:00'), np.datetime64('2022-09-01T23:15:00')]

ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

S_para_toroidal_analysis    = S_para_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
S_para_poloidal_analysis    = S_para_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from datetime import datetime
import matplotlib.dates as mdates

mpl.rcParams['font.size'] = 22

fig = plt.figure(figsize=(10, 15))
gs = fig.add_gridspec(5, 2, height_ratios=[1, 1, 0.52546, 0.75, 1], width_ratios=[1, 0.025], wspace=0.05, hspace=0.15)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)

cax_0 = fig.add_subplot(gs[0, 1])
cax_1 = fig.add_subplot(gs[1, 1])
cax_2 = fig.add_subplot(gs[2, 1])
cax_4 = fig.add_subplot(gs[4, 1])

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)

axes = [ax_0, ax_1, ax_4]
caxes = [cax_0, cax_1, cax_4]
for ax, cax, (v, ylab, unit) in zip(axes, caxes, targets_EB64_analysis):
    if v not in joined_EB64_fac_cwt_analysis: continue
    da, coi = joined_EB64_fac_cwt_analysis[v]
    if v != "EB64_Spara_exby" and v != "EB64_Spara_eybx":
        plot_cwt_on_ax(
            ax, da, coi, t0=time_windows_analysis[0], t1=time_windows_analysis[1], minutes=None,
            f_range=(1e-2, 64.),
            zrange=(1e-6, 1e3),
            cmap="turbo",
            ylabel=ylab, unit_right=unit, cax=cax
        )
    else:
        plot_Spara_on_ax(
            ax,
            da*1E3,
            da_coi=None,
            t0=time_windows_analysis[0], t1=time_windows_analysis[1], minutes=None,
            f_range=(1e-2, 64.),
            zrange=(-1e-2, 1e-2),
            cmap="BrBG",
            ylabel=ylab, unit_right=unit, cax=cax, linthresh=1E-5
        )

axes_phi = [ax_2]
caxes_phi = [cax_2]
for i, (w_var, p_var, lab) in enumerate(targets_EB64_xwt):
    if i==0:
        if lab not in joined_EB64_xwt: continue
        w_da, p_da = joined_EB64_xwt[lab]
        plot_xwt_phase_on_ax(axes_phi[i], w_da, p_da, t0=time_windows_analysis[0], minutes=50, mode="phase",
                             yrange=(1e-2, 1e0), unit='[deg]', cax=caxes_phi[i])


# ax_2はPhase (abs) Ex-By (128 Hzのみ)

ax_3.scatter(S_para_toroidal_analysis.time, S_para_toroidal_analysis.data*1E3, c='k', s=0.1, label='64 Hz')

ax_0.set_ylabel(r'$E_{x}$'                      + '\n' + r'[$\mathrm{Hz}$]')
ax_1.set_ylabel(r'$B_{y}$'                      + '\n' + r'[$\mathrm{Hz}$]')
ax_2.set_ylabel(r'$|\phi|$'      + '\n' + r'[$\mathrm{Hz}$]')
ax_3.set_ylabel(r'$S_{\parallel}^{E_{x} B_{y}}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
ax_4.set_ylabel(r'$S_{\parallel}^{E_{x} B_{y}}$' + '\n' + r'[$\mathrm{Hz}$]')


ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)
ax_4.minorticks_on()
ax_4.grid(which='both', alpha=0.5)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_4.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_4.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.90):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
pos_da = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)  # (Nt, 3)
t_pos_py = to_py_datetime(pos_da.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

pos_X   = np.asarray(pos_da[:, 0], dtype=float)
pos_Y   = np.asarray(pos_da[:, 1], dtype=float)
pos_Z   = np.asarray(pos_da[:, 2], dtype=float)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    pos_X_i = np.interp(x_num, t_pos_num, pos_X, left=np.nan, right=np.nan)
    pos_Y_i = np.interp(x_num, t_pos_num, pos_Y, left=np.nan, right=np.nan)
    pos_Z_i = np.interp(x_num, t_pos_num, pos_Z, left=np.nan, right=np.nan)
    return pos_X_i, pos_Y_i, pos_Z_i

# 目盛フォーマッタ
def pos_formatter(x, pos=None):
    pos_X_i, pos_Y_i, pos_Z_i = interp_at(x)
    if np.any(~np.isfinite([pos_X_i, pos_Y_i, pos_Z_i])):
        return ""  # 範囲外は空
    return (f"{pos_X_i:0.2f}\n"
            f"{pos_Y_i:0.2f}\n"
            f"{pos_Z_i:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_4.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(pos_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_4.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_4.get_xticks())

fig.text(0.09, 0.085, "hhmm", ha='center', va='center')
fig.text(0.09, 0.059, r"X-GSM", ha='center', va='center')
fig.text(0.09, 0.037, r"Y-GSM", ha='center', va='center')
fig.text(0.09, 0.015, r"Z-GSM", ha='center', va='center')

add_panel_label(ax_0, '(f)')
add_panel_label(ax_1, '(g)')
add_panel_label(ax_2, '(h)')
add_panel_label(ax_3, '(i)')
add_panel_label(ax_4, '(j)')

fig.suptitle('Arase', y=0.995)   # タイトルは上に寄せる

fig.subplots_adjust(
    left=0.20,
    right=0.85,
    top=0.975,
    bottom=0.10,
    hspace=0.10,
    wspace=0.05,
)

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'Arase_wavelet_overview.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

# E/B plot

- Poloidal components: $B_{x}$, $E_{y}$
- Toroidal components: $B_{y}$, $E_{x}$

In [ ]:
ds_wco_0 = ds_EB64_fac_cwt_segs[0]['wco_sig95'].broadcast_like(ds_EB64_fac_cwt_segs[0]['EB64_wco_exby'])
ds_wco_1 = ds_EB64_fac_cwt_segs[1]['wco_sig95'].broadcast_like(ds_EB64_fac_cwt_segs[1]['EB64_wco_exby'])
ds_wco_2 = ds_EB64_fac_cwt_segs[2]['wco_sig95'].broadcast_like(ds_EB64_fac_cwt_segs[2]['EB64_wco_exby'])
ds_wco_3 = ds_EB64_fac_cwt_segs[3]['wco_sig95'].broadcast_like(ds_EB64_fac_cwt_segs[3]['EB64_wco_exby'])

In [ ]:
ds_wco_0 = ds_wco_0.where(ds_wco_0 >= 0.5, 0.5)
ds_wco_1 = ds_wco_1.where(ds_wco_1 >= 0.5, 0.5)
ds_wco_2 = ds_wco_2.where(ds_wco_2 >= 0.5, 0.5)
ds_wco_3 = ds_wco_3.where(ds_wco_3 >= 0.5, 0.5)

In [ ]:
ds_EB64_fac_toroidal  = xr.Dataset({
    'E64':          xr.concat([ds_EB64_fac_cwt_segs[0]['E64_fac_x_cwt'], ds_EB64_fac_cwt_segs[1]['E64_fac_x_cwt'], ds_EB64_fac_cwt_segs[2]['E64_fac_x_cwt'], ds_EB64_fac_cwt_segs[3]['E64_fac_x_cwt']], dim='time'),
    'B64':          xr.concat([ds_EB64_fac_cwt_segs[0]['B64_fac_y_cwt'], ds_EB64_fac_cwt_segs[1]['B64_fac_y_cwt'], ds_EB64_fac_cwt_segs[2]['B64_fac_y_cwt'], ds_EB64_fac_cwt_segs[3]['B64_fac_y_cwt']], dim='time'),
    'coherency':    xr.concat([ds_EB64_fac_cwt_segs[0]['EB64_wco_exby'], ds_EB64_fac_cwt_segs[1]['EB64_wco_exby'], ds_EB64_fac_cwt_segs[2]['EB64_wco_exby'], ds_EB64_fac_cwt_segs[3]['EB64_wco_exby']], dim='time'),
    'phase':        xr.concat([ds_EB64_fac_cwt_segs[0]['EB64_phase_exby'], ds_EB64_fac_cwt_segs[1]['EB64_phase_exby'], ds_EB64_fac_cwt_segs[2]['EB64_phase_exby'], ds_EB64_fac_cwt_segs[3]['EB64_phase_exby']], dim='time'),
    'wco_sig95':    xr.concat([ds_wco_0, ds_wco_1, ds_wco_2, ds_wco_3], dim='time'),
    'Spara':        xr.concat([ds_EB64_fac_cwt_segs[0]['EB64_Spara_exby'], ds_EB64_fac_cwt_segs[1]['EB64_Spara_exby'], ds_EB64_fac_cwt_segs[2]['EB64_Spara_exby'], ds_EB64_fac_cwt_segs[3]['EB64_Spara_exby']], dim='time'),
})

print(ds_EB64_fac_toroidal)

ds_EB64_fac_poloidal  = xr.Dataset({
    'E64':          xr.concat([ds_EB64_fac_cwt_segs[0]['E64_fac_y_cwt'], ds_EB64_fac_cwt_segs[1]['E64_fac_y_cwt'], ds_EB64_fac_cwt_segs[2]['E64_fac_y_cwt'], ds_EB64_fac_cwt_segs[3]['E64_fac_y_cwt']], dim='time'),
    'B64':          xr.concat([ds_EB64_fac_cwt_segs[0]['B64_fac_x_cwt'], ds_EB64_fac_cwt_segs[1]['B64_fac_x_cwt'], ds_EB64_fac_cwt_segs[2]['B64_fac_x_cwt'], ds_EB64_fac_cwt_segs[3]['B64_fac_x_cwt']], dim='time'),
    'coherency':    xr.concat([ds_EB64_fac_cwt_segs[0]['EB64_wco_eybx'], ds_EB64_fac_cwt_segs[1]['EB64_wco_eybx'], ds_EB64_fac_cwt_segs[2]['EB64_wco_eybx'], ds_EB64_fac_cwt_segs[3]['EB64_wco_eybx']], dim='time'),
    'phase':        xr.concat([ds_EB64_fac_cwt_segs[0]['EB64_phase_eybx'], ds_EB64_fac_cwt_segs[1]['EB64_phase_eybx'], ds_EB64_fac_cwt_segs[2]['EB64_phase_eybx'], ds_EB64_fac_cwt_segs[3]['EB64_phase_eybx']], dim='time'),
    'wco_sig95':    xr.concat([ds_wco_0, ds_wco_1, ds_wco_2, ds_wco_3], dim='time'),
    'Spara':        xr.concat([ds_EB64_fac_cwt_segs[0]['EB64_Spara_eybx'], ds_EB64_fac_cwt_segs[1]['EB64_Spara_eybx'], ds_EB64_fac_cwt_segs[2]['EB64_Spara_eybx'], ds_EB64_fac_cwt_segs[3]['EB64_Spara_eybx']], dim='time'),
})

print(ds_EB64_fac_poloidal)

## 各時間で出力してみる

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

new_time = pd.DatetimeIndex(ds_EB64_fac_toroidal.time.values)

ds_EB64_fac_toroidal_interp = ds_EB64_fac_toroidal.interp(time=new_time, method='linear').assign_coords(time=new_time)
ds_EB64_fac_poloidal_interp = ds_EB64_fac_poloidal.interp(time=new_time, method='linear').assign_coords(time=new_time)

da_Spara_toroidal_interp    = S_para_toroidal.interp(time=new_time, method='linear').assign_coords(time=new_time)
da_Spara_poloidal_interp    = S_para_poloidal.interp(time=new_time, method='linear').assign_coords(time=new_time)

ds_parameter_interp             = ds_parameter.interp(time=new_time, method='linear').assign_coords(time=new_time)
ds_velocity_ms_toroidal_interp  = ds_velocity_ms_toroidal.interp(time=new_time, method='linear').assign_coords(time=new_time)
ds_velocity_ms_poloidal_interp  = ds_velocity_ms_poloidal.interp(time=new_time, method='linear').assign_coords(time=new_time)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def _fit_powerlaw_kappa(freq, psd, mask, min_points=5):

    f = np.asarray(freq)
    y = np.asarray(psd)

    msk = np.asarray(mask, dtype=bool)
    msk &= np.isfinite(f) & np.isfinite(y) & (f > 0) & (y > 0)

    if msk.sum() < min_points:
        return np.nan, np.nan

    x = np.log10(f[msk])
    yy = np.log10(y[msk])

    N = len(x)

    # 線形回帰
    slope, intercept = np.polyfit(x, yy, 1)

    # 残差
    y_fit = intercept + slope * x
    residual = yy - y_fit

    # 残差分散
    sigma2 = np.sum(residual**2) / (N - 2)

    Sxx = np.sum((x - x.mean())**2)

    if Sxx == 0:
        return np.nan, np.nan

    slope_std = np.sqrt(sigma2 / Sxx)

    kappa = -slope
    kappa_std = slope_std

    return kappa, kappa_std


def _corr_log_model(freq, y_obs, y_model_at_freq, min_points=5):
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model_at_freq)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    if msk.sum() < min_points:
        return np.nan, int(msk.sum())

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    # 分散が小さいと相関が不安定なので弾く
    if np.std(logy) < 1e-6 or np.std(logm) < 1e-6:
        return np.nan, int(msk.sum())

    r = np.corrcoef(logy, logm)[0, 1]
    return float(r), int(msk.sum())

def _logrmse_model(freq, y_obs, y_model, min_points=5, allow_offset=False):
    """
    logRMSE = sqrt(mean((log10(y_obs) - (log10(y_model)+a))^2))
    allow_offset=True: a を平均差で最小二乗フィット（縦オフセット許容）
    """
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    n = int(msk.sum())
    if n < min_points:
        return np.nan, np.nan, n  # (logRMSE, a, n)

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    a = 0.0
    if allow_offset:
        a = float(np.mean(logy - logm))
        resid = logy - (logm + a)
    else:
        resid = logy - logm

    rmse = float(np.sqrt(np.mean(resid**2)))
    return rmse, a, n



def plot_freq_spectrum(time, ds_64, ds_par, ds_vel, da_Spara, title_label,
                      E2_label, B2_label, phi_label, EBratio_label, vsys_label, S_para_label,
                      fit_range=[3.0, 30.0], fig_plot=True):

    time    = pd.Timestamp(time)

    ds_64_time      = ds_64.sel(time=time, method="nearest")
    ds_par_time     = ds_par.sel(time=time, method="nearest")
    ds_vel_time     = ds_vel.sel(time=time, method="nearest")
    da_Spara_time   = da_Spara.sel(time=time, method="nearest")

    # 理論曲線
    f_sc    = np.logspace(-2, 2, 1000)
    tau     = ds_par_time['i-e_temp_ratio'].item()
    f_ci    = ds_par_time['proton_cycl_freq_Hz'].item() * proton_mass_kg / ds_par_time['ion_mass_kg'].item()
    v_thi   = ds_vel_time['ion_thermal_speed'].item()
    v_sys   = ds_vel_time['perp_sys_speed'].item()
    S_para  = da_Spara_time.item()
    KAW_dr  = (1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    if fit_range[1] == 0:
        fit_range[1] = np.sqrt(ds_par_time['ion_mass_kg'].item() / 9.1093837E-31 * tau)

    f_sc_krho_1     = np.abs(f_ci * v_sys / v_thi)
    f_sc_krho_low   = f_sc_krho_1 * fit_range[0]
    f_sc_krho_high  = f_sc_krho_1 * fit_range[1]

    f_spin_1 = 0.125
    f_spin_2 = f_spin_1 * 2.
    f_spin_3 = f_spin_1 * 3.
    f_spin_4 = f_spin_1 * 4.
    f_spin_5 = f_spin_1 * 5.

    # PSD
    E_64        = ds_64_time['E64']
    B_64        = ds_64_time['B64']

    coherency   = ds_64_time['coherency']
    wco_sig95   = ds_64_time['wco_sig95']
    phase       = ds_64_time['phase']

    v_A     = ds_vel_time['Alfven_speed_MID'].item()

    EB_64_ratio     = np.sqrt(E_64 / B_64) * 1E6 / v_A

    # --- fit用マスク（coherency + fit_range） ---
    freq = E_64.freq.values
    mask_coh = (coherency >= wco_sig95).values
    mask_fit = (freq >= f_sc_krho_low) & (freq <= f_sc_krho_high) & (freq >= 1E-2)
    mask_all = mask_coh & mask_fit

    KAW_dr_corr = (1. + (freq / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (freq / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    for_max_n_EB_dr     = KAW_dr_corr[mask_fit]
    for_max_n_EB_ratio  = EB_64_ratio[mask_fit]

    KAW_dr_corr = KAW_dr_corr[mask_all]
    EB_64_ratio_corr = EB_64_ratio[mask_all]
    EB_64_ratio_coh     = EB_64_ratio.where(coherency >= wco_sig95)

    phase_coh   = phase.where(coherency >= wco_sig95)

    _, n_EB_max             = _corr_log_model(for_max_n_EB_ratio.freq, for_max_n_EB_ratio.data, for_max_n_EB_dr, min_points=10)

    r_EB, n_EB              = _corr_log_model(EB_64_ratio_corr.freq, EB_64_ratio_corr.data, KAW_dr_corr, min_points=10)
    logrmse_EB_abs, _, _    = _logrmse_model(EB_64_ratio_corr.freq, EB_64_ratio_corr.data, KAW_dr_corr, min_points=10)

    # κ推定（失敗なら NaN）
    kappa_E, kappa_E_err    = _fit_powerlaw_kappa(freq, E_64.values, mask_all, min_points=10)
    kappa_B, kappa_B_err    = _fit_powerlaw_kappa(freq, B_64.values, mask_all, min_points=10)

    # Spara (frequency依存)
    Spara_freq_64   = ds_64_time['Spara'] * 1E3 # [mW m-2]
    Spara_freq_64   = Spara_freq_64.where(coherency >= wco_sig95)

    #v_g_para_64     = v_A * np.sqrt(1. + 0.5 * (Spara_freq_64.freq / f_ci * v_thi / v_sys)**2. * (1. + 1. / tau))   # [m s-1]

    #Effective_wave_energy_density_64    = np.abs(Spara_freq_64 / v_g_para_64) * 1E-3 / elementary_charge * 1E-6 # [eV cm-3]

    wave_energy_density_64  = 0.5 / mu_0 / v_A**2. / (1. + 0.5 * (Spara_freq_64.freq / f_ci * v_thi / v_sys)**2.) * E_64*1E-6 / elementary_charge * 1E-6   # [eV cm-3 Hz-1]

    wave_energy_density_64 = wave_energy_density_64.where(coherency >= wco_sig95)

    if fig_plot == True:
        # plot
        import matplotlib as mpl
        import matplotlib.pyplot as plt

        mpl.rcParams['font.size'] = 20

        fig     = plt.figure(figsize=(10, 20))
        gs      = fig.add_gridspec(6, 1, height_ratios=[4, 3, 3, 4, 4, 4], hspace=0.15)
        ax_0    = fig.add_subplot(gs[0, 0])
        ax_1    = fig.add_subplot(gs[1, 0], sharex=ax_0)
        ax_2    = fig.add_subplot(gs[2, 0], sharex=ax_0)
        ax_3    = fig.add_subplot(gs[3, 0], sharex=ax_0)
        ax_4    = fig.add_subplot(gs[4, 0], sharex=ax_0)
        ax_5    = fig.add_subplot(gs[5, 0], sharex=ax_0)

        ax_0.tick_params(axis='x', which='both', labelbottom=False)
        ax_1.tick_params(axis='x', which='both', labelbottom=False)
        ax_2.tick_params(axis='x', which='both', labelbottom=False)
        ax_3.tick_params(axis='x', which='both', labelbottom=False)
        ax_4.tick_params(axis='x', which='both', labelbottom=False)

        # 灰色マスク（coherency>=sig95 だけ。fit_rangeではなく“灰色領域”を強調したいならこれ）
        mask_gray = mask_coh
        ax_0.plot(E_64.freq,   E_64,  lw=2, linestyle='solid',  c='green')
        ax_0.plot(B_64.freq,   B_64,  lw=2, linestyle='solid',  c='purple')
        ax_0.set_yscale('log')
        ax_0.set_ylim(1E-6, 1E4)
        ax_0.set_yticks([1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
        ax_0.set_xscale('log')
        ax_0.set_xlim(1E-2, 64)
        ax_0.minorticks_on()
        ax_0.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_0.fill_between(freq, 1E-6, 1E4, where=mask_gray, color='gray', alpha=0.30)

#        if np.isfinite(kappa_E):
#            txt_E = rf'$\kappa_E$={kappa_E:.2f}±{kappa_E_err:.2f}'
#            ax_0.text(
#                0.98, 0.95, txt_E,
#                transform=ax_0.transAxes,
#                ha='right', va='top',
#                color='green',
#                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
#            )
#
#        if np.isfinite(kappa_B):
#            txt_B = rf'$\kappa_B$={kappa_B:.2f}±{kappa_B_err:.2f}'
#            ax_0.text(
#                0.98, 0.82, txt_B,
#                transform=ax_0.transAxes,
#                ha='right', va='top',
#                color='purple',
#                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
#            )

        ax_0.set_ylabel(E2_label + r' [$\mathrm{(mV/m)^{2}/Hz}$]' + '\n' + B2_label + r' [$\mathrm{nT^{2}/Hz}$]')

        ax_1.plot(wco_sig95.freq, wco_sig95, lw=2, linestyle='dotted', c='r')
        ax_1.plot(coherency.freq, coherency, lw=2, linestyle='solid',  c='k')
        #ax_1.set_yscale('log')
        #ax_1.set_ylim(1E-3, 1)
        ax_1.set_ylim(0, 1)
        ax_1.minorticks_on()
        ax_1.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_1.set_ylabel('Coherence')

        ax_2.plot(phase_coh.freq, np.abs(phase_coh), lw=2, linestyle='solid', c='k')
        ax_2.axhline(90, lw=2, linestyle='dashed', c='gray', alpha=0.5)
        ax_2.axhline(60, lw=2, linestyle='dashed', c='red', alpha=0.5)
        ax_2.set_ylim(0, 180)
        ax_2.set_yticks([0, 45, 90, 135, 180])
        ax_2.minorticks_on()
        ax_2.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_2.set_ylabel('|' + phi_label + '| [deg]')

        ax_3.plot(EB_64_ratio_coh.freq,  EB_64_ratio_coh,  lw=2, linestyle='solid',  c='k')
        ax_3.plot(f_sc, KAW_dr, lw=2, linestyle='dotted', c='r')
        ax_3.set_yscale('log')
        ax_3.set_ylim(1E-1, 1E3)
        ax_3.minorticks_on()
        ax_3.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_3.set_ylabel(EBratio_label)
        if np.isfinite(r_EB):
            ax_3.text(
                0.98, 0.47, r'$r_{\mathrm{KAW}}$ =' + f'{r_EB:.2f}',
                transform=ax_3.transAxes, ha='right', va='center',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
            )
        if np.isfinite(logrmse_EB_abs):
            ax_3.text(
                0.98, 0.34, rf'logRMSE={logrmse_EB_abs:.2f}',
                transform=ax_3.transAxes, ha='right', va='center',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
            )
        ax_3.text(0.98, 0.21, r'$n_{\mathrm{KAW}}$' + f'= {n_EB}',
                  transform=ax_3.transAxes, ha='right', va='center',
                  bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))
        ax_3.text(0.98, 0.08, r'$n_{\mathrm{KAW, max}}$' + f'= {n_EB_max}',
                  transform=ax_3.transAxes, ha='right', va='center',
                  bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

        ax_4.plot(Spara_freq_64.freq, Spara_freq_64, lw=2, linestyle='solid', c='r')
        ax_4.plot(Spara_freq_64.freq, -Spara_freq_64, lw=2, linestyle='solid', c='b')
        ax_4.set_yscale('log')
        ax_4.set_ylim(1E-7, 1E-1)
        ax_4.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1])
        ax_4.minorticks_on()
        ax_4.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_4.set_ylabel(S_para_label + r' [$\mathrm{mW}/\mathrm{m}^{2}$]')

        #ax_5.plot(Effective_wave_energy_density_64.freq, Effective_wave_energy_density_64, lw=2, linestyle='solid', c='k')
        ax_5.plot(wave_energy_density_64.freq, wave_energy_density_64, lw=2, linestyle='solid', c='k')
        ax_5.set_yscale('log')
        ax_5.set_ylim(1E-7, 1E3)
        ax_5.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3])
        ax_5.minorticks_on()
        ax_5.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_5.set_ylabel(r'$W$ [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]')
        ax_5.set_xlabel('Frequency [Hz]')

        ax_0.set_title(
            f"{time.strftime('%H:%M:%S.%f')}" + '\n Arase, ' +
            S_para_label + f' = {(S_para*1E3):.3f} ' + r'[$\mathrm{mW/m^{2}}$]'
        )

        for ax in [ax_0, ax_1, ax_2, ax_3, ax_4, ax_5]:
            ax.axvline(f_spin_1, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_2, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_3, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_4, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_5, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_sc_krho_1,   lw=2, linestyle='dashed', c='orange',  alpha=0.7)
            ax.axvline(f_sc_krho_low, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
            ax.axvline(f_sc_krho_high,lw=2, linestyle='dashed', c='magenta', alpha=0.7)

        def add_panel_label(ax, label, x=-0.15, y=0.95):
            ax.text(x, y, label, transform=ax.transAxes,
                ha='right', va='bottom', clip_on=False)

        add_panel_label(ax_0, '(a)')
        add_panel_label(ax_1, '(b)')
        add_panel_label(ax_2, '(c)')
        add_panel_label(ax_3, '(d)')
        add_panel_label(ax_4, '(e)')
        add_panel_label(ax_5, '(f)')

        fig.tight_layout()
        fig.savefig(f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_toroidal_eachtime/PDF/{time}.pdf', bbox_inches='tight')
        fig.savefig(f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_toroidal_eachtime/PNG/{time}.png', bbox_inches='tight')

    elif fig_plot == False:
        fig = None

    fit_results = {
        'time':         time,
        'kappa_E':      kappa_E if kappa_E else np.nan,
        'kappa_E_err':  kappa_E_err if kappa_E_err else np.nan,
        'kappa_B':      kappa_B if kappa_B else np.nan,
        'kappa_B_err':  kappa_B_err if kappa_B_err else np.nan,
        'r_EB':         r_EB if r_EB else np.nan,
        'logrmse_EB':   logrmse_EB_abs if logrmse_EB_abs else np.nan,
        'n_EB':         n_EB if n_EB else np.nan,
        'n_EB_max':     n_EB_max if n_EB_max else np.nan
    }


    return fig, fit_results

In [ ]:
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

class TqdmJoblib(tqdm):
    """
    joblib.Parallel の進捗を「完了ベース」で tqdm に反映するコンテキストマネージャ
    """
    def __enter__(self):
        self._old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = self

        class _BatchCompletionCallBack(self._old_cb):
            def __call__(self, *args, **kwargs):
                # 完了したバッチサイズ分だけ進捗を進める
                pbar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _BatchCompletionCallBack
        return super().__enter__()

    def __exit__(self, exc_type, exc, tb):
        joblib.parallel.BatchCompletionCallBack = self._old_cb
        return super().__exit__(exc_type, exc, tb)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

direction           = 'toroidal'
E2_label            = r'$E_{x}^{2}$'
B2_label            = r'$B_{y}^{2}$'
phi_label           = r'$\phi_{\mathrm{tor}}$'
EBratio_label       = r'$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$'
vsys_label          = r'$V_{\mathrm{sys}x}$'
Spara_label         = r'$S_{\parallel\mathrm{tor}}$'

ds_64               = ds_EB64_fac_toroidal_interp
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_toroidal_interp
ds_Spara            = da_Spara_toroidal_interp

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}_eachtime'
os.makedirs(out_dir, exist_ok=True)

out_dir_pdf = f'{out_dir}/PDF/'
os.makedirs(out_dir_pdf, exist_ok=True)
out_dir_png = f'{out_dir}/PNG/'
os.makedirs(out_dir_png, exist_ok=True)

def process_and_save_plot(t):
    fig, fit_results = plot_freq_spectrum(t, ds_64, ds_par, ds_vel, ds_Spara,
                                          direction, E2_label, B2_label, phi_label,
                                          EBratio_label, vsys_label, Spara_label, fit_range=[3, 0], fig_plot=False) # np.sqrt(1.67262192E-27 / 9.1093837E-31)
    if fig is None and fit_results is None:
        return None
    elif fig is None and fit_results is not None:
        return fit_results

    ts = pd.Timestamp(t)
    base = ts.strftime('%Y-%m-%dT%H%M%S%f')
    png_path = os.path.join(out_dir_png, base + '.png')
    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')

    try:
        fig.savefig(png_path, dpi=200, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
    finally:
        plt.close(fig)

    return fit_results

time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']

t_min, t_max    = pd.to_datetime(time_range)
time_grid = ds_64.sel(time=slice(t_min, t_max)).time.values

with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
    results_list = Parallel(n_jobs=-1, backend="loky", verbose=0)(
        delayed(process_and_save_plot)(t) for t in time_grid
    )
print('Finished saving all plots!')

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

In [ ]:
plot_freq_spectrum('2022-09-01T22:38:20.394913280', ds_EB64_fac_toroidal_interp, ds_parameter_interp, ds_velocity_ms_toroidal_interp, da_Spara_toroidal_interp, r'toroidal', r'$E_{x}^{2}$', r'$B_{y}^{2}$', r'$\phi$', r'$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$', r'$V_{\mathrm{sys}x}$', r'$S_{\parallel}^{E_{x} B_{y}}$', fit_range=[3, 0])

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr

mpl.rcParams['font.size'] = 20

direction           = 'toroidal'

ds_64               = ds_EB64_fac_toroidal_interp
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_toroidal_interp
S_par_64            = da_Spara_toroidal_interp

S_par_label         = r'$S_{\parallel}^{E_{x} B_{y}}$'
v_sys_label         = r'$V_{\mathrm{sys}x}$'

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}_eachtime'

time_range_data = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
t_min_data, t_max_data    = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime('%Y%m%d_%H%M%S')
time_str_end_data = t_max_data.strftime('%Y%m%d_%H%M%S')

kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv'
csv_path = os.path.join(out_dir, kappa_csv_filename)

data = pd.read_csv(csv_path)
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").set_index("time").to_xarray()

def plot_one_window(t_min: pd.Timestamp, t_max: pd.Timestamp) -> None:
    data_win = data.sel(time=slice(t_min, t_max))

    S_par_64_window = S_par_64.sel(time=slice(t_min, t_max))
    v_sys_window    = ds_vel['perp_sys_speed'].sel(time=slice(t_min, t_max))
    B_total_window  = ds_par['B_total_nT'].sel(time=slice(t_min, t_max))

    data_win, S_par_64_window, v_sys_window = xr.align(
        data_win, S_par_64_window, v_sys_window, join="inner"
    )

    m = (
        (data_win['r_EB'].data >= 0.7) &
        (data_win['logrmse_EB'].data <= 1.0) &
        (data_win['n_EB'].data >= data_win['n_EB_max'].data / 3.) &
        (data_win['n_EB_max'].data >= 100) &
        (np.abs(S_par_64_window.data * 1E3) >= 9E-3)
    )


    x  = pd.to_datetime(data_win['time'].values[m])
    yB = data_win['kappa_B'].values[m]
    eB = data_win['kappa_B_err'].values[m]
    yE = data_win['kappa_E'].values[m]
    eE = data_win['kappa_E_err'].values[m]

    yB_valid = yB[np.isfinite(yB)]
    yE_valid = yE[np.isfinite(yE)]

    # 平均・標準偏差（サンプル標準偏差 ddof=1 推奨。母集団なら ddof=0）
    if yB_valid.size > 0:
        kB_mean = float(np.mean(yB_valid))
        kB_std  = float(np.std(yB_valid, ddof=1)) if yB_valid.size > 1 else 0.0
    else:
        kB_mean, kB_std = np.nan, np.nan

    if yE_valid.size > 0:
        kE_mean = float(np.mean(yE_valid))
        kE_std  = float(np.std(yE_valid, ddof=1)) if yE_valid.size > 1 else 0.0
    else:
        kE_mean, kE_std = np.nan, np.nan

    # --- plot ---
    fig_kappa = plt.figure(figsize=(15, 8))
    gs = fig_kappa.add_gridspec(4, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
    ax_2 = fig_kappa.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.scatter(x, yB, s=1, color='purple')
    #ax_0.errorbar(x, yB, yerr=eB, fmt='o', linestyle='none',
    #              color='purple', label=r'$\kappa_{\mathrm{B}}$', ms=4, capsize=3, elinewidth=1)
    ax_0.axhline(7./3., lw=2, linestyle='dotted', c='purple',
                 label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
    if np.isfinite(kB_mean):
        ax_0.axhline(kB_mean, lw=2, linestyle='--', c='purple',
                label=rf'$\kappa_{{\mathrm{{B}}}}$ mean ({kB_mean:.2f})')

    ax_0.scatter(x, yE, s=1, color='green')
    #ax_0.errorbar(x, yE, yerr=eE, fmt='o', linestyle='none',
    #              color='green', label=r'$\kappa_{\mathrm{E}}$', ms=4, capsize=3, elinewidth=1)
    ax_0.axhline(1./3., lw=2, linestyle='dotted', c='green',
                 label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
    if np.isfinite(kE_mean):
        ax_0.axhline(kE_mean, lw=2, linestyle='--', c='green',
                label=rf'$\kappa_{{\mathrm{{E}}}}$ mean ({kE_mean:.2f})')

    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)' + f' ({direction})' + '\n' + rf'$\kappa_{{\mathrm{{E}}}} = {kE_mean:.2f} \pm {kE_std:.2f}$, $\kappa_{{\mathrm{{B}}}} = {kB_mean:.2f} \pm {kB_std:.2f}$')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_0.set_xlim(t_min, t_max)
    ax_0.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax_1.plot(S_par_64_window.time, S_par_64_window.data*1E3, c='k', lw=1)    #  / B_total_window.data
    ax_1.set_ylabel(S_par_label + '\n' + r'[$\mathrm{mW/m^{2}}$]')           #  + r'$/B_{{0}}$' /nT
    ax_1.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')

    ax_2.plot(v_sys_window.time, v_sys_window.data*1E-3, c='k', linewidth=1)
    ax_2.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
    ax_2.set_ylabel(v_sys_label + '\n' + r'[$\mathrm{km/s}$]')
    ax_2.minorticks_on()
    ax_2.grid(True, linestyle=':', which='both')
    ax_2.set_xlabel('Time')

    fig_kappa.tight_layout()

    # ファイル名
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S%f')
    time_str_end   = t_max.strftime('%Y%m%d_%H%M%S%f')
    kappa_base     = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}'
    kappa_png_path = os.path.join(out_dir, kappa_base + '.png')
    kappa_pdf_path = os.path.join(out_dir, kappa_base + '.pdf')

    fig_kappa.savefig(kappa_png_path, dpi=200, bbox_inches='tight')
    fig_kappa.savefig(kappa_pdf_path, bbox_inches='tight')
    plt.close(fig_kappa)

# --- 5分窓でループ ---
start = pd.Timestamp('2022-09-01T21:05:00')
end   = pd.Timestamp('2022-09-02T00:00:00')
step  = pd.Timedelta(minutes=5)

t = start
while t < end:
    plot_one_window(t, t + step)
    t += step

plot_one_window(pd.Timestamp('2022-09-01T22:25:00'), pd.Timestamp('2022-09-01T23:15:00'))
plot_one_window(pd.Timestamp('2022-09-01T22:28:45'), pd.Timestamp('2022-09-01T22:31:15'))
plot_one_window(pd.Timestamp('2022-09-01T22:41:15'), pd.Timestamp('2022-09-01T22:47:30'))
plot_one_window(pd.Timestamp('2022-09-01T22:53:45'), pd.Timestamp('2022-09-01T23:00:00'))
plot_one_window(pd.Timestamp('2022-09-01T23:09:30'), pd.Timestamp('2022-09-01T23:10:30'))
plot_one_window(pd.Timestamp('2022-09-01T23:08:45'), pd.Timestamp('2022-09-01T23:15:00'))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr

mpl.rcParams['font.size'] = 20

direction           = 'toroidal'

ds_64               = ds_EB64_fac_toroidal_interp
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_toroidal
S_par_64            = da_Spara_toroidal_interp

S_par_label         = r'$S_{\parallel}^{E_{x} B_{y}}$'
v_sys_label         = r'$V_{\mathrm{sys}x}$'
title_label         = r'$E_{x}$‒$B_{y}$'

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}_eachtime'

time_range_data = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
t_min_data, t_max_data    = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime('%Y%m%d_%H%M%S')
time_str_end_data = t_max_data.strftime('%Y%m%d_%H%M%S')

kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv'
csv_path = os.path.join(out_dir, kappa_csv_filename)

data = pd.read_csv(csv_path)
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").set_index("time").to_xarray()

def true_intervals(time_values, mask):
    """
    mask=True の連続時間区間を返す。
    time_values: datetime64 array or pandas DatetimeIndex
    mask: bool array
    """
    time_index = pd.DatetimeIndex(time_values)
    mask = np.asarray(mask, dtype=bool)

    if len(time_index) == 0 or not np.any(mask):
        return []

    # True/False の変化点を検出
    padded = np.r_[False, mask, False]
    changes = np.diff(padded.astype(int))
    start_idx = np.where(changes == 1)[0]
    end_idx = np.where(changes == -1)[0] - 1

    intervals = []
    for i0, i1 in zip(start_idx, end_idx):
        t0 = time_index[i0]
        t1 = time_index[i1]

        # 最後の点だけだと幅がゼロになるので、次点との差を使って少し幅を持たせる
        if i1 + 1 < len(time_index):
            t1 = time_index[i1 + 1]
        elif len(time_index) >= 2:
            dt = time_index[i1] - time_index[i1 - 1]
            t1 = time_index[i1] + dt

        intervals.append((t0, t1))

    return intervals

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def plot_one_window(t_min: pd.Timestamp, t_max: pd.Timestamp) -> None:
    data_win = data.sel(time=slice(t_min, t_max))

    S_par_64_window = S_par_64.sel(time=slice(t_min, t_max))
    v_sys_window    = ds_vel['perp_sys_speed'].sel(time=slice(t_min, t_max))
    B_total_window  = ds_par['B_total_nT'].sel(time=slice(t_min, t_max))

    data_win, S_par_64_window = xr.align(
        data_win, S_par_64_window, join="inner"
    )

    m = (
        (data_win['r_EB'].data >= 0.7) &
        (data_win['logrmse_EB'].data <= 1.0) &
        (data_win['n_EB'].data >= data_win['n_EB_max'].data / 3.) &
        (data_win['n_EB_max'].data >= 100) &
        (np.abs(S_par_64_window.data * 1E3) >= 9E-3)
    )
    # --- count grids ---
    n_grid = data_win.sizes["time"]
    n_kaw  = int(np.count_nonzero(m))
    kaw_frac = n_kaw / n_grid if n_grid > 0 else np.nan


    # --- plot ---
    fig_kappa = plt.figure(figsize=(15, 6))
    gs = fig_kappa.add_gridspec(5, 1)
    ax_0 = fig_kappa.add_subplot(gs[0, 0])
    ax_1 = fig_kappa.add_subplot(gs[1:3, 0], sharex=ax_0)
    ax_2 = fig_kappa.add_subplot(gs[3:5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.set_title(
        f'Arase, {title_label},'
        f' {n_kaw}/{n_grid} time samples ({kaw_frac:.3%})'
    )

    ax_0.set_xlim(t_min, t_max)
    ax_0.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax_0.set_ylim(0, 1)
    ax_0.set_yticks([])
    ax_0.set_ylabel("KAW\ndetection")
    ax_0.grid(True, linestyle=':', which='both', axis='x')
    ax_0.minorticks_on()

    for t0_det, t1_det in true_intervals(data_win.time.values, m):
        ax_0.axvspan(t0_det, t1_det, color='tab:red', alpha=0.6, lw=1)

    ax_0.axhline(0.5, color='gray', lw=0.8, alpha=0.5)

    ax_1.plot(S_par_64_window.time, S_par_64_window.data*1E3, c='k', lw=1)    #  / B_total_window.data
    ax_1.set_ylabel(S_par_label + '\n' + r'[$\mathrm{mW/m^{2}}$]')           #  + r'$/B_{{0}}$' /nT
    ax_1.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
    ax_1.minorticks_on()
    ax_1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax_1.grid(True, linestyle=':', which='both')

    ax_2.plot(v_sys_window.time, v_sys_window.data*1E-3, c='k', linewidth=1)
    ax_2.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
    ax_2.set_ylabel(v_sys_label + '\n' + r'[$\mathrm{km/s}$]')
    ax_2.minorticks_on()
    ax_2.grid(True, linestyle=':', which='both')
    ax_2.set_xlabel('Time')

    add_panel_label(ax_0, '(d)', x=-0.05)
    add_panel_label(ax_1, '(e)', x=-0.05)
    add_panel_label(ax_2, '(f)', x=-0.05)

    fig_kappa.tight_layout()

    # ファイル名
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S%f')
    time_str_end   = t_max.strftime('%Y%m%d_%H%M%S%f')
    kappa_base     = f'detection_time_{direction}_{time_str_start}_to_{time_str_end}'
    kappa_png_path = os.path.join(out_dir, kappa_base + '.png')
    kappa_pdf_path = os.path.join(out_dir, kappa_base + '.pdf')

    fig_kappa.savefig(kappa_png_path, dpi=200, bbox_inches='tight')
    fig_kappa.savefig(kappa_pdf_path, bbox_inches='tight')
    plt.close(fig_kappa)

# --- 5分窓でループ ---
start = pd.Timestamp('2022-09-01T21:05:00')
end   = pd.Timestamp('2022-09-02T00:00:00')
step  = pd.Timedelta(minutes=5)

t = start
while t < end:
    plot_one_window(t, t + step)
    t += step

plot_one_window(pd.Timestamp('2022-09-01T22:25:00'), pd.Timestamp('2022-09-01T23:15:00'))
plot_one_window(pd.Timestamp('2022-09-01T22:28:45'), pd.Timestamp('2022-09-01T22:31:15'))
plot_one_window(pd.Timestamp('2022-09-01T22:41:15'), pd.Timestamp('2022-09-01T22:47:30'))
plot_one_window(pd.Timestamp('2022-09-01T22:53:45'), pd.Timestamp('2022-09-01T23:00:00'))
plot_one_window(pd.Timestamp('2022-09-01T23:09:30'), pd.Timestamp('2022-09-01T23:10:30'))
plot_one_window(pd.Timestamp('2022-09-01T23:08:45'), pd.Timestamp('2022-09-01T23:15:00'))

In [ ]:
import numpy as np
from scipy.stats import binned_statistic

def mean_by_kbin(values_2d, k_2d, k_edges):
    """
    values_2d: shape (ntime, nfreq)
    k_2d     : shape (ntime, nfreq)
    k_edges  : bin edges for kperp*rhoi

    returns:
        mean   : shape (len(k_edges)-1,)
        count  : shape (len(k_edges)-1,)
    """
    valid = (
        np.isfinite(values_2d) &
        np.isfinite(k_2d) &
        (k_2d > 0)
    )

    mean, _, _ = binned_statistic(
        k_2d[valid].ravel(),
        values_2d[valid].ravel(),
        statistic='mean',
        bins=k_edges,
    )

    count, _, _ = binned_statistic(
        k_2d[valid].ravel(),
        values_2d[valid].ravel(),
        statistic='count',
        bins=k_edges,
    )

    return mean, count

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

def _fit_powerlaw_kappa(freq, psd, mask, min_points=5):

    f = np.asarray(freq, dtype=float)
    y = np.asarray(psd, dtype=float)

    # 必ずコピーを作り、入力maskを変更しない
    msk = np.array(mask, dtype=bool, copy=True)

    msk &= (
        np.isfinite(f)
        & np.isfinite(y)
        & (f > 0)
        & (y > 0)
    )

    if np.count_nonzero(msk) < min_points:
        return np.nan, np.nan

    x = np.log10(f[msk])
    yy = np.log10(y[msk])

    n = len(x)

    slope, intercept = np.polyfit(x, yy, 1)

    y_fit = intercept + slope * x
    residual = yy - y_fit

    sigma2 = np.sum(residual**2) / (n - 2)
    sxx = np.sum((x - x.mean())**2)

    if not np.isfinite(sxx) or sxx <= 0:
        return np.nan, np.nan

    slope_std = np.sqrt(sigma2 / sxx)

    kappa = -slope
    kappa_std = slope_std

    return kappa, kappa_std

def _fit_kappa_from_prelogged(
    logk,
    logpsd,
    fit_mask,
    min_points=30,
):
    """
    log10(k), log10(PSD)からpower-law index kappaを高速に計算する。

    log10(PSD) = intercept - kappa * log10(k)
    """

    valid = (
        np.asarray(fit_mask, dtype=bool)
        & np.isfinite(logk)
        & np.isfinite(logpsd)
    )

    n = int(np.count_nonzero(valid))

    if n < min_points:
        return np.nan

    x = logk[valid]
    y = logpsd[valid]

    sx = np.sum(x, dtype=np.float64)
    sy = np.sum(y, dtype=np.float64)
    sxx = np.sum(x * x, dtype=np.float64)
    sxy = np.sum(x * y, dtype=np.float64)

    denominator = sxx - sx * sx / n

    if not np.isfinite(denominator) or denominator <= 0:
        return np.nan

    slope = (sxy - sx * sy / n) / denominator

    return -slope


def block_bootstrap_kappa_parallel(
    times,
    accepted_time_mask,
    frequency,
    E2_all,
    B2_all,
    f_ci_all,
    v_thi_all,
    v_sys_all,
    tau_all,
    block_duration="10s",
    n_boot=2000,
    k_lower=3.0,
    rho_e_scale_reference=None,
    recompute_rho_e_scale=True,
    min_points=30,
    seed=42,
    n_jobs=-1,
    backend="threading",
    verbose=0,
):
    """
    時間方向のnon-overlapping block bootstrapにより、
    kappa_E, kappa_Bの不確かさを並列評価する。

    Parameters
    ----------
    times : array-like of datetime64
        tmask適用前の全時間軸。shape (ntime,)

    accepted_time_mask : array-like of bool
        KAW detection criteriaを満たす時間。shape (ntime,)

    frequency : array-like
        使用周波数。shape (nfreq,)

    E2_all, B2_all : array-like
        coherence/phase criterion適用後のPSD。
        使用不可点はNaN。shape (ntime, nfreq)

    f_ci_all, v_thi_all, v_sys_all, tau_all : array-like
        各時間のプラズマパラメータ。shape (ntime,)

    block_duration : str or Timedelta
        bootstrapの時間ブロック長。

    n_boot : int
        bootstrap回数。

    k_lower : float
        fitting範囲の下限。

    rho_e_scale_reference : float or None
        recompute_rho_e_scale=Falseの場合の固定上限。

    recompute_rho_e_scale : bool
        各試行でtau平均から上限を再計算するか。

    min_points : int
        fittingに必要な最小time-frequency点数。

    seed : int
        乱数seed。

    n_jobs : int
        並列数。-1で全論理CPUを使用。

    backend : {"threading", "loky"}
        threading:
            配列を共有するため低メモリ。まずはこちらを推奨。
        loky:
            process並列。CPU並列性は高いがメモリ負荷も高い。

    verbose : int
        joblibの進捗表示。10程度で進捗を表示。

    Returns
    -------
    dict
        bootstrap分布、percentile、標準偏差など。
    """

    # --------------------------------------------------------
    # Input conversion
    # --------------------------------------------------------

    times = pd.DatetimeIndex(pd.to_datetime(times))

    accepted_time_mask = np.asarray(
        accepted_time_mask,
        dtype=bool,
    )

    frequency = np.asarray(
        frequency,
        dtype=np.float64,
    )

    E2_all = np.asarray(
        E2_all,
        dtype=np.float64,
    )

    B2_all = np.asarray(
        B2_all,
        dtype=np.float64,
    )

    f_ci_all = np.asarray(
        f_ci_all,
        dtype=np.float64,
    )

    v_thi_all = np.asarray(
        v_thi_all,
        dtype=np.float64,
    )

    v_sys_all = np.asarray(
        v_sys_all,
        dtype=np.float64,
    )

    tau_all = np.asarray(
        tau_all,
        dtype=np.float64,
    )

    ntime = len(times)
    nfreq = len(frequency)

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    if ntime == 0:
        raise ValueError("times is empty")

    if accepted_time_mask.shape != (ntime,):
        raise ValueError(
            "accepted_time_mask must have shape (ntime,)"
        )

    if E2_all.shape != B2_all.shape:
        raise ValueError(
            "E2_all and B2_all must have the same shape"
        )

    if E2_all.shape != (ntime, nfreq):
        raise ValueError(
            "E2_all and B2_all must have shape (ntime, nfreq)"
        )

    for name, array in {
        "f_ci_all": f_ci_all,
        "v_thi_all": v_thi_all,
        "v_sys_all": v_sys_all,
        "tau_all": tau_all,
    }.items():
        if array.shape != (ntime,):
            raise ValueError(
                f"{name} must have shape (ntime,)"
            )

    if n_boot < 1:
        raise ValueError("n_boot must be >= 1")

    if min_points < 3:
        raise ValueError("min_points must be >= 3")

    if k_lower <= 0:
        raise ValueError("k_lower must be positive")

    block_ns = pd.to_timedelta(block_duration).value

    if block_ns <= 0:
        raise ValueError(
            "block_duration must be positive"
        )

    if not recompute_rho_e_scale:
        if (
            rho_e_scale_reference is None
            or not np.isfinite(rho_e_scale_reference)
            or rho_e_scale_reference <= 0
        ):
            raise ValueError(
                "A positive finite rho_e_scale_reference is required "
                "when recompute_rho_e_scale=False"
            )

    # --------------------------------------------------------
    # Define time blocks
    # --------------------------------------------------------

    elapsed_ns = times.asi8 - times.asi8[0]

    block_id = (
        elapsed_ns // block_ns
    ).astype(np.int64)

    n_blocks = int(block_id.max()) + 1

    # 各blockについて、最初からaccepted timeだけを保存する。
    # bootstrap内で毎回accepted_time_maskを適用する必要がなくなる。
    accepted_block_members = []

    for iblock in range(n_blocks):

        idx = np.flatnonzero(
            (block_id == iblock)
            & accepted_time_mask
        )

        accepted_block_members.append(idx)

    # --------------------------------------------------------
    # Precompute log10(k) and log10(PSD)
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore",
    ):
        k_all = np.abs(
            frequency[None, :]
            / f_ci_all[:, None]
            * v_thi_all[:, None]
            / v_sys_all[:, None]
        )

    logk_all = np.full(
        k_all.shape,
        np.nan,
        dtype=np.float64,
    )

    valid_k = (
        np.isfinite(k_all)
        & (k_all > 0)
    )

    logk_all[valid_k] = np.log10(
        k_all[valid_k]
    )

    # k_all自体は以後不要なので削除可能
    del k_all

    logE_all = np.full(
        E2_all.shape,
        np.nan,
        dtype=np.float64,
    )

    valid_E = (
        np.isfinite(E2_all)
        & (E2_all > 0)
    )

    logE_all[valid_E] = np.log10(
        E2_all[valid_E]
    )

    logB_all = np.full(
        B2_all.shape,
        np.nan,
        dtype=np.float64,
    )

    valid_B = (
        np.isfinite(B2_all)
        & (B2_all > 0)
    )

    logB_all[valid_B] = np.log10(
        B2_all[valid_B]
    )

    logk_lower = np.log10(k_lower)

    mass_ratio = (
        1.67262192e-27
        / 9.1093837e-31
    )

    # --------------------------------------------------------
    # Generate bootstrap draws serially
    # --------------------------------------------------------

    # 並列数に依存せず、同じseedなら同じ結果を得るため、
    # block抽出乱数は並列実行前にまとめて生成する。
    rng = np.random.default_rng(seed)

    selected_blocks_all = rng.integers(
        low=0,
        high=n_blocks,
        size=(n_boot, n_blocks),
        dtype=np.int32,
    )

    # --------------------------------------------------------
    # One bootstrap realization
    # --------------------------------------------------------

    def run_one_bootstrap(selected_blocks):

        selected_parts = [
            accepted_block_members[iblock]
            for iblock in selected_blocks
            if accepted_block_members[iblock].size > 0
        ]

        if len(selected_parts) == 0:
            return np.nan, np.nan, 0

        selected_idx = np.concatenate(
            selected_parts
        )

        n_selected = selected_idx.size

        if n_selected == 0:
            return np.nan, np.nan, 0

        # --------------------------------------------
        # Upper fitting boundary
        # --------------------------------------------

        if recompute_rho_e_scale:

            tau_b = tau_all[selected_idx]

            valid_tau = (
                np.isfinite(tau_b)
                & (tau_b > 0)
            )

            if not np.any(valid_tau):
                return np.nan, np.nan, n_selected

            rho_e_scale_b = np.sqrt(
                mass_ratio
                * np.mean(tau_b[valid_tau])
            )

        else:
            rho_e_scale_b = rho_e_scale_reference

        if (
            not np.isfinite(rho_e_scale_b)
            or rho_e_scale_b <= k_lower
        ):
            return np.nan, np.nan, n_selected

        logk_upper = np.log10(
            rho_e_scale_b
        )

        # --------------------------------------------
        # Extract selected rows
        # --------------------------------------------

        logk_b = logk_all[selected_idx, :]

        fit_mask_b = (
            (logk_b >= logk_lower)
            & (logk_b <= logk_upper)
        )

        # --------------------------------------------
        # Fit E and B
        # --------------------------------------------

        kappa_E_b = _fit_kappa_from_prelogged(
            logk=logk_b,
            logpsd=logE_all[selected_idx, :],
            fit_mask=fit_mask_b,
            min_points=min_points,
        )

        kappa_B_b = _fit_kappa_from_prelogged(
            logk=logk_b,
            logpsd=logB_all[selected_idx, :],
            fit_mask=fit_mask_b,
            min_points=min_points,
        )

        return (
            kappa_E_b,
            kappa_B_b,
            n_selected,
        )

    # --------------------------------------------------------
    # Parallel execution
    # --------------------------------------------------------

    parallel_kwargs = {
        "n_jobs": n_jobs,
        "backend": backend,
        "verbose": verbose,
        "batch_size": 1,
    }

    # process並列時には大配列をmemmap経由で共有する
    if backend == "loky":
        parallel_kwargs.update({
            "max_nbytes": "50M",
            "mmap_mode": "r",
        })

    results = Parallel(
        **parallel_kwargs
    )(
        delayed(run_one_bootstrap)(
            selected_blocks_all[iboot]
        )
        for iboot in range(n_boot)
    )

    kappa_E_boot = np.asarray(
        [result[0] for result in results],
        dtype=np.float64,
    )

    kappa_B_boot = np.asarray(
        [result[1] for result in results],
        dtype=np.float64,
    )

    n_selected_time_boot = np.asarray(
        [result[2] for result in results],
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    def summarize(samples):

        samples = np.asarray(
            samples,
            dtype=np.float64,
        )

        samples = samples[
            np.isfinite(samples)
        ]

        if samples.size == 0:
            return {
                "samples": samples,
                "n_valid": 0,
                "mean": np.nan,
                "median": np.nan,
                "std": np.nan,
                "q16": np.nan,
                "q84": np.nan,
                "q025": np.nan,
                "q975": np.nan,
            }

        q025, q16, q50, q84, q975 = np.percentile(
            samples,
            [2.5, 16, 50, 84, 97.5],
        )

        return {
            "samples": samples,
            "n_valid": samples.size,
            "mean": np.mean(samples),
            "median": q50,
            "std": np.std(samples, ddof=1),
            "q16": q16,
            "q84": q84,
            "q025": q025,
            "q975": q975,
        }

    return {
        "E": summarize(kappa_E_boot),
        "B": summarize(kappa_B_boot),
        "n_selected_time": n_selected_time_boot,
        "n_boot": n_boot,
        "n_blocks": n_blocks,
        "block_duration": str(block_duration),
        "n_jobs": n_jobs,
        "backend": backend,
    }

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["font.size"] = 20

# ----------------------------------------------------------------------
# Assumes these functions/constants/datasets are already defined:
#   mean_by_kbin
#   _fit_powerlaw_kappa
#   block_bootstrap_kappa
#   mu_0, elementary_charge
#   ds_EB_combined_fac_toroidal_interp
#   ds_parameter_interp
#   ds_velocity_ms_toroidal_interp
#   da_Spara_toroidal_combined_interp
# ----------------------------------------------------------------------

PHASE_MODE_CONFIG = {
    "north_traveling": {
        "title": r"Northward traveling ($|\phi| \leq 60^\circ$)",
    },
    "south_traveling": {
        "title": r"Southward traveling ($|\phi| \geq 120^\circ$)",
    },
    "standing": {
        "title": r"Standing-wave-like ($60^\circ < |\phi| < 120^\circ$)",
    },
    "all": {
        "title": "All phase components",
    },
}


def make_phase_mask(phase, mode):
    """
    Select phase components using the absolute phase difference.

    north_traveling : |phi| <= 60 deg
    south_traveling : |phi| >= 120 deg
    standing        : 60 deg < |phi| < 120 deg
    all             : no phase-angle restriction, but undefined phase is excluded
    """
    if mode not in PHASE_MODE_CONFIG:
        raise ValueError(
            f"Unknown phase mode: {mode}. "
            f"Choose from {list(PHASE_MODE_CONFIG)}"
        )

    abs_phase = np.abs(np.asarray(phase, dtype=float))
    finite = np.isfinite(abs_phase)

    if mode == "north_traveling":
        return finite & (abs_phase <= 60.0)

    if mode == "south_traveling":
        return finite & (abs_phase >= 120.0)

    if mode == "standing":
        return finite & (abs_phase > 60.0) & (abs_phase < 120.0)

    return finite


def safe_log10(a):
    """Return log10(a), replacing nonpositive/nonfinite values with NaN."""
    a = np.asarray(a, dtype=float)
    out = np.full(a.shape, np.nan, dtype=float)
    valid = np.isfinite(a) & (a > 0)
    out[valid] = np.log10(a[valid])
    return out


direction = "toroidal"

E2_label = r"$E_{x}^{2}$"
B2_label = r"$B_{y}^{2}$"
EBratio_label = r"$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$"
Spara_label = r"$S_{\parallel}^{E_{x} B_{y}}$"

ds_128 = ds_EB64_fac_toroidal_interp
ds_par = ds_parameter_interp
ds_vel = ds_velocity_ms_toroidal_interp
S_par_128 = da_Spara_toroidal_interp

out_dir = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/"
    f"2022-09-01_third/EB_ratio_{direction}_eachtime"
)
os.makedirs(out_dir, exist_ok=True)

time_range_data = ["2022-09-01T21:00:00", "2022-09-02T00:00:00"]
t_min_data, t_max_data = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime("%Y%m%d_%H%M%S")
time_str_end_data = t_max_data.strftime("%Y%m%d_%H%M%S")

kappa_csv_filename = (
    f"kappa_timeseries_{direction}_"
    f"{time_str_start_data}_to_{time_str_end_data}.csv"
)
csv_path = os.path.join(out_dir, kappa_csv_filename)

time_range_analysis = ["2022-09-01T22:25:00", "2022-09-01T23:15:00"]
t_min, t_max = pd.to_datetime(time_range_analysis)
time_str_start = t_min.strftime("%Y%m%d_%H%M%S")
time_str_end = t_max.strftime("%Y%m%d_%H%M%S")

data = pd.read_csv(csv_path)
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").set_index("time").to_xarray()
data = data.sel(time=slice(t_min, t_max))

ds_128_window = ds_128.sel(time=slice(t_min, t_max))
ds_par_window = ds_par.sel(time=slice(t_min, t_max))
ds_vel_window = ds_vel.sel(time=slice(t_min, t_max))
S_par_128_window = S_par_128.sel(time=slice(t_min, t_max))

(
    data,
    ds_128_window,
    ds_par_window,
    ds_vel_window,
    S_par_128_window,
) = xr.align(
    data,
    ds_128_window,
    ds_par_window,
    ds_vel_window,
    S_par_128_window,
    join="inner",
)

# ----------------------------------------------------------------------
# Common masks and arrays
# ----------------------------------------------------------------------

tmask = (
    (data["r_EB"].data >= 0.7)
    & (data["logrmse_EB"].data <= 1.0)
    & (data["n_EB"].data >= data["n_EB_max"].data / 3.0)
    & (data["n_EB_max"].data >= 100)
    & (np.abs(S_par_128_window.data * 1e3) >= 9e-3)
)

fmask = ds_128_window["freq"].values >= 1e-2
frequency_mask = ds_128_window["freq"].values[fmask]

coherency_all = ds_128_window["coherency"].values[:, fmask]
wco_sig95_all = ds_128_window["wco_sig95"].values[:, fmask]
phase_all = ds_128_window["phase"].values[:, fmask]

E_all = ds_128_window["E64"].values[:, fmask]
B_all = ds_128_window["B64"].values[:, fmask]
Spara_all = ds_128_window["Spara"].values[:, fmask]

f_ci_all = ds_par_window["proton_cycl_freq_Hz"].values
v_thi_all = ds_vel_window["ion_thermal_speed"].values
v_sys_all = ds_vel_window["perp_sys_speed"].values
v_A_all = ds_vel_window["Alfven_speed_MID"].values
tau_all = ds_par_window["i-e_temp_ratio"].values
B_total_all = ds_par_window["B_total_nT"].values

f_ci_mask = f_ci_all[tmask]
v_thi_mask = v_thi_all[tmask]
v_sys_mask = v_sys_all[tmask]
v_A_mask = v_A_all[tmask]
tau_mask = tau_all[tmask]
B_total_mask = B_total_all[tmask]

with np.errstate(divide="ignore", invalid="ignore"):
    kperp_rhoi_abs = np.abs(
        frequency_mask[None, :]
        / f_ci_mask[:, None]
        * v_thi_mask[:, None]
        / v_sys_mask[:, None]
    )

valid_k = np.isfinite(kperp_rhoi_abs) & (kperp_rhoi_abs > 0)
if not np.any(valid_k):
    raise RuntimeError("No finite positive |kx| rho_i values were obtained.")

kmin = np.nanmin(kperp_rhoi_abs[valid_k])
kmax = np.nanmax(kperp_rhoi_abs[valid_k])

nkbin = 100
k_edges = np.logspace(np.log10(kmin), np.log10(kmax), nkbin + 1)
k_centers = np.sqrt(k_edges[:-1] * k_edges[1:])

valid_tau = np.isfinite(tau_mask) & (tau_mask > 0)
if not np.any(valid_tau):
    raise RuntimeError("No finite positive ion-to-electron temperature ratios.")

mass_ratio = 1.67262192e-27 / 9.1093837e-31
rho_e_scale = np.sqrt(mass_ratio * np.mean(tau_mask[valid_tau]))

kappa_mask = (
    (kperp_rhoi_abs >= 3.0)
    & (kperp_rhoi_abs <= rho_e_scale)
)


def run_one_phase_mode(
    phase_mode,
    *,
    n_boot=2000,
    block_duration="10s",
    seed=42,
    save_pdf=False,
):
    """Calculate, bootstrap, and plot one phase-component selection."""
    phase_title = PHASE_MODE_CONFIG[phase_mode]["title"]

    # Apply exactly the same phase/coherence rule to the ordinary fit
    # and to the bootstrap input.
    phase_condition_all = make_phase_mask(phase_all, phase_mode)
    cmask_all = (
        (coherency_all >= wco_sig95_all)
        & phase_condition_all
    )

    E_all_for_fit = np.where(cmask_all, E_all, np.nan)
    B_all_for_fit = np.where(cmask_all, B_all, np.nan)

    cmask = cmask_all[tmask]
    E_mask = np.where(cmask, E_all[tmask], np.nan)
    B_mask = np.where(cmask, B_all[tmask], np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        Spara_freq_mask = (
            Spara_all[tmask]
            * 1e3
            / B_total_mask[:, None]
        )
        Spara_freq_mask = np.abs(
            np.where(cmask, Spara_freq_mask, np.nan)
        )

        EB_ratio_mask = (
            np.sqrt(E_mask / B_mask)
            / v_A_mask[:, None]
            * 1e6
        )

        W_wave_mask = (
            0.5
            / mu_0
            / v_A_mask[:, None] ** 2
            / (1.0 + 0.5 * kperp_rhoi_abs ** 2)
            * E_mask
            * 1e-6
            / elementary_charge
            * 1e-6
        )

    log10_E_mean_k, E_count_k = mean_by_kbin(
        safe_log10(E_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_B_mean_k, B_count_k = mean_by_kbin(
        safe_log10(B_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_EB_ratio_mean_k, EB_ratio_count_k = mean_by_kbin(
        safe_log10(EB_ratio_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_Spara_mean_k, Spara_count_k = mean_by_kbin(
        safe_log10(Spara_freq_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_W_wave_mean_k, W_wave_count_k = mean_by_kbin(
        safe_log10(W_wave_mask),
        kperp_rhoi_abs,
        k_edges,
    )

    kappa_E, kappa_E_err_formal = _fit_powerlaw_kappa(
        kperp_rhoi_abs,
        E_mask,
        kappa_mask,
        min_points=30,
    )
    kappa_B, kappa_B_err_formal = _fit_powerlaw_kappa(
        kperp_rhoi_abs,
        B_mask,
        kappa_mask,
        min_points=30,
    )

    bootstrap_result = block_bootstrap_kappa_parallel(
        times=data["time"].values,
        accepted_time_mask=np.asarray(tmask, dtype=bool),
        frequency=frequency_mask,
        E2_all=E_all_for_fit,
        B2_all=B_all_for_fit,
        f_ci_all=f_ci_all,
        v_thi_all=v_thi_all,
        v_sys_all=v_sys_all,
        tau_all=tau_all,
        block_duration=block_duration,
        n_boot=n_boot,
        k_lower=3.0,
        recompute_rho_e_scale=True,
        rho_e_scale_reference=rho_e_scale,
        min_points=30,
        seed=seed,

        # 全論理CPUを使用
        n_jobs=-1,
    
        # 大配列を共有する
        backend="threading",
    
        # 進捗表示
        verbose=10,
    )

    E_boot = bootstrap_result["E"]
    B_boot = bootstrap_result["B"]

    print(f"\n[{phase_mode}] {phase_title}")
    print(
        f"E: kappa={kappa_E:.4f}, formal={kappa_E_err_formal:.4g}, "
        f"bootstrap 16--84%=[{E_boot['q16']:.4f}, {E_boot['q84']:.4f}]"
    )
    print(
        f"B: kappa={kappa_B:.4f}, formal={kappa_B_err_formal:.4g}, "
        f"bootstrap 16--84%=[{B_boot['q16']:.4f}, {B_boot['q84']:.4f}]"
    )

    # ------------------------------------------------------------------
    # Main figure
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=(10, 16))
    gs = fig.add_gridspec(4, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis="x", which="both", labelbottom=False)
    ax_1.tick_params(axis="x", which="both", labelbottom=False)
    ax_2.tick_params(axis="x", which="both", labelbottom=False)

    ax_0.scatter(
        kperp_rhoi_abs,
        E_mask,
        s=0.05,
        alpha=1 / 51,
        c="lime",
    )
    ax_0.plot(
        k_centers,
        10 ** log10_E_mean_k,
        lw=2,
        linestyle="solid",
        c="green",
    )
    ax_0.scatter(
        kperp_rhoi_abs,
        B_mask,
        s=0.05,
        alpha=1 / 51,
        c="violet",
    )
    ax_0.plot(
        k_centers,
        10 ** log10_B_mean_k,
        lw=2,
        linestyle="solid",
        c="purple",
    )
    ax_0.set_yscale("log")
    ax_0.set_ylim(1e-6, 1e4)
    ax_0.set_yticks(
        [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
         1e0, 1e1, 1e2, 1e3, 1e4]
    )
    ax_0.set_xscale("log")
    ax_0.set_xlim(1e-1, 4e2)
    ax_0.minorticks_on()
    ax_0.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_0.set_ylabel(
        E2_label
        + r" [$\mathrm{(mV/m)^{2}/Hz}$]"
        + "\n"
        + B2_label
        + r" [$\mathrm{nT^{2}/Hz}$]"
    )

    if np.isfinite(kappa_E):
        E_err_minus = kappa_E - E_boot["q16"]
        E_err_plus = E_boot["q84"] - kappa_E
        txt_E = (
            rf"$\kappa_E={kappa_E:.2f}"
            rf"^{{+{E_err_plus:.2f}}}"
            rf"_{{-{E_err_minus:.2f}}}$"
        )
        ax_0.text(
            0.02,
            0.15,
            txt_E,
            transform=ax_0.transAxes,
            ha="left",
            va="bottom",
            color="green",
            bbox=dict(
                facecolor="white",
                alpha=0.7,
                edgecolor="none",
                pad=2,
            ),
        )

    if np.isfinite(kappa_B):
        B_err_minus = kappa_B - B_boot["q16"]
        B_err_plus = B_boot["q84"] - kappa_B
        txt_B = (
            rf"$\kappa_B={kappa_B:.2f}"
            rf"^{{+{B_err_plus:.2f}}}"
            rf"_{{-{B_err_minus:.2f}}}$"
        )
        ax_0.text(
            0.02,
            0.02,
            txt_B,
            transform=ax_0.transAxes,
            ha="left",
            va="bottom",
            color="purple",
            bbox=dict(
                facecolor="white",
                alpha=0.7,
                edgecolor="none",
                pad=2,
            ),
        )

    ax_1.scatter(
        kperp_rhoi_abs,
        EB_ratio_mask,
        s=0.05,
        alpha=1 / 51,
        c="gray",
    )
    ax_1.plot(
        k_centers,
        10 ** log10_EB_ratio_mean_k,
        lw=2,
        linestyle="solid",
        c="k",
    )
    ax_1.set_yscale("log")
    ax_1.set_ylim(1e-1, 1e2)
    ax_1.set_yticks([1e-1, 1e0, 1e1, 1e2])
    ax_1.minorticks_on()
    ax_1.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_1.set_ylabel(EBratio_label)

    ax_2.scatter(
        kperp_rhoi_abs,
        Spara_freq_mask,
        s=0.05,
        alpha=1 / 51,
        c="gray",
    )
    ax_2.plot(
        k_centers,
        10 ** log10_Spara_mean_k,
        lw=2,
        linestyle="solid",
        c="k",
    )
    ax_2.set_yscale("log")
    ax_2.set_ylim(1e-9, 1e-3)
    ax_2.set_yticks([1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3])
    ax_2.minorticks_on()
    ax_2.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_2.set_ylabel(
        Spara_label
        + r"$/B_{0}$"
        + "\n"
        + r"[$\mathrm{mW}/\mathrm{m}^{2}/\mathrm{nT}$]"
    )

    ax_3.scatter(
        kperp_rhoi_abs,
        W_wave_mask,
        s=0.05,
        alpha=1 / 51,
        c="gray",
    )
    ax_3.plot(
        k_centers,
        10 ** log10_W_wave_mean_k,
        lw=2,
        linestyle="solid",
        c="k",
    )
    ax_3.set_yscale("log")
    ax_3.set_ylim(1e-7, 1e3)
    ax_3.set_yticks(
        [1e-7, 1e-6, 1e-5, 1e-4, 1e-3,
         1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3]
    )
    ax_3.minorticks_on()
    ax_3.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_3.set_ylabel(r"$W$ [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]")
    ax_3.set_xlabel(r"$\left| k_{x} \right| \rho_{\mathrm{i}}$")

    ax_0.set_title(
        f"Arase, {phase_title}\n"
        f"{t_min.strftime('%H:%M:%S.%f')} - "
        f"{t_max.strftime('%H:%M:%S.%f')}"
    )

    def add_panel_label(ax, label, x=-0.10, y=0.95):
        ax.text(
            x,
            y,
            label,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            clip_on=False,
        )

    add_panel_label(ax_0, "(e)")
    add_panel_label(ax_1, "(f)")
    add_panel_label(ax_2, "(g)")
    add_panel_label(ax_3, "(h)")

    for ax in [ax_0, ax_1, ax_2, ax_3]:
        ax.axvline(
            3,
            lw=2,
            linestyle="dashed",
            c="magenta",
            alpha=0.7,
        )
        ax.axvline(
            rho_e_scale,
            lw=2,
            linestyle="dashed",
            c="magenta",
            alpha=0.7,
        )

    fig.tight_layout()

    kappa_base = (
        f"kperp_rhoi_averaged_{direction}_{phase_mode}_"
        f"{time_str_start}_to_{time_str_end}"
    )
    main_png_path = os.path.join(out_dir, kappa_base + ".png")
    main_pdf_path = os.path.join(out_dir, kappa_base + ".pdf")

    fig.savefig(main_png_path, dpi=200, bbox_inches="tight")
    if save_pdf:
        fig.savefig(main_pdf_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {main_png_path}")

    # ------------------------------------------------------------------
    # Bootstrap histogram
    # ------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(
        E_boot["samples"],
        bins=40,
        alpha=0.5,
        label=r"$\kappa_E$",
    )
    ax.hist(
        B_boot["samples"],
        bins=40,
        alpha=0.5,
        label=r"$\kappa_B$",
    )
    ax.axvline(kappa_E, linestyle="--")
    ax.axvline(kappa_B, linestyle="--")
    ax.set_xlabel(r"$\kappa$")
    ax.set_ylabel("Count")
    ax.set_title(phase_title)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()

    bootstrap_base = kappa_base + "_bootstrap"
    bootstrap_png_path = os.path.join(
        out_dir,
        bootstrap_base + ".png",
    )
    fig.savefig(
        bootstrap_png_path,
        dpi=200,
        bbox_inches="tight",
    )
    plt.close(fig)
    print(f"Saved to {bootstrap_png_path}")

    n_phase_points = int(np.count_nonzero(cmask))
    n_coherent_points = int(
        np.count_nonzero((coherency_all[tmask] >= wco_sig95_all[tmask]))
    )

    return {
        "phase_mode": phase_mode,
        "phase_label": phase_title,
        "n_phase_coherent_points": n_phase_points,
        "n_all_coherent_points": n_coherent_points,
        "phase_fraction_of_coherent": (
            n_phase_points / n_coherent_points
            if n_coherent_points > 0
            else np.nan
        ),
        "kappa_E": kappa_E,
        "kappa_E_formal_err": kappa_E_err_formal,
        "kappa_E_boot_median": E_boot["median"],
        "kappa_E_boot_std": E_boot["std"],
        "kappa_E_q16": E_boot["q16"],
        "kappa_E_q84": E_boot["q84"],
        "kappa_B": kappa_B,
        "kappa_B_formal_err": kappa_B_err_formal,
        "kappa_B_boot_median": B_boot["median"],
        "kappa_B_boot_std": B_boot["std"],
        "kappa_B_q16": B_boot["q16"],
        "kappa_B_q84": B_boot["q84"],
        "main_figure": main_png_path,
        "bootstrap_figure": bootstrap_png_path,
    }


# ----------------------------------------------------------------------
# Run all four phase selections
# ----------------------------------------------------------------------

phase_modes = [
    "north_traveling",
    "south_traveling",
    "standing",
    "all",
]

results = []

for phase_mode in phase_modes:
    result = run_one_phase_mode(
        phase_mode,
        n_boot=2000,
        block_duration="10s",
        seed=42,
        save_pdf=False,
    )
    results.append(result)

summary_df = pd.DataFrame(results)

summary_filename = (
    f"kappa_phase_summary_{direction}_"
    f"{time_str_start}_to_{time_str_end}.csv"
)
summary_path = os.path.join(out_dir, summary_filename)
summary_df.to_csv(summary_path, index=False)

print("\nSummary")
print(
    summary_df[
        [
            "phase_mode",
            "n_phase_coherent_points",
            "phase_fraction_of_coherent",
            "kappa_E",
            "kappa_E_q16",
            "kappa_E_q84",
            "kappa_B",
            "kappa_B_q16",
            "kappa_B_q84",
        ]
    ].to_string(index=False)
)
print(f"\nSaved summary to {summary_path}")

In [ ]:
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#mpl.rcParams['font.size'] = 20
#
#direction           = 'toroidal'
#E2_label            = r'$E_{x}^{2}$'
#B2_label            = r'$B_{y}^{2}$'
#EBratio_label       = r'$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$'
#Spara_label         = r'$S_{\parallel}^{E_{x} B_{y}}$'
#
#ds_64               = ds_EB64_fac_toroidal_interp
#ds_par              = ds_parameter_interp
#ds_vel              = ds_velocity_ms_toroidal_interp
#S_par_64            = da_Spara_toroidal_interp
#
#S_par_label         = Spara_label
#v_sys_label         = r'$V_{\mathrm{sys}x}$'
#
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}_eachtime'
#
#time_range_data     = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
#t_min_data, t_max_data    = pd.to_datetime(time_range_data)
#time_str_start_data = t_min_data.strftime('%Y%m%d_%H%M%S')
#time_str_end_data = t_max_data.strftime('%Y%m%d_%H%M%S')
#
#kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv'
#csv_path = os.path.join(out_dir, kappa_csv_filename)
#
#time_range_analysis = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range_analysis)
#time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#
#data = pd.read_csv(csv_path)
#
#data["time"] = pd.to_datetime(data["time"])
#data = data.sort_values("time")
#data = data.set_index("time").to_xarray()
#data = data.sel(time=slice(t_min, t_max))
#
#ds_64_window        = ds_64.sel(time=slice(t_min, t_max))
#ds_par_window       = ds_par.sel(time=slice(t_min, t_max))
#ds_vel_window       = ds_vel.sel(time=slice(t_min, t_max))
#S_par_64_window     = S_par_64.sel(time=slice(t_min, t_max))
#
#data, ds_64_window, ds_par_window, ds_vel_window, S_par_64_window   = xr.align(
#    data, ds_64_window, ds_par_window, ds_vel_window, S_par_64_window, join='inner'
#)
#
## --- time mask ---
#tmask = (
#    (data['r_EB'].data >= 0.7) &
#    (data['logrmse_EB'].data <= 1.0) &
#    (data['n_EB'].data >= data['n_EB_max']/3.) &
#    (data['n_EB_max'].data >= 100) &
#    (np.abs(S_par_64_window.data * 1E3) >= 9E-3)
#)
#
## --- frequency mask ---
#fmask = ds_64_window['freq'].values >= 1E-2
#
## --- coherency mask ---
#coherency_mask  = ds_64_window['coherency'].values[tmask][:, fmask]
#wco_sig95_mask  = ds_64_window['wco_sig95'].values[tmask][:, fmask]
#S_par_mask      = ds_64_window['Spara'].values[tmask][:, fmask]
#phase_mask      = ds_64_window['phase'].values[tmask][:, fmask]
#cmask = (coherency_mask >= wco_sig95_mask) & (np.abs(phase_mask) <= 60.) #& (np.abs(phase_mask) <= 120.) # & (S_par_mask > 0.)
#
#time_mask = pd.to_datetime(data['time'].values[tmask])
#frequency_mask = ds_64_window['freq'].values[fmask]
#
#E64_mask            = ds_64_window['E64'].values[tmask][:, fmask]
#E64_mask            = np.where(cmask, E64_mask, np.nan)
#B64_mask            = ds_64_window['B64'].values[tmask][:, fmask]
#B64_mask            = np.where(cmask, B64_mask, np.nan)
#Spara_freq_64_mask  = ds_64_window['Spara'].values[tmask][:, fmask] * 1E3 / ds_par_window['B_total_nT'].values[tmask][:, None]  # [mW m-2 nT-1]
#Spara_freq_64_mask  = np.abs(np.where(cmask, Spara_freq_64_mask, np.nan))
#
#tau_mask    = ds_par_window['i-e_temp_ratio'].values[tmask]
#f_ci_mask   = ds_par_window['proton_cycl_freq_Hz'].values[tmask] * proton_mass_kg / ds_par_window['ion_mass_kg'].values[tmask]
#v_thi_mask  = ds_vel_window['ion_thermal_speed'].values[tmask]
#v_sys_mask  = ds_vel_window['perp_sys_speed'].values[tmask]
#v_A_mask    = ds_vel_window['Alfven_speed_MID'].values[tmask]
#
#kperp_rhoi_abs  = np.abs(frequency_mask[None, :] / f_ci_mask[:, None] * v_thi_mask[:, None] / v_sys_mask[:, None])
#
#EB_ratio_mask   = np.sqrt(E64_mask / B64_mask) / v_A_mask[:, None] * 1E6
#print(EB_ratio_mask)
#W_wave_mask     = 0.5 / mu_0 / v_A_mask[:, None]**2. / (1. + 0.5 * kperp_rhoi_abs**2.) * E64_mask*1E-6 / elementary_charge * 1E-6  # [eV cm-3 Hz-1]
#
#kperp_rhoi_abs  = np.abs(frequency_mask[None, :] / f_ci_mask[:, None] * v_thi_mask[:, None] / v_sys_mask[:, None])
#
## 0 以下や NaN/inf を除くため、log bin の下限・上限を有限値から決める
#valid_k = np.isfinite(kperp_rhoi_abs) & (kperp_rhoi_abs > 0)
#kmin = np.nanmin(kperp_rhoi_abs[valid_k])
#kmax = np.nanmax(kperp_rhoi_abs[valid_k])
#print(kmin, kmax)
#
## 例: 対数 bin
#nkbin = 100
#k_edges = np.logspace(np.log10(kmin), np.log10(kmax), nkbin + 1)
#k_centers = np.sqrt(k_edges[:-1] * k_edges[1:])
#
#log10_E64_mean_k, E64_count_k         = mean_by_kbin(np.log10(E64_mask), kperp_rhoi_abs, k_edges)
#log10_B64_mean_k, B64_count_k         = mean_by_kbin(np.log10(B64_mask), kperp_rhoi_abs, k_edges)
#log10_EB_ratio_mean_k, EB_ratio_count_k = mean_by_kbin(np.log10(EB_ratio_mask), kperp_rhoi_abs, k_edges)
#log10_Spara_mean_k, Spara_count_k       = mean_by_kbin(np.log10(Spara_freq_64_mask), kperp_rhoi_abs, k_edges)
#log10_W_wave_mean_k, W_wave_count_k     = mean_by_kbin(np.log10(W_wave_mask), kperp_rhoi_abs, k_edges)
#
#rho_e_scale = np.sqrt(np.average(ds_par_window['ion_mass_kg'].values[tmask]) / 9.1093837E-31 * np.average(tau_mask))
#
#kappa_mask              = (kperp_rhoi_abs >= 3E0) & (kperp_rhoi_abs <= rho_e_scale)
#kappa_E, kappa_E_err    = _fit_powerlaw_kappa(kperp_rhoi_abs, E64_mask, kappa_mask, min_points=30)
#kappa_B, kappa_B_err    = _fit_powerlaw_kappa(kperp_rhoi_abs, B64_mask, kappa_mask, min_points=30)
#print(kappa_E, kappa_E_err)
#print(kappa_B, kappa_B_err)
#
## plot
#fig     = plt.figure(figsize=(10, 16))
#gs      = fig.add_gridspec(4, 1)
#ax_0    = fig.add_subplot(gs[0, 0])
#ax_1    = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2    = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3    = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.scatter(kperp_rhoi_abs, E64_mask, s=0.05, alpha=1/51, c='lime')
#ax_0.plot(k_centers, 10**log10_E64_mean_k, lw=2, linestyle='solid', c='green')
#ax_0.scatter(kperp_rhoi_abs, B64_mask, s=0.05, alpha=1/51, c='violet')
#ax_0.plot(k_centers, 10**log10_B64_mean_k, lw=2, linestyle='solid', c='purple')
#ax_0.set_yscale('log')
#ax_0.set_ylim(1E-6, 1E4)
#ax_0.set_yticks([1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
#ax_0.set_xscale('log')
#ax_0.set_xlim(1E-1, 4E2)
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.3, linestyle='dashed')
#ax_0.set_ylabel(E2_label + r' [$\mathrm{(mV/m)^{2}/Hz}$]' + '\n' + B2_label + r' [$\mathrm{nT^{2}/Hz}$]')
#
#if np.isfinite(kappa_E):
#    txt_E = rf'$\kappa_E$={kappa_E:.2f}±{kappa_E_err:.2f}'
#    ax_0.text(
#        0.02, 0.15, txt_E,
#        transform=ax_0.transAxes,
#        ha='left', va='bottom',
#        color='green',
#        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
#    )
#if np.isfinite(kappa_B):
#    txt_B = rf'$\kappa_B$={kappa_B:.2f}±{kappa_B_err:.2f}'
#    ax_0.text(
#        0.02, 0.02, txt_B,
#        transform=ax_0.transAxes,
#        ha='left', va='bottom',
#        color='purple',
#        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
#    )
#
#ax_1.scatter(kperp_rhoi_abs, EB_ratio_mask, s=0.05, alpha=1/51, c='gray')
#ax_1.plot(k_centers, 10**log10_EB_ratio_mean_k, lw=2, linestyle='solid', c='k')
#ax_1.set_yscale('log')
#ax_1.set_ylim(1E-1, 1E2)
#ax_1.set_yticks([1E-1, 1E0, 1E1, 1E2])
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.3, linestyle='dashed')
#ax_1.set_ylabel(EBratio_label)
#
#ax_2.scatter(kperp_rhoi_abs, np.abs(Spara_freq_64_mask), s=0.05, alpha=1/51, c='gray')
#ax_2.plot(k_centers, 10**log10_Spara_mean_k, lw=2, linestyle='solid', c='k')
#ax_2.set_yscale('log')
#ax_2.set_ylim(1E-9, 1E-3)
#ax_2.set_yticks([1E-9, 1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3])
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.3, linestyle='dashed')
#ax_2.set_ylabel(Spara_label + r'$/B_{0}$' + '\n' + r'[$\mathrm{mW}/\mathrm{m}^{2}/\mathrm{nT}$]')
#
#ax_3.scatter(kperp_rhoi_abs, W_wave_mask, s=0.05, alpha=1/51, c='gray')
#ax_3.plot(k_centers, 10**log10_W_wave_mean_k, lw=2, linestyle='solid', c='k')
#ax_3.set_yscale('log')
#ax_3.set_ylim(1E-7, 1E3)
#ax_3.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3])
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.3, linestyle='dashed')
#ax_3.set_ylabel(r'$W$ [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]')
#ax_3.set_xlabel(r'$\left| k_{x} \right| \rho_{\mathrm{i}}$')
#
#ax_0.set_title(
#    f"Arase\n {t_min.strftime('%H:%M:%S.%f')} - {t_max.strftime('%H:%M:%S.%f')}"
#)
#
#def add_panel_label(ax, label, x=-0.10, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#add_panel_label(ax_0, '(e)')
#add_panel_label(ax_1, '(f)')
#add_panel_label(ax_2, '(g)')
#add_panel_label(ax_3, '(h)')
#
#for ax in [ax_0, ax_1, ax_2, ax_3]:
#    ax.axvline(3, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
#    ax.axvline(rho_e_scale, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
#
#fig.tight_layout()
#
## プロットを画像として保存
#kappa_base      = f'kperp_rhoi_averaged_{direction}_north_traveling_{time_str_start}_to_{time_str_end}'
#kappa_png_path  = os.path.join(out_dir, kappa_base + '.png')
#fig.savefig(kappa_png_path, dpi=200, bbox_inches='tight')
#plt.close(fig)
#print(f"Saved to {kappa_png_path}")

## 以下、0.4 secで平均化したもの

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

ds_EB64_fac_toroidal_avg    = (
    ds_EB64_fac_toroidal
    .resample(time="400ms", offset="200ms")
    .mean()
)

ds_EB64_fac_poloidal_avg    = (
    ds_EB64_fac_poloidal
    .resample(time="400ms", offset="200ms")
    .mean()
)

da_Spara_toroidal_avg       = (
    S_para_toroidal
    .resample(time="400ms", offset="200ms")
    .mean()
)

da_Spara_poloidal_avg       = (
    S_para_poloidal
    .resample(time="400ms", offset="200ms")
    .mean()
)

new_time = pd.DatetimeIndex(ds_EB64_fac_toroidal_avg.time.values)

ds_EB64_fac_toroidal_avg = ds_EB64_fac_toroidal_avg.reindex(
    time=new_time
)

ds_EB64_fac_poloidal_avg = ds_EB64_fac_poloidal_avg.reindex(
    time=new_time
)

ds_parameter_interp = ds_parameter.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

ds_velocity_ms_toroidal_interp  = ds_velocity_ms_toroidal.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

ds_velocity_ms_poloidal_interp  = ds_velocity_ms_poloidal.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

print(ds_EB64_fac_toroidal_avg)
print(ds_EB64_fac_poloidal_avg)
print(ds_parameter_interp)
print(ds_velocity_ms_toroidal_interp)
print(ds_velocity_ms_poloidal_interp)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def _fit_powerlaw_kappa(freq, psd, mask, min_points=5):

    f = np.asarray(freq)
    y = np.asarray(psd)

    msk = np.asarray(mask, dtype=bool)
    msk &= np.isfinite(f) & np.isfinite(y) & (f > 0) & (y > 0)

    if msk.sum() < min_points:
        return np.nan, np.nan

    x = np.log10(f[msk])
    yy = np.log10(y[msk])

    N = len(x)

    # 線形回帰
    slope, intercept = np.polyfit(x, yy, 1)

    # 残差
    y_fit = intercept + slope * x
    residual = yy - y_fit

    # 残差分散
    sigma2 = np.sum(residual**2) / (N - 2)

    Sxx = np.sum((x - x.mean())**2)

    if Sxx == 0:
        return np.nan, np.nan

    slope_std = np.sqrt(sigma2 / Sxx)

    kappa = -slope
    kappa_std = slope_std

    return kappa, kappa_std


def _corr_log_model(freq, y_obs, y_model_at_freq, min_points=5):
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model_at_freq)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    if msk.sum() < min_points:
        return np.nan, int(msk.sum())

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    # 分散が小さいと相関が不安定なので弾く
    if np.std(logy) < 1e-6 or np.std(logm) < 1e-6:
        return np.nan, int(msk.sum())

    r = np.corrcoef(logy, logm)[0, 1]
    return float(r), int(msk.sum())

def _logrmse_model(freq, y_obs, y_model, min_points=5, allow_offset=False):
    """
    logRMSE = sqrt(mean((log10(y_obs) - (log10(y_model)+a))^2))
    allow_offset=True: a を平均差で最小二乗フィット（縦オフセット許容）
    """
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    n = int(msk.sum())
    if n < min_points:
        return np.nan, np.nan, n  # (logRMSE, a, n)

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    a = 0.0
    if allow_offset:
        a = float(np.mean(logy - logm))
        resid = logy - (logm + a)
    else:
        resid = logy - logm

    rmse = float(np.sqrt(np.mean(resid**2)))
    return rmse, a, n



def plot_freq_spectrum(time, ds_64, ds_par, ds_vel, da_Spara, title_label,
                      E2_label, B2_label, phi_label, EBratio_label, vsys_label, S_para_label,
                      fit_range=[3.0, 30.0]):

    time    = pd.Timestamp(time)

    ds_64_time      = ds_64.sel(time=time, method="nearest")
    ds_par_time     = ds_par.sel(time=time, method="nearest")
    ds_vel_time     = ds_vel.sel(time=time, method="nearest")
    da_Spara_time   = da_Spara.sel(time=time, method="nearest")

    t_center = pd.Timestamp(ds_64_time.time.values)
    time_0   = t_center - pd.Timedelta(milliseconds=200)
    time_1   = t_center + pd.Timedelta(milliseconds=200)

    # 理論曲線
    f_sc    = np.logspace(-2, 2, 1000)
    tau     = ds_par_time['i-e_temp_ratio'].item()
    f_ci    = ds_par_time['proton_cycl_freq_Hz'].item() * proton_mass_kg / ds_par_time['ion_mass_kg'].item()
    v_thi   = ds_vel_time['ion_thermal_speed'].item()
    v_sys   = ds_vel_time['perp_sys_speed'].item()
    S_para  = da_Spara_time.item()
    KAW_dr  = (1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    if fit_range[1] == 0:
        fit_range[1] = np.sqrt(ds_par_time['ion_mass_kg'].item() / 9.1093837E-31 * tau)

    f_sc_krho_1     = np.abs(f_ci * v_sys / v_thi)
    f_sc_krho_low   = f_sc_krho_1 * fit_range[0]
    f_sc_krho_high  = f_sc_krho_1 * fit_range[1]

    f_spin_1 = 0.125
    f_spin_2 = f_spin_1 * 2.
    f_spin_3 = f_spin_1 * 3.
    f_spin_4 = f_spin_1 * 4.
    f_spin_5 = f_spin_1 * 5.

    # PSD
    E_64        = ds_64_time['E64']
    B_64        = ds_64_time['B64']

    coherency   = ds_64_time['coherency']
    wco_sig95   = ds_64_time['wco_sig95']
    phase       = ds_64_time['phase']

    v_A     = ds_vel_time['Alfven_speed_MID'].item()

    EB_64_ratio     = np.sqrt(E_64 / B_64) * 1E6 / v_A

    # --- fit用マスク（coherency + fit_range） ---
    freq = E_64.freq.values
    mask_coh = (coherency >= wco_sig95).values
    mask_fit = (freq >= f_sc_krho_low) & (freq <= f_sc_krho_high) & (freq >= 1E-2)
    mask_all = mask_coh & mask_fit

    KAW_dr_corr = (1. + (freq / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (freq / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    for_max_n_EB_dr     = KAW_dr_corr[mask_fit]
    for_max_n_EB_ratio  = EB_64_ratio[mask_fit]

    KAW_dr_corr = KAW_dr_corr[mask_all]
    EB_64_ratio_corr = EB_64_ratio[mask_all]
    EB_64_ratio_coh     = EB_64_ratio.where(coherency >= wco_sig95)

    phase_coh   = phase.where(coherency >= wco_sig95)

    _, n_EB_max             = _corr_log_model(for_max_n_EB_ratio.freq, for_max_n_EB_ratio.data, for_max_n_EB_dr, min_points=10)

    r_EB, n_EB              = _corr_log_model(EB_64_ratio_corr.freq, EB_64_ratio_corr.data, KAW_dr_corr, min_points=10)
    logrmse_EB_abs, _, _    = _logrmse_model(EB_64_ratio_corr.freq, EB_64_ratio_corr.data, KAW_dr_corr, min_points=10)

    # κ推定（失敗なら NaN）
    kappa_E, kappa_E_err    = _fit_powerlaw_kappa(freq, E_64.values, mask_all, min_points=10)
    kappa_B, kappa_B_err    = _fit_powerlaw_kappa(freq, B_64.values, mask_all, min_points=10)

    # Spara (frequency依存)
    Spara_freq_64   = ds_64_time['Spara'] * 1E3 # [mW m-2]
    Spara_freq_64   = Spara_freq_64.where(coherency >= wco_sig95)

    #v_g_para_64     = v_A * np.sqrt(1. + 0.5 * (Spara_freq_64.freq / f_ci * v_thi / v_sys)**2. * (1. + 1. / tau))   # [m s-1]

    #Effective_wave_energy_density_64    = np.abs(Spara_freq_64 / v_g_para_64) * 1E-3 / elementary_charge * 1E-6 # [eV cm-3]

    wave_energy_density_64  = 0.5 / mu_0 / v_A**2. / (1. + 0.5 * (Spara_freq_64.freq / f_ci * v_thi / v_sys)**2.) * E_64*1E-6 / elementary_charge * 1E-6   # [eV cm-3 Hz-1]

    wave_energy_density_64 = wave_energy_density_64.where(coherency >= wco_sig95)

#    fig = None

    # plot
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    mpl.rcParams['font.size'] = 20

    fig     = plt.figure(figsize=(10, 20))
    gs      = fig.add_gridspec(6, 1, height_ratios=[4, 3, 3, 4, 4, 4], hspace=0.15)
    ax_0    = fig.add_subplot(gs[0, 0])
    ax_1    = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2    = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3    = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4    = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5    = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # 灰色マスク（coherency>=sig95 だけ。fit_rangeではなく“灰色領域”を強調したいならこれ）
    mask_gray = mask_coh
    ax_0.plot(E_64.freq,   E_64,  lw=2, linestyle='solid',  c='green')
    ax_0.plot(B_64.freq,   B_64,  lw=2, linestyle='solid',  c='purple')
    ax_0.set_yscale('log')
    ax_0.set_ylim(1E-6, 1E4)
    ax_0.set_yticks([1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
    ax_0.set_xscale('log')
    ax_0.set_xlim(1E-2, 64)
    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_0.fill_between(freq, 1E-6, 1E4, where=mask_gray, color='gray', alpha=0.30)

    if np.isfinite(kappa_E):
        txt_E = rf'$\kappa_E$={kappa_E:.2f}±{kappa_E_err:.2f}'
        ax_0.text(
            0.98, 0.95, txt_E,
            transform=ax_0.transAxes,
            ha='right', va='top',
            color='green',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )

    if np.isfinite(kappa_B):
        txt_B = rf'$\kappa_B$={kappa_B:.2f}±{kappa_B_err:.2f}'
        ax_0.text(
            0.98, 0.82, txt_B,
            transform=ax_0.transAxes,
            ha='right', va='top',
            color='purple',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )

    ax_0.set_ylabel(E2_label + r' [$\mathrm{(mV/m)^{2}/Hz}$]' + '\n' + B2_label + r' [$\mathrm{nT^{2}/Hz}$]')

    ax_1.plot(wco_sig95.freq, wco_sig95, lw=2, linestyle='dotted', c='r')
    ax_1.plot(coherency.freq, coherency, lw=2, linestyle='solid',  c='k')
    #ax_1.set_yscale('log')
    #ax_1.set_ylim(1E-3, 1)
    ax_1.set_ylim(0, 1)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_1.set_ylabel('Coherence')

    ax_2.plot(phase_coh.freq, np.abs(phase_coh), lw=2, linestyle='solid', c='k')
    ax_2.axhline(90, lw=2, linestyle='dashed', c='gray', alpha=0.5)
    ax_2.set_ylim(0, 180)
    ax_2.set_yticks([0, 45, 90, 135, 180])
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_2.set_ylabel('|' + phi_label + '| [deg]')

    ax_3.plot(EB_64_ratio_coh.freq,  EB_64_ratio_coh,  lw=2, linestyle='solid',  c='k')
    ax_3.plot(f_sc, KAW_dr, lw=2, linestyle='dotted', c='r')
    ax_3.set_yscale('log')
    ax_3.set_ylim(1E-1, 1E3)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_3.set_ylabel(EBratio_label)
    if np.isfinite(r_EB):
        ax_3.text(
            0.98, 0.34, r'$r_{\mathrm{EB}}$ =' + f'{r_EB:.2f}',
            transform=ax_3.transAxes, ha='right', va='center',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )
    if np.isfinite(logrmse_EB_abs):
        ax_3.text(
            0.98, 0.21, rf'logRMSE={logrmse_EB_abs:.2f}',
            transform=ax_3.transAxes, ha='right', va='center',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )
    ax_3.text(0.98, 0.08, vsys_label + f'={v_sys*1E-3:.2f} [km/s]',
              transform=ax_3.transAxes, ha='right', va='center',
              bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))
    
    ax_4.plot(Spara_freq_64.freq, Spara_freq_64, lw=2, linestyle='solid', c='r')
    ax_4.plot(Spara_freq_64.freq, -Spara_freq_64, lw=2, linestyle='solid', c='b')
    ax_4.set_yscale('log')
    ax_4.set_ylim(1E-7, 1E-1)
    ax_4.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1])
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_4.set_ylabel(S_para_label + r' [$\mathrm{mW}/\mathrm{m}^{2}$]')

    W_para_label = S_para_label.replace('S_{\\parallel', 'W_').replace('}}$', '}$')
    
    #ax_5.plot(Effective_wave_energy_density_64.freq, Effective_wave_energy_density_64, lw=2, linestyle='solid', c='k')
    ax_5.plot(wave_energy_density_64.freq, wave_energy_density_64, lw=2, linestyle='solid', c='k')
    ax_5.set_yscale('log')
    ax_5.set_ylim(1E-7, 1E3)
    ax_5.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3])
    ax_5.minorticks_on()
    ax_5.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_5.set_ylabel(W_para_label + r' [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]')
    ax_5.set_xlabel('Frequency [Hz]')

    ax_0.set_title(
        f"{time_0.strftime('%H:%M:%S.%f')} - {time_1.strftime('%H:%M:%S.%f')}" + '\n' +
        title_label + ', ' + S_para_label + f' = {(S_para*1E3):.3f} ' + r'[$\mathrm{mW/m^{2}}$]'
    )

    for ax in [ax_0, ax_1, ax_2, ax_3, ax_4, ax_5]:
        ax.axvline(f_spin_1, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_2, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_3, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_4, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_5, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_sc_krho_1,   lw=2, linestyle='dashed', c='orange',  alpha=0.7)
        ax.axvline(f_sc_krho_low, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
        ax.axvline(f_sc_krho_high,lw=2, linestyle='dashed', c='magenta', alpha=0.7)

    fig.tight_layout()

    fit_results = {
        'time':         time,
        'kappa_E':      kappa_E if kappa_E else np.nan,
        'kappa_E_err':  kappa_E_err if kappa_E_err else np.nan,
        'kappa_B':      kappa_B if kappa_B else np.nan,
        'kappa_B_err':  kappa_B_err if kappa_B_err else np.nan,
        'r_EB':         r_EB if r_EB else np.nan,
        'logrmse_EB':   logrmse_EB_abs if logrmse_EB_abs else np.nan,
        'n_EB':         n_EB if n_EB else np.nan,
        'n_EB_max':     n_EB_max if n_EB_max else np.nan
    }

    return fig, fit_results

In [ ]:
plot_freq_spectrum('2022-09-01T22:37:08.000000', ds_EB64_fac_toroidal_avg, ds_parameter_interp, ds_velocity_ms_toroidal_interp, da_Spara_toroidal_avg, r'toroidal', r'$E_{x}^{2}$', r'$B_{y}^{2}$', r'$\phi_{\mathrm{tor}}$', r'$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$', r'$v_{\mathrm{sys}x}$', r'$S_{\parallel\mathrm{tor}}$', fit_range=[3, 100])

In [ ]:
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

class TqdmJoblib(tqdm):
    """
    joblib.Parallel の進捗を「完了ベース」で tqdm に反映するコンテキストマネージャ
    """
    def __enter__(self):
        self._old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = self

        class _BatchCompletionCallBack(self._old_cb):
            def __call__(self, *args, **kwargs):
                # 完了したバッチサイズ分だけ進捗を進める
                pbar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _BatchCompletionCallBack
        return super().__enter__()

    def __exit__(self, exc_type, exc, tb):
        joblib.parallel.BatchCompletionCallBack = self._old_cb
        return super().__exit__(exc_type, exc, tb)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

direction           = 'poloidal'
E2_label            = r'$E_{y}^{2}$'
B2_label            = r'$B_{x}^{2}$'
phi_label           = r'$\phi_{\mathrm{pol}}$'
EBratio_label       = r'$\sqrt{E_{y}^{2} / B_{x}^{2}} / v_{\mathrm{A}}$'
vsys_label          = r'$V_{\mathrm{sys}y}$'
Spara_label         = r'$S_{\parallel\mathrm{pol}}$'

ds_64               = ds_EB64_fac_poloidal_avg
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_poloidal_interp
ds_Spara            = da_Spara_poloidal_avg

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}'
os.makedirs(out_dir, exist_ok=True)

out_dir_pdf = f'{out_dir}/PDF/'
os.makedirs(out_dir_pdf, exist_ok=True)
out_dir_png = f'{out_dir}/PNG/'
os.makedirs(out_dir_png, exist_ok=True)

def process_and_save_plot(t):
    fig, fit_results = plot_freq_spectrum(t, ds_64, ds_par, ds_vel, ds_Spara,
                                          direction, E2_label, B2_label, phi_label,
                                          EBratio_label, vsys_label, Spara_label, fit_range=[3, 0]) # np.sqrt(1.67262192E-27 / 9.1093837E-31)
    if fig is None and fit_results is None:
        return None
    elif fig is None and fit_results is not None:
        return fit_results

    ts = pd.Timestamp(t)
    base = ts.strftime('%Y-%m-%dT%H%M%S%f')
    png_path = os.path.join(out_dir_png, base + '.png')
    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')

    try:
        fig.savefig(png_path, dpi=200, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
    finally:
        plt.close(fig)

    return fit_results

time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']

t_min, t_max    = pd.to_datetime(time_range)
time_grid = ds_64.sel(time=slice(t_min, t_max)).time.values

with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
    results_list = Parallel(n_jobs=-1, backend="loky", verbose=0)(
        delayed(process_and_save_plot)(t) for t in time_grid
    )
print('Finished saving all plots!')

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr

mpl.rcParams['font.size'] = 20

direction           = 'toroidal'

ds_64               = ds_EB64_fac_toroidal_avg
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_toroidal_interp
S_par_64            = da_Spara_toroidal_avg

S_par_label         = r'$S_{\parallel\mathrm{tor}}$'
v_sys_label         = r'$V_{\mathrm{sys}x}$'

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}'

time_range_data = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
t_min_data, t_max_data    = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime('%Y%m%d_%H%M%S')
time_str_end_data = t_max_data.strftime('%Y%m%d_%H%M%S')

kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv'
csv_path = os.path.join(out_dir, kappa_csv_filename)

data = pd.read_csv(csv_path)
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").set_index("time").to_xarray()

def plot_one_window(t_min: pd.Timestamp, t_max: pd.Timestamp) -> None:
    data_win = data.sel(time=slice(t_min, t_max))

    S_par_64_window = S_par_64.sel(time=slice(t_min, t_max))
    v_sys_window    = ds_vel['perp_sys_speed'].sel(time=slice(t_min, t_max))
    B_total_window  = ds_par['B_total_nT'].sel(time=slice(t_min, t_max))

    data_win, S_par_64_window, v_sys_window = xr.align(
        data_win, S_par_64_window, v_sys_window, join="inner"
    )

    m = (
        (data_win['r_EB'].data >= 0.7) &
        (data_win['logrmse_EB'].data <= 1.0) &
        (data_win['n_EB'].data >= data_win['n_EB_max'].data / 3.) &
        (np.abs(S_par_64_window.data * 1E3) >= 9E-3)
    )


    x  = pd.to_datetime(data_win['time'].values[m])
    yB = data_win['kappa_B'].values[m]
    eB = data_win['kappa_B_err'].values[m]
    yE = data_win['kappa_E'].values[m]
    eE = data_win['kappa_E_err'].values[m]

    yB_valid = yB[np.isfinite(yB)]
    yE_valid = yE[np.isfinite(yE)]

    # 平均・標準偏差（サンプル標準偏差 ddof=1 推奨。母集団なら ddof=0）
    if yB_valid.size > 0:
        kB_mean = float(np.mean(yB_valid))
        kB_std  = float(np.std(yB_valid, ddof=1)) if yB_valid.size > 1 else 0.0
    else:
        kB_mean, kB_std = np.nan, np.nan

    if yE_valid.size > 0:
        kE_mean = float(np.mean(yE_valid))
        kE_std  = float(np.std(yE_valid, ddof=1)) if yE_valid.size > 1 else 0.0
    else:
        kE_mean, kE_std = np.nan, np.nan

    # --- plot ---
    fig_kappa = plt.figure(figsize=(15, 8))
    gs = fig_kappa.add_gridspec(4, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
    ax_2 = fig_kappa.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.errorbar(x, yB, yerr=eB, fmt='o', linestyle='none',
                  color='purple', label=r'$\kappa_{\mathrm{B}}$', ms=4, capsize=3, elinewidth=1)
    ax_0.axhline(7./3., lw=2, linestyle='dotted', c='purple',
                 label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
    if np.isfinite(kB_mean):
        ax_0.axhline(kB_mean, lw=2, linestyle='--', c='purple',
                label=rf'$\kappa_{{\mathrm{{B}}}}$ mean ({kB_mean:.2f})')

    ax_0.errorbar(x, yE, yerr=eE, fmt='o', linestyle='none',
                  color='green', label=r'$\kappa_{\mathrm{E}}$', ms=4, capsize=3, elinewidth=1)
    ax_0.axhline(1./3., lw=2, linestyle='dotted', c='green',
                 label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
    if np.isfinite(kE_mean):
        ax_0.axhline(kE_mean, lw=2, linestyle='--', c='green',
                label=rf'$\kappa_{{\mathrm{{E}}}}$ mean ({kE_mean:.2f})')

    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)' + f' ({direction})' + '\n' + rf'$\kappa_{{\mathrm{{E}}}} = {kE_mean:.2f} \pm {kE_std:.2f}$, $\kappa_{{\mathrm{{B}}}} = {kB_mean:.2f} \pm {kB_std:.2f}$')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_0.set_xlim(t_min, t_max)
    ax_0.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax_1.plot(S_par_64_window.time, S_par_64_window.data*1E3, c='k', lw=1)    #  / B_total_window.data
    ax_1.set_ylabel(S_par_label + '\n' + r'[$\mathrm{mW/m^{2}}$]')           #  + r'$/B_{{0}}$' /nT
    ax_1.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')

    ax_2.plot(v_sys_window.time, v_sys_window.data*1E-3, c='k', linewidth=1)
    ax_2.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
    ax_2.set_ylabel(v_sys_label + '\n' + r'[$\mathrm{km/s}$]')
    ax_2.minorticks_on()
    ax_2.grid(True, linestyle=':', which='both')
    ax_2.set_xlabel('Time')

    fig_kappa.tight_layout()

    # ファイル名
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S%f')
    time_str_end   = t_max.strftime('%Y%m%d_%H%M%S%f')
    kappa_base     = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}'
    kappa_png_path = os.path.join(out_dir, kappa_base + '.png')
    kappa_pdf_path = os.path.join(out_dir, kappa_base + '.pdf')

    fig_kappa.savefig(kappa_png_path, dpi=200, bbox_inches='tight')
    fig_kappa.savefig(kappa_pdf_path, bbox_inches='tight')
    plt.close(fig_kappa)

# --- 5分窓でループ ---
start = pd.Timestamp('2022-09-01T21:05:00')
end   = pd.Timestamp('2022-09-02T00:00:00')
step  = pd.Timedelta(minutes=5)

t = start
while t < end:
    plot_one_window(t, t + step)
    t += step

plot_one_window(pd.Timestamp('2022-09-01T22:25:00'), pd.Timestamp('2022-09-01T23:15:00'))
plot_one_window(pd.Timestamp('2022-09-01T22:28:45'), pd.Timestamp('2022-09-01T22:31:15'))
plot_one_window(pd.Timestamp('2022-09-01T22:41:15'), pd.Timestamp('2022-09-01T22:47:30'))
plot_one_window(pd.Timestamp('2022-09-01T22:53:45'), pd.Timestamp('2022-09-01T23:00:00'))
plot_one_window(pd.Timestamp('2022-09-01T23:09:30'), pd.Timestamp('2022-09-01T23:10:30'))
plot_one_window(pd.Timestamp('2022-09-01T23:08:45'), pd.Timestamp('2022-09-01T23:15:00'))

# KAW検出時刻データの平均plot

In [ ]:
import numpy as np
from scipy.stats import binned_statistic

def mean_by_kbin(values_2d, k_2d, k_edges):
    """
    values_2d: shape (ntime, nfreq)
    k_2d     : shape (ntime, nfreq)
    k_edges  : bin edges for kperp*rhoi

    returns:
        mean   : shape (len(k_edges)-1,)
        count  : shape (len(k_edges)-1,)
    """
    valid = (
        np.isfinite(values_2d) &
        np.isfinite(k_2d) &
        (k_2d > 0)
    )

    mean, _, _ = binned_statistic(
        k_2d[valid].ravel(),
        values_2d[valid].ravel(),
        statistic='mean',
        bins=k_edges,
    )

    count, _, _ = binned_statistic(
        k_2d[valid].ravel(),
        values_2d[valid].ravel(),
        statistic='count',
        bins=k_edges,
    )

    return mean, count

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['font.size'] = 20

direction           = 'poloidal'
E2_label            = r'$E_{y}^{2}$'
B2_label            = r'$B_{x}^{2}$'
EBratio_label       = r'$\sqrt{E_{y}^{2} / B_{x}^{2}} / v_{\mathrm{A}}$'
Spara_label         = r'$S_{\parallel\mathrm{pol}}$'

ds_64               = ds_EB64_fac_poloidal_avg
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_poloidal_interp
S_par_64            = da_Spara_poloidal_avg

S_par_label         = r'$S_{\parallel\mathrm{pol}}$'
v_sys_label         = r'$V_{\mathrm{sys}y}$'

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/EB_ratio_{direction}'

time_range_data     = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
t_min_data, t_max_data    = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime('%Y%m%d_%H%M%S')
time_str_end_data = t_max_data.strftime('%Y%m%d_%H%M%S')

kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv'
csv_path = os.path.join(out_dir, kappa_csv_filename)

time_range_analysis = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range_analysis)
time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
time_str_end = t_max.strftime('%Y%m%d_%H%M%S')

data = pd.read_csv(csv_path)

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time")
data = data.set_index("time").to_xarray()
data = data.sel(time=slice(t_min, t_max))

ds_64_window        = ds_64.sel(time=slice(t_min, t_max))
ds_par_window       = ds_par.sel(time=slice(t_min, t_max))
ds_vel_window       = ds_vel.sel(time=slice(t_min, t_max))
S_par_64_window     = S_par_64.sel(time=slice(t_min, t_max))

data, ds_64_window, ds_par_window, ds_vel_window, S_par_64_window   = xr.align(
    data, ds_64_window, ds_par_window, ds_vel_window, S_par_64_window, join='inner'
)

# --- time mask ---
tmask = (
    (data['r_EB'].data >= 0.7) &
    (data['logrmse_EB'].data <= 1.0) &
    (data['n_EB'].data >= data['n_EB_max']/3.) &
    (np.abs(S_par_64_window.data * 1E3) >= 9E-3)
)

# --- frequency mask ---
fmask = ds_64_window['freq'].values >= 1E-2

# --- coherency mask ---
coherency_mask  = ds_64_window['coherency'].values[tmask][:, fmask]
wco_sig95_mask  = ds_64_window['wco_sig95'].values[tmask][:, fmask]
S_par_mask      = ds_64_window['Spara'].values[tmask][:, fmask]
cmask = (coherency_mask >= wco_sig95_mask) & (S_par_mask > 0.)

time_mask = pd.to_datetime(data['time'].values[tmask])
frequency_mask = ds_64_window['freq'].values[fmask]

E64_mask            = ds_64_window['E64'].values[tmask][:, fmask]
E64_mask            = np.where(cmask, E64_mask, np.nan)
B64_mask            = ds_64_window['B64'].values[tmask][:, fmask]
B64_mask            = np.where(cmask, B64_mask, np.nan)
Spara_freq_64_mask  = ds_64_window['Spara'].values[tmask][:, fmask] * 1E3 / ds_par_window['B_total_nT'].values[tmask][:, None]  # [mW m-2 nT-1]
Spara_freq_64_mask  = np.abs(np.where(cmask, Spara_freq_64_mask, np.nan))

tau_mask    = ds_par_window['i-e_temp_ratio'].values[tmask]
f_ci_mask   = ds_par_window['proton_cycl_freq_Hz'].values[tmask] * proton_mass_kg / ds_par_window['ion_mass_kg'].values[tmask]
v_thi_mask  = ds_vel_window['ion_thermal_speed'].values[tmask]
v_sys_mask  = ds_vel_window['perp_sys_speed'].values[tmask]
v_A_mask    = ds_vel_window['Alfven_speed_MID'].values[tmask]

kperp_rhoi_abs  = np.abs(frequency_mask[None, :] / f_ci_mask[:, None] * v_thi_mask[:, None] / v_sys_mask[:, None])

EB_ratio_mask   = np.sqrt(E64_mask / B64_mask) / v_A_mask[:, None] * 1E6
print(EB_ratio_mask)
W_wave_mask     = 0.5 / mu_0 / v_A_mask[:, None]**2. / (1. + 0.5 * kperp_rhoi_abs**2.) * E64_mask*1E-6 / elementary_charge * 1E-6  # [eV cm-3 Hz-1]

kperp_rhoi_abs  = np.abs(frequency_mask[None, :] / f_ci_mask[:, None] * v_thi_mask[:, None] / v_sys_mask[:, None])

# 0 以下や NaN/inf を除くため、log bin の下限・上限を有限値から決める
valid_k = np.isfinite(kperp_rhoi_abs) & (kperp_rhoi_abs > 0)
kmin = np.nanmin(kperp_rhoi_abs[valid_k])
kmax = np.nanmax(kperp_rhoi_abs[valid_k])
print(kmin, kmax)

# 例: 対数 bin
nkbin = 100
k_edges = np.logspace(np.log10(kmin), np.log10(kmax), nkbin + 1)
k_centers = np.sqrt(k_edges[:-1] * k_edges[1:])

log10_E64_mean_k, E64_count_k         = mean_by_kbin(np.log10(E64_mask), kperp_rhoi_abs, k_edges)
log10_B64_mean_k, B64_count_k         = mean_by_kbin(np.log10(B64_mask), kperp_rhoi_abs, k_edges)
log10_EB_ratio_mean_k, EB_ratio_count_k = mean_by_kbin(np.log10(EB_ratio_mask), kperp_rhoi_abs, k_edges)
log10_Spara_mean_k, Spara_count_k       = mean_by_kbin(np.log10(Spara_freq_64_mask), kperp_rhoi_abs, k_edges)
log10_W_wave_mean_k, W_wave_count_k     = mean_by_kbin(np.log10(W_wave_mask), kperp_rhoi_abs, k_edges)

rho_e_scale = np.sqrt(np.average(ds_par_window['ion_mass_kg'].values[tmask]) / 9.1093837E-31 * np.average(tau_mask))

kappa_mask              = (kperp_rhoi_abs >= 3E0) & (kperp_rhoi_abs <= rho_e_scale)
kappa_E, kappa_E_err    = _fit_powerlaw_kappa(kperp_rhoi_abs, E64_mask, kappa_mask, min_points=30)
kappa_B, kappa_B_err    = _fit_powerlaw_kappa(kperp_rhoi_abs, B64_mask, kappa_mask, min_points=30)
print(kappa_E, kappa_E_err)
print(kappa_B, kappa_B_err)

# plot
fig     = plt.figure(figsize=(10, 16))
gs      = fig.add_gridspec(4, 1)
ax_0    = fig.add_subplot(gs[0, 0])
ax_1    = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2    = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_3    = fig.add_subplot(gs[3, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)

ax_0.scatter(kperp_rhoi_abs, E64_mask, s=1, alpha=0.1, c='lime')
ax_0.plot(k_centers, 10**log10_E64_mean_k, lw=2, linestyle='solid', c='green')
ax_0.scatter(kperp_rhoi_abs, B64_mask, s=1, alpha=0.1, c='violet')
ax_0.plot(k_centers, 10**log10_B64_mean_k, lw=2, linestyle='solid', c='purple')
ax_0.set_yscale('log')
ax_0.set_ylim(1E-6, 1E4)
ax_0.set_yticks([1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
ax_0.set_xscale('log')
ax_0.set_xlim(1E-1, 4E2)
ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.3, linestyle='dashed')
ax_0.set_ylabel(E2_label + r' [$\mathrm{(mV/m)^{2}/Hz}$]' + '\n' + B2_label + r' [$\mathrm{nT^{2}/Hz}$]')

if np.isfinite(kappa_E):
    txt_E = rf'$\kappa_E$={kappa_E:.2f}±{kappa_E_err:.2f}'
    ax_0.text(
        0.02, 0.15, txt_E,
        transform=ax_0.transAxes,
        ha='left', va='bottom',
        color='green',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
    )
if np.isfinite(kappa_B):
    txt_B = rf'$\kappa_B$={kappa_B:.2f}±{kappa_B_err:.2f}'
    ax_0.text(
        0.02, 0.02, txt_B,
        transform=ax_0.transAxes,
        ha='left', va='bottom',
        color='purple',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
    )

ax_1.scatter(kperp_rhoi_abs, EB_ratio_mask, s=1, alpha=0.1, c='gray')
ax_1.plot(k_centers, 10**log10_EB_ratio_mean_k, lw=2, linestyle='solid', c='k')
ax_1.set_yscale('log')
ax_1.set_ylim(1E-1, 1E2)
ax_1.set_yticks([1E-1, 1E0, 1E1, 1E2])
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.3, linestyle='dashed')
ax_1.set_ylabel(EBratio_label)

ax_2.scatter(kperp_rhoi_abs, np.abs(Spara_freq_64_mask), s=1, alpha=0.1, c='gray')
ax_2.plot(k_centers, 10**log10_Spara_mean_k, lw=2, linestyle='solid', c='k')
ax_2.set_yscale('log')
ax_2.set_ylim(1E-10, 1E-4)
ax_2.set_yticks([1E-10, 1E-9, 1E-8, 1E-7, 1E-6, 1E-5, 1E-4])
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.3, linestyle='dashed')
ax_2.set_ylabel(Spara_label + r'$/B_{0}$' + '\n' + r'[$\mathrm{mW}/\mathrm{m}^{2}/\mathrm{nT}$]')

W_label = Spara_label.replace('S_{\\parallel', 'W_').replace('}}$', '}$')

ax_3.scatter(kperp_rhoi_abs, W_wave_mask, s=1, alpha=0.1, c='gray')
ax_3.plot(k_centers, 10**log10_W_wave_mean_k, lw=2, linestyle='solid', c='k')
ax_3.set_yscale('log')
ax_3.set_ylim(1E-7, 1E3)
ax_3.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3])
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.3, linestyle='dashed')
ax_3.set_ylabel(W_label + r' [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]')
ax_3.set_xlabel(r'$k_{\perp} \rho_{\mathrm{i}}$')

ax_0.set_title(
    f"Arase\n {t_min.strftime('%H:%M:%S.%f')} - {t_max.strftime('%H:%M:%S.%f')}"
)

def add_panel_label(ax, label, x=-0.10, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)
add_panel_label(ax_0, '(e)')
add_panel_label(ax_1, '(f)')
add_panel_label(ax_2, '(g)')
add_panel_label(ax_3, '(h)')

for ax in [ax_0, ax_1, ax_2, ax_3]:
    ax.axvline(3, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
    ax.axvline(rho_e_scale, lw=2, linestyle='dashed', c='magenta', alpha=0.7)

fig.tight_layout()

# プロットを画像として保存
kappa_base      = f'kperp_rhoi_averaged_{direction}_{time_str_start}_to_{time_str_end}'
kappa_png_path  = os.path.join(out_dir, kappa_base + '.png')
kappa_pdf_path  = os.path.join(out_dir, kappa_base + '.pdf')
fig.savefig(kappa_png_path, dpi=200, bbox_inches='tight')
fig.savefig(kappa_pdf_path, bbox_inches='tight')
plt.close(fig)
print(f"Saved to {kappa_png_path}")

# 以下、過去

# 各成分のNoise (Median) を抽出

In [ ]:
#import numpy as np
#import xarray as xr
#import matplotlib.pyplot as plt
#
#pairs_sorted = []
#
#pairs = [
#    ("E64_fac_x_cwt",   da_E64_fac_x_cwt),
#    ("E64_fac_y_cwt",   da_E64_fac_y_cwt),
#    ("E64_fac_z_cwt",   da_E64_fac_z_cwt),
#    ("B64_fac_x_cwt",   da_B64_fac_x_cwt),
#    ("B64_fac_y_cwt",   da_B64_fac_y_cwt),
#    ("B64_fac_z_cwt",   da_B64_fac_z_cwt),
#]
#
#for name, da in pairs:
#    if isinstance(da, tuple):
#        da = da[0]
#    if isinstance(da, xr.DataArray):
#        pairs_sorted.append((name, da.sortby('time')))
#
#noise_t0, noise_t1 = np.datetime64('2022-09-01T21:30:00'), np.datetime64('2022-09-01T21:55:00')
#
#def compute_median_dict(pairs_sorted, t0, t1):
#    d = {}
#    for name, da in pairs_sorted:
#        sub = da.sel(time=slice(t0, t1))
#        if sub.sizes.get('time', 0) == 0:
#            continue
#        if np.iscomplexobj(sub.data):
#            sub = (sub.real**2 + sub.imag**2)
#        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
#    return d
#
#noise_da_dict   = compute_median_dict(pairs_sorted, noise_t0, noise_t1)
#ds_noise_median = xr.Dataset(noise_da_dict)

In [ ]:
## ---- 入力 ----
#pairs = [
#    ("E_64_FAC_x_cwt",   da_E64_fac_x_cwt),
#    ("E_64_FAC_y_cwt",   da_E64_fac_y_cwt),
#    ("E_64_FAC_z_cwt",   da_E64_fac_z_cwt),
#    ("B_64_FAC_x_cwt",   da_B64_fac_x_cwt),
#    ("B_64_FAC_y_cwt",   da_B64_fac_y_cwt),
#    ("B_64_FAC_z_cwt",   da_B64_fac_z_cwt),
#]
#outdir      = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
#)
#os.makedirs(outdir, exist_ok=True)
#
#def plot_median_dict(mdict, t0, t1, outdir=None):
#    fig, ax = plt.subplots(figsize=(8, 8))
#    for name, med in mdict.items():
#        prefix  = name.split('_')[0][0]
#        comp    = name.split('_')[2]
#        coor    = name.split('_')[1].upper()
#        label  = f"${prefix}_{comp}$ ({coor})"
#        ax.loglog(med['freq'], med, label=label)
#
#    ax.minorticks_on()
#    ax.set_xlabel('Frequency [Hz]')
#    ax.set_ylabel('Median PSD')
#    ax.set_title(f"Median {str(t0)[11:16]}–{str(t1)[11:16]}  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
#    ax.grid(True, which='both', ls=':')
#    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
#    ax.legend(ncol=2)
#    ax.set_xlim(1e-2, 32)
#    ax.set_ylim(1e-8, 1e4)
#    plt.tight_layout()
#
#    if outdir and os.path.isdir(outdir):
#        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
#        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)
#
#plot_median_dict(noise_da_dict, noise_t0, noise_t1, outdir)

In [ ]:
#import xarray as xr
#import numpy as np
#
#def make_cwt_clean_segments_64(dsets_64, noise_ds):
#    """
#    入力:
#      dsets_64 : [ds_64_fac_cwt_seg0, ...]
#      noise_ds: 各成分のノイズ床（frequency or freq 次元, 1D）
#    出力:
#      cleaned_64: [ds_64_fac_cwt_clean_seg0, ...]
#    """
#    def _noise_for(var):
#        if var not in noise_ds:
#            return None
#        nda = noise_ds[var]
#        if "frequency" in nda.dims:
#            nda = nda.rename({"frequency": "freq"})
#        return nda
#
#    target_vars = [
#        "E64_fac_x_cwt","E64_fac_y_cwt","E64_fac_z_cwt",
#        "B64_fac_x_cwt","B64_fac_y_cwt","B64_fac_z_cwt",
#    ]
#
#    cleaned_list = []
#    for count, ds in enumerate(dsets_64):
#        print(f"[seg {count}] dims={ds.dims}")
#        # time/freq を持たないセグメントはスキップ
#        if ("time" not in ds.coords) or ("freq" not in ds.coords):
#            print(f"  -> skip (no time/freq coord)")
#            continue
#
#        new_vars = {}
#        coords = {"time": ds["time"], "freq": ds["freq"]}
#
#        for v in target_vars:
#            if v not in ds:
#                continue
#            n_da = _noise_for(v)
#            if n_da is None:
#                continue
#
#            # ノイズ床を各dsのfreqに合わせる
#            n_interp = n_da.interp(freq=ds[v].freq)
#
#            cleaned = ds[v] - n_interp
#            cleaned = cleaned.where(cleaned > 0)
#
#            new_vars[f"{v}_clean"] = xr.DataArray(
#                cleaned.astype(np.float32),
#                dims=("time", "freq"),
#                coords=coords,
#                attrs={**ds[v].attrs, "noise_removed": True}
#            )
#
#            coi_name = v.replace("_cwt", "_coi")
#            if coi_name in ds:
#                new_vars[coi_name] = ds[coi_name]
#
#        cleaned_list.append(xr.Dataset(new_vars, coords=coords))
#
#    return cleaned_list
#
#ds_EB64_fac_cwt_clean_segs = make_cwt_clean_segments_64(ds_EB64_fac_cwt_segs, ds_noise_median)
#
#print(ds_EB64_fac_cwt_clean_segs)

In [ ]:
#targets = [
#    ("E64_fac_x_cwt_clean", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_y_cwt_clean", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_z_cwt_clean", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B64_fac_x_cwt_clean", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_y_cwt_clean", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_z_cwt_clean", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#joined_EB64_fac_cwt_clean = {}
#for v, _, _ in targets:
#    da_, coi = concat_cwt_segments(ds_EB64_fac_cwt_clean_segs, v)
#    da_ = da_.sortby('freq')
#    if da_ is not None: joined_EB64_fac_cwt_clean[v] = (da_.sortby("freq"), coi)

In [ ]:
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined_EB64_fac_cwt_clean: continue
#        da, coi = joined_EB64_fac_cwt_clean[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
#mu_0 = 4.*np.pi*1E-7
#
#Ex_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_x'], ds_EB64_fac_segs[2]['E64_fac_x'], ds_EB64_fac_segs[3]['E64_fac_x'], ds_EB64_fac_segs[4]['E64_fac_x']], dim='time')
#Ey_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_y'], ds_EB64_fac_segs[2]['E64_fac_y'], ds_EB64_fac_segs[3]['E64_fac_y'], ds_EB64_fac_segs[4]['E64_fac_y']], dim='time')
#Ez_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_z'], ds_EB64_fac_segs[2]['E64_fac_z'], ds_EB64_fac_segs[3]['E64_fac_z'], ds_EB64_fac_segs[4]['E64_fac_z']], dim='time')
#Bx_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_x'], ds_EB64_fac_segs[2]['B64_fac_x'], ds_EB64_fac_segs[3]['B64_fac_x'], ds_EB64_fac_segs[4]['B64_fac_x']], dim='time')
#By_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_y'], ds_EB64_fac_segs[2]['B64_fac_y'], ds_EB64_fac_segs[3]['B64_fac_y'], ds_EB64_fac_segs[4]['B64_fac_y']], dim='time')
#Bz_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_z'], ds_EB64_fac_segs[2]['B64_fac_z'], ds_EB64_fac_segs[3]['B64_fac_z'], ds_EB64_fac_segs[4]['B64_fac_z']], dim='time')
#
#S_para  = (Ex_fac * By_fac - Ey_fac * Bx_fac) / mu_0 * 1E-12
#
#S_para_toroidal = Ex_fac * By_fac / mu_0 * 1E-12
#S_para_poloidal = - Ey_fac * Bx_fac / mu_0 * 1E-12
#
#print(S_para)
#print(S_para_toroidal)
#print(S_para_poloidal)

In [ ]:
#Vph_toroidal    = np.abs(Ey_fac / Bx_fac) * 1E6 # [m/s]
#Vph_poloidal    = np.abs(Ex_fac / By_fac) * 1E6 # [m/s]
#
#Vph_perp_comp   = np.sqrt((Ex_fac**2E0 + Ey_fac**2E0) / (Bx_fac**2E0 + By_fac**2E0)) * 1E6 # [m/s]

In [ ]:
#import matplotlib as mpl
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#targets = [
#    ("E64_fac_x_cwt_clean", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_y_cwt_clean", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E64_fac_z_cwt_clean", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B64_fac_x_cwt_clean", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_y_cwt_clean", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B64_fac_z_cwt_clean", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#
#time_windows = [
#    (np.datetime64('2022-09-01T22:35:00'),
#     np.datetime64('2022-09-01T22:40:00')),
#
#    (np.datetime64('2022-09-01T22:50:30'),
#     np.datetime64('2022-09-01T22:53:00')),
#
#    (np.datetime64('2022-09-01T23:09:00'),
#     np.datetime64('2022-09-01T23:12:00')),
#]
#
#def add_panel_label(ax, label, x=-0.10, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#for t0, t1 in time_windows:
#    fig, axes = plt.subplots(len(targets)+2, 1, figsize=(14, 18), sharex=True)
#    axes_cwt = axes[:len(targets)]
#    for ax, (v, ylab, unit) in zip(axes_cwt, targets):
#        if v not in joined_EB64_fac_cwt_clean: continue
#        da, coi = joined_EB64_fac_cwt_clean[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, t1=t1, minutes=None,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    S_para_window   = S_para.sel(time=slice(t0, t1))
#    axes[len(targets)].plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    axes[len(targets)].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    axes[len(targets)].minorticks_on()
#    axes[len(targets)].grid(which='both', alpha=0.5)
#
#    Vph_poloidal_window = Vph_poloidal.sel(time=slice(t0, t1))
#    Vph_toroidal_window = Vph_toroidal.sel(time=slice(t0, t1))
#    Vph_perp_comp_window = Vph_perp_comp.sel(time=slice(t0, t1))
#    VA_LEP_window       = ds_velocity_ms_perp_clean['Alfven_speed_LEP'].sel(time=slice(t0, t1))
#    VA_MID_window       = ds_velocity_ms_perp_clean['Alfven_speed_MID'].sel(time=slice(t0, t1))
#    VA_HFA_window       = ds_velocity_ms_perp_clean['Alfven_speed_HFA'].sel(time=slice(t0, t1))
#    axes[len(targets)+1].plot(Vph_perp_comp_window.time, Vph_perp_comp_window.data*1E-3, c='k', lw=0.5, linestyle='solid', label=r'$|\mathbf{E}_{\perp}| / |\mathbf{B}_{\perp}|$')
#    #axes[len(targets)+1].plot(Vph_poloidal_window.time, Vph_poloidal_window.data*1E-3, c='green', lw=0.5, linestyle='solid', #label='poloidal')
#    #axes[len(targets)+1].plot(Vph_toroidal_window.time, Vph_toroidal_window.data*1E-3, c='orange', lw=0.5, #linestyle='solid', label='toroidal')
#    axes[len(targets)+1].plot(VA_LEP_window.time, VA_LEP_window.data*1E-3, c='blue', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$ (LEP)')
#    axes[len(targets)+1].plot(VA_MID_window.time, VA_MID_window.data*1E-3, c='k', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$ (MID)')
#    axes[len(targets)+1].plot(VA_HFA_window.time, VA_HFA_window.data*1E-3, c='red', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$ (HFA)')
#    axes[len(targets)+1].set_ylabel(r'Velocity' + '\n' + r'[$\mathrm{km/s}$]')
#    axes[len(targets)+1].set_ylim(ymin=1E3, ymax=1E6)
#    axes[len(targets)+1].set_yscale('log')
#    axes[len(targets)+1].minorticks_on()
#    axes[len(targets)+1].grid(which='both', alpha=0.5)
#    axes[len(targets)+1].legend(ncol=5, loc='upper left', fontsize=15)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#    fig.subplots_adjust(hspace=0.1)
#
#    add_panel_label(axes[0], '(1)')
#    add_panel_label(axes[1], '(2)')
#    add_panel_label(axes[2], '(3)')
#    add_panel_label(axes[3], '(4)')
#    add_panel_label(axes[4], '(5)')
#    add_panel_label(axes[5], '(6)')
#    add_panel_label(axes[6], '(7)')
#    add_panel_label(axes[7], '(8)')
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}_for_figure.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# E/B plot

- Poloidal components: $B_{x}$, $E_{y}$
- Toroidal components: $B_{y}$, $E_{x}$

In [ ]:
#da_zeros    = xr.zeros_like(joined_EB64_fac_cwt_clean['E64_fac_x_cwt_clean'][0])
#
#ds_EB64_fac_cwt_clean_toroidal  = xr.Dataset({
#    'E64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_x_cwt_clean'][0],
#    'E64_fac_y_cwt_clean':  da_zeros,
#    'E64_fac_z_cwt_clean':  da_zeros,
#    'B64_fac_x_cwt_clean':  da_zeros,
#    'B64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_y_cwt_clean'][0],
#    'B64_fac_z_cwt_clean':  da_zeros,
#})
#
#ds_EB64_fac_cwt_clean_poloidal  = xr.Dataset({
#    'E64_fac_x_cwt_clean':  da_zeros,
#    'E64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_y_cwt_clean'][0],
#    'E64_fac_z_cwt_clean':  da_zeros,
#    'B64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_x_cwt_clean'][0],
#    'B64_fac_y_cwt_clean':  da_zeros,
#    'B64_fac_z_cwt_clean':  da_zeros,
#})
#
#ds_EB64_fac_cwt_clean_perp      = xr.Dataset({
#    'E64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_x_cwt_clean'][0],
#    'E64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_y_cwt_clean'][0],
#    'E64_fac_z_cwt_clean':  da_zeros,
#    'B64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_x_cwt_clean'][0],
#    'B64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_y_cwt_clean'][0],
#    'B64_fac_z_cwt_clean':  da_zeros,
#})
#
#print(ds_EB64_fac_cwt_clean_toroidal)
#print('')
#print(ds_EB64_fac_cwt_clean_poloidal)
#print('')
#print(ds_EB64_fac_cwt_clean_perp)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'perp'
#dsets_64    = ds_EB64_fac_cwt_clean_perp
#ds_velocity_ms  = ds_velocity_ms_perp_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
#    if fig is None:
#        return
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
##process_and_save_plot(t_min, data_dict, dt_time, out_dir)
#
#Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'toroidal'
#dsets_64    = ds_EB64_fac_cwt_clean_toroidal
#ds_velocity_ms  = ds_velocity_ms_toroidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
#    if fig is None:
#        return
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
##process_and_save_plot(t_min, data_dict, dt_time, out_dir)
#
#Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'poloidal'
#dsets_64    = ds_EB64_fac_cwt_clean_poloidal
#ds_velocity_ms  = ds_velocity_ms_poloidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
#    if fig is None:
#        return
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
##process_and_save_plot(t_min, data_dict, dt_time, out_dir)
#
#Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'poloidal'
#dsets_64    = ds_EB64_fac_cwt_clean_poloidal
#ds_velocity_ms  = ds_velocity_ms_poloidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'toroidal'
#dsets_64    = ds_EB64_fac_cwt_clean_toroidal
#ds_velocity_ms  = ds_velocity_ms_toroidal
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    ## kappa_Bのプロット（エラーバー付き）
#    #ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#    #            fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 15
#
#direction   = 'perp'
#dsets_64    = ds_EB64_fac_cwt_clean_perp
#ds_velocity_ms  = ds_velocity_ms_perp_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
##time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
##time_range = ['2022-09-01T22:50:30', '2022-09-01T22:53:00']
#time_range = ['2022-09-01T23:09:00', '2022-09-01T23:12:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    #ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#    #            fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    #ax_0.plot(df_results.index, df_results['kappa_B'], marker='.', c='blue', label=r'$\kappa_{\mathrm{B}}$', lw=2)
#    #ax_0.axhline(7./3., lw=2, linestyle='dotted', c='blue', label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
#    
#    # kappa_Eのプロット（エラーバー付き）
#    #ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#    #            fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    ax_0.plot(df_results.index, df_results['kappa_E'], marker='.', c='orange', label=r'$\kappa_{\mathrm{E}}$', lw=0.5)
#    ax_0.axhline(1./3., lw=2, linestyle='dotted', c='orange', label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)' + f' ({direction})')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend(loc='best', ncol=4, fontsize=12)
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'poloidal'
#dsets_64    = ds_EB64_fac_cwt_clean_poloidal
#ds_velocity_ms  = ds_velocity_ms_poloidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'toroidal'
#dsets_64    = ds_EB64_fac_cwt_clean_toroidal
#ds_velocity_ms  = ds_velocity_ms_toroidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'perp'
#dsets_64    = ds_EB64_fac_cwt_clean_perp
#ds_velocity_ms  = ds_velocity_ms_perp_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'poloidal'
#dsets_64    = ds_EB64_fac_cwt_clean_poloidal
#ds_velocity_ms  = ds_velocity_ms_poloidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'toroidal'
#dsets_64    = ds_EB64_fac_cwt_clean_toroidal
#ds_velocity_ms  = ds_velocity_ms_toroidal_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.psd_plotter_Arase_xarray as psdpAx
#import importlib
#importlib.reload(psdpAx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction   = 'perp'
#dsets_64    = ds_EB64_fac_cwt_clean_perp
#ds_velocity_ms  = ds_velocity_ms_perp_clean
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 0.4
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdpAx.build_data_dict_xr_erg(
#    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())
#
#    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
#    if out is None:
#        return None
#    fig, fit_results = out
#    if fig is None:
#        return None
#
#    try:
#        if isinstance(dt, (float, np.floating)):
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
#        else:
#            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    if not fit_results:
#        return None
#    if 'time' not in fit_results:
#        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
#    else:
#        fit_results['time'] = pd.Timestamp(fit_results['time'])
#    return fit_results
#
#time_range = ['2022-09-01T23:09:00', '2022-09-01T23:12:00']
##time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#print(results_list)
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")